# Do brain-tumour classifiers look at the tumour? — Run 6 (FINAL notebook)
**Main task:** 3-class tumour typing (glioma / meningioma / pituitary) on Figshare T1c slices, patient-level split.

**Story of the paper (Runs 1-5)**
1. **Problem:** standard models classify Figshare tumours *without needing the tumour* (pituitary 97-99% correct with
   the tumour hidden); public "external" datasets are mostly copies of Figshare.
2. **Existing fixes fail:** multi-task segmentation, RRR, CAM guidance, counterfactual training, foundation models.
3. **Solutions that work (Run 5):** ROI models (classify only the found tumour region) and *tumour transplant*
   training for an ordinary whole-slice model; skull control, shortcut alarm and metric checks passed.

**What Run 6 adds (final)**
| # | Addition | Why |
|---|---|---|
| 1 | **Bug fix:** an ROI model that finds no tumour gives an "unsure" answer; it is now counted as **abstained**, not as glioma | correct per-class numbers |
| 2 | **ROI-m06-Transplant** = ROI crop with more context (margin 0.6) **+** tumour transplant | can we keep context *and* remove its shortcut? |
| 3 | Two extra seeds of **ROI-Crop-m06** (best trade-off in Run 5) | stable numbers for the paper |
| 4 | **High-resolution explanations** for every model incl. the ROI models (crop-level maps pasted back), blood-red colour scale | the main model's heat maps |
| 5 | **Faithfulness test** (deletion curves) + **3D heat-map surfaces** + ROI sanity check | do the red areas really drive the decision? |
| 6 | **Main results at a glance** figure + final success-criteria check | paper-ready summary |

**How to run (final):** attach ALL artifact datasets (`mtk_planA_artifacts`, `mtk-run3-part2`, `mtk-run5-artifacts`,
`mtk-run6-artifacts`) + the data inputs, click **Run All**. Every model is found -> no training, ~1.5 h of tables/figures.
(Fresh start without artifacts: the same notebook trains everything that is missing.) Download the zips as soon as
cell 3.6 shows the links (and again at the end, cell 12.3).

In [ ]:
# ─── 0.1 Install (Kaggle already has torch / torchvision / timm) ───
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imagehash"], check=False)

In [ ]:
# ─── 0.2 Imports + configuration ───
import os, re, gc, json, time, math, random, hashlib, shutil, warnings, platform
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"      # hide the noisy download bars of pretrained weights
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as T
import timm
from scipy import stats as sps
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit, GroupKFold
from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve
from tqdm import tqdm   # plain text bars (no 'Error displaying widget')
from multiprocessing import Pool
warnings.filterwarnings("ignore")

# ===================== RUN SWITCHES =====================
RUN_TRAINING   = True    # False = evaluation only (uses saved checkpoints)
RUN_LR_SEARCH  = True    # short LR search on fold 1 (done once, then remembered)
RUN_EVALUATION = True    # CV / test / external / calibration tables
RUN_XAI        = "auto"  # "auto" = as soon as the main models (seed 0) are trained; results are cached
RUN_RELIANCE   = True    # tumour-hiding test for every trained model (cached)
RUN_SHORTCUT   = True    # old 4-class shortcut case study (no training)

# FULL training plan, in priority order. You do NOT edit this between runs:
# each run trains whatever is not finished yet, then stops in time and makes all tables.
TRAIN_PLAN = [
    # --- already trained in Runs 1-3 (loaded from the attached artifacts, not retrained) ---
    ("MTK", [0, 1, 2]), ("CNN-Backbone", [0]), ("CNN-Backbone-Wide", [0]),
    ("ResNet-50", [0]), ("EfficientNet-B0", [0]), ("ConvNeXt-Tiny", [0]), ("DeiT-Small", [0]),
    ("MTK-noSeq", [0]), ("MTK-noKAN", [0]), ("MTK-noCNN", [0]),
    ("CNX-Plain", [0]), ("CTR-Net", [0]), ("CTR-noCF", [0]), ("CTR-noSeg", [0]),
    ("MTL-Seg", [0]), ("RRR", [0]), ("CAM-Guide", [0]),
    ("CTR-Net", [1, 2]), ("CNX-Plain", [1, 2]), ("CTR-MambaKAN", [0]),
    # --- Run 4 remedy (its checkpoints could not be downloaded -> retrained in Run 5) ---
    ("ROI-Crop", [0]),
    # --- NEW in Run 5: the solutions (seed 0) ---
    ("ROI-Loc", [0]),            # 1: appearance + legitimate location
    ("CNX-Transplant", [0]),     # 2: training-time fix for an ordinary whole-slice model
    ("CNX-Strip", [0]),          # 3: skull control
    ("ROI-Crop-m15", [0]),       # 1b: context sweep (margin 1.5)
    ("ROI-Crop-m06", [0]),       # 1b: context sweep (margin 0.6)
    ("ROI-Mask", [0]), ("ROI-Loc", [1, 2]), ("ROI-Crop", [1, 2]), ("CNX-Transplant", [1, 2]),
    # --- NEW in Run 6 (final) ---
    ("ROI-m06-Transplant", [0]),   # context (margin 0.6) + tumour transplant
    ("ROI-Crop-m06", [1, 2]),      # extra seeds of the best trade-off model
    ("ROI-m06-Transplant", [1, 2]),# extra seeds if time allows (otherwise reported with seed 0 only)
]
TRAIN_QUEUE = TRAIN_PLAN        # trained in this order; whatever does not fit is left for the next run
MAX_TRAIN_HOURS = 6.5    # stop STARTING new training jobs after this (evaluation + new XAI need ~3 h)
SESSION_START = time.time()   # clock starts when the notebook starts

# ===================== PATHS =====================
KAGGLE_IN = Path("/kaggle/input")
FIGSHARE_DIR   = KAGGLE_IN / "datasets/raufuumme/brain-tumor-patient-organized/brain_tumor_patient_organized"
EXT_DIR_BRISC  = KAGGLE_IN / "datasets/farhan1910/brisc-images/brisc_images"
BRISC_MASK_DIR = KAGGLE_IN / "datasets/farhan1910/brisc25-masks/brisc25_masks"
EXT_DIR_MENDELEY = KAGGLE_IN / "datasets/mdkayumhossain/mendeley-dataset/Mendeley_N/Mendeley_N"
OLD_ARTIFACT_DIR = KAGGLE_IN / "datasets/mdkayumhossain/barain-all-saved-models"   # old 4-class run
BRATS_DIR = KAGGLE_IN / "brats-2d-slices"          # folder that contains manifest.csv
BRATS_MIN_TC_PIXELS = 100
FIGSHARE_MAT_DIR = KAGGLE_IN / "datasets/farhan1910/figshare-original-mat"   # original .mat files (tumour masks)
BDNEURO_DIR = KAGGLE_IN / "datasets/farhan1910/bdneuro-mri"                   # Bangladeshi dataset (mostly copies)
BRATS_FREE_DIR = KAGGLE_IN / "datasets/farhan1910/brats-free-slices"          # tumour-free BraTS slices
MSOUD_DIR = KAGGLE_IN / "datasets/masoudnickparvar/brain-tumor-mri-dataset"   # popular Kaggle dataset (audit only)
# Viewing plane of each cluster of cell 1.9 (read from the Run 4 sheet). Clusters are matched by their SIZE, so a
# different cluster numbering cannot mix them up; if the sizes differ, the plane analysis is skipped with a warning.
PLANE_BY_SIZE = {165: "axial", 321: "coronal", 438: "sagittal", 253: "sagittal", 320: "coronal", 317: "axial",
                 228: "sagittal", 172: "axial", 138: "mixed", 118: "mixed", 365: "coronal", 229: "axial"}
ARTIFACT_INPUT_DIRS = []   # leave empty: the previous run's output is found automatically

WORK = Path("/kaggle/working")
MODEL_DIR, OUT_DIR, FIG_DIR = WORK / "models", WORK / "outputs", WORK / "figures"
for d in [MODEL_DIR, OUT_DIR, FIG_DIR, OUT_DIR / "runs", OUT_DIR / "oof", OUT_DIR / "preds", OUT_DIR / "tables", OUT_DIR / "xai"]:
    d.mkdir(parents=True, exist_ok=True)

def locate(path, marker=None):
    """Return `path` if it exists, otherwise search /kaggle/input for a folder
    with the same name (and containing `marker` if given)."""
    path = Path(path)
    if path.exists():
        return path
    if KAGGLE_IN.exists():
        for cand in KAGGLE_IN.rglob(path.name):
            if cand.is_dir() and (marker is None or any(cand.rglob(marker))):
                print(f"  auto-located {path.name}: {cand}")
                return cand
    return path

FIGSHARE_DIR = locate(FIGSHARE_DIR, "*.png")
if not BDNEURO_DIR.exists() and KAGGLE_IN.exists():
    BDNEURO_DIR = next((d for d in KAGGLE_IN.glob("*/*/*") if d.is_dir() and "bdneuro" in d.name.lower()), BDNEURO_DIR)
MASK_NPZ_INPUT = next(iter(KAGGLE_IN.rglob("figshare_masks_224.npz")), None) if KAGGLE_IN.exists() else None
if not (BRATS_FREE_DIR / "manifest_free.csv").exists() and KAGGLE_IN.exists():
    BRATS_FREE_DIR = next((h.parent for h in KAGGLE_IN.rglob("manifest_free.csv")), BRATS_FREE_DIR)
if not (MSOUD_DIR / "Training").exists() and KAGGLE_IN.exists():
    MSOUD_DIR = next((d.parent for d in KAGGLE_IN.glob("**/Training") if (d / "glioma").exists() and (d.parent / "Testing").exists()), MSOUD_DIR)
EXT_DIR_BRISC = locate(EXT_DIR_BRISC)
BRISC_MASK_DIR = locate(BRISC_MASK_DIR)
EXT_DIR_MENDELEY = locate(EXT_DIR_MENDELEY)
OLD_ARTIFACT_DIR = locate(OLD_ARTIFACT_DIR)
if not any(FIGSHARE_MAT_DIR.glob("brainTumorDataPublic_*")) and KAGGLE_IN.exists():
    _hits = sorted({h.parent for h in KAGGLE_IN.glob("**/brainTumorDataPublic_*")})
    if _hits:
        FIGSHARE_MAT_DIR = _hits[0]
if not (BRATS_DIR / "manifest.csv").exists() and KAGGLE_IN.exists():
    hits = list(KAGGLE_IN.rglob("manifest.csv"))
    hits = [h.parent for h in hits if (h.parent / "gli").exists() or (h.parent / "men").exists()]
    if hits:
        BRATS_DIR = hits[0]

# ===================== EXPERIMENT SETTINGS =====================
CLASSES = ["glioma", "meningioma", "pituitary"]
N_CLASSES = len(CLASSES)
CLASS2IDX = {c: i for i, c in enumerate(CLASSES)}
IDX2CLASS = {i: c for c, i in CLASS2IDX.items()}
IDX2CLASS[-1] = "unsure (abstained)"   # ROI model found no tumour
CLASSES_4 = CLASSES + ["notumor"]           # only for the shortcut study
PREFIX = {"glioma": "g", "meningioma": "m", "pituitary": "p", "notumor": "n"}

IMG_SIZE, BATCH_SIZE = 224, 32
NUM_EPOCHS, PATIENCE, WARMUP_EPOCHS = 80, 15, 3
WEIGHT_DECAY, LABEL_SMOOTHING, MIX_PROB = 1e-2, 0.10, 0.5
N_FOLDS, SPLIT_SEED, TEST_FRACTION = 5, 42, 0.15
DEFAULT_LR = {"scratch": 1e-4, "pretrained": 5e-5}
LR_GRID = {"scratch": [3e-4, 1e-4], "pretrained": [1e-4, 5e-5, 1e-5]}
# LR search is run once per model FAMILY; ablations use the LR of their parent model
LR_SEARCH_MODELS = ["MTK", "CNN-Backbone", "ResNet-50", "EfficientNet-B0", "ConvNeXt-Tiny", "DeiT-Small"]
LR_INHERIT = {"MTK-noSeq": "MTK", "MTK-noKAN": "MTK", "MTK-noCNN": "MTK", "CNN-Backbone-Wide": "CNN-Backbone",
              # TA-MTK = ConvNeXt-Tiny backbone + new heads: backbone uses ConvNeXt-Tiny's LR, new heads 10x
              "TA-MTK": "ConvNeXt-Tiny", "TA-MTK-noSeg": "ConvNeXt-Tiny",
              "TA-MTK-noSeq": "ConvNeXt-Tiny", "TA-MTK-noKAN": "ConvNeXt-Tiny",
              "CNX-Plain": "ConvNeXt-Tiny", "CTR-Net": "ConvNeXt-Tiny", "CTR-noCF": "ConvNeXt-Tiny",
              "CTR-noSeg": "ConvNeXt-Tiny", "MTL-Seg": "ConvNeXt-Tiny", "RRR": "ConvNeXt-Tiny",
              "CAM-Guide": "ConvNeXt-Tiny", "CTR-MambaKAN": "ConvNeXt-Tiny",
              "ROI-Crop": "ConvNeXt-Tiny", "ROI-Mask": "ConvNeXt-Tiny",
              "ROI-Loc": "ConvNeXt-Tiny", "ROI-Crop-m06": "ConvNeXt-Tiny", "ROI-Crop-m15": "ConvNeXt-Tiny",
              "CNX-Transplant": "ConvNeXt-Tiny", "CNX-Strip": "ConvNeXt-Tiny", "ROI-m06-Transplant": "ConvNeXt-Tiny"}
ROI_SEG_SOURCE = "MTL-Seg"   # its tumour map (same fold) decides where ROI models look
ROI_MARGIN = 0.2             # box around the found tumour, +20% on each side (some context)
ROI_MIN_PIXELS = 20          # smaller "tumour" = nothing found -> the ROI model says "unsure" (uniform)
CF_HIDE_WEIGHT = 0.5       # counterfactual rule 1: tumour hidden -> prediction must be uniform (unsure)
CF_KEEP_WEIGHT = 0.5       # counterfactual rule 2: random spot hidden -> same label as the original
GUIDE_WEIGHT = 1.0         # RRR / CAM-Guide: weight of the "evidence outside the tumour" penalty
GUIDE_BATCH = 8            # images per batch used for the (double-backprop) guidance penalties
HEAD_LR_MULT = 10          # new (randomly initialised) heads of TA-MTK learn 10x faster than the pretrained backbone
SEG_LOSS_WEIGHT = 0.5      # loss = classification + 0.5 x (BCE + Dice) on the tumour mask (fixed, not tuned)
# --- Run 5 solution settings (fixed before the run, not tuned) ---
TRANSPLANT_PROB = 0.5      # share of training images that receive a tumour from a patient of ANOTHER class
STRIP_ERODE = 17           # skull stripping = head mask eroded by a 17x17 kernel (same "interior" as the skull test)
ALARM_DROP = 0.20          # shortcut alarm: answer only if hiding the found tumour lowers the confidence by >= 0.20
METRIC_SIZES = [9, 25, 41] # hidden-area sizes for the metric check (9 = the main test; kernel size in pixels)
NUM_WORKERS = min(4, os.cpu_count() or 2)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_DP_FOR_PRETRAINED = False   # one GPU for every model (DataParallel leaked RAM and was 15x slower)
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
N_BOOT = 2000
CONF_THRESHOLD = 0.85

plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 300, "font.size": 9,
                     "axes.spines.top": False, "axes.spines.right": False})
from matplotlib.colors import LinearSegmentedColormap
# Blood-red heat-map colour scale (transparent -> dark red -> bright red), SAME scale (0-1) for every model
BLOOD = LinearSegmentedColormap.from_list("blood", [(0.0, (0.35, 0, 0, 0)), (0.2, (0.4, 0, 0, 0)), (0.4, (0.6, 0, 0, 0.45)),
                                                    (0.7, (0.85, 0, 0, 0.78)), (1.0, (1.0, 0.05, 0.05, 0.93))])
BLOOD_SOLID = LinearSegmentedColormap.from_list("blood_solid", ["#1a0000", "#5c0000", "#a30000", "#e00000", "#ff3b3b"])
CLASS_COLORS = {"glioma": "#d1495b", "meningioma": "#00798c", "pituitary": "#edae49", "notumor": "#30638e"}

print("Torch", torch.__version__, "| timm", timm.__version__, "| GPUs:", torch.cuda.device_count())
for name, p in [("Figshare", FIGSHARE_DIR), ("BRISC", EXT_DIR_BRISC), ("BRISC masks", BRISC_MASK_DIR),
                ("Mendeley", EXT_DIR_MENDELEY), ("Old 4-class artifacts", OLD_ARTIFACT_DIR), ("BraTS slices", BRATS_DIR),
                ("Figshare .mat (masks)", FIGSHARE_MAT_DIR), ("BDNeuro-MRI", BDNEURO_DIR),
                ("Mask file (optional)", MASK_NPZ_INPUT or Path("/none")),
                ("BraTS tumour-free slices", BRATS_FREE_DIR), ("Masoud Kaggle dataset", MSOUD_DIR)]:
    print(f"  {'OK ' if Path(p).exists() else 'MISSING'}  {name:22s} {p}")

In [ ]:
# ─── 0.3 Utilities: seeds, hashing, json, artifact sync ───
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def save_json(obj, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))

def load_json(path, default=None):
    path = Path(path)
    return json.load(open(path)) if path.exists() else default

def banner(title, note=""):
    print("\n" + "=" * 70 + f"\n▶ {title}" + (f"\n  {note}" if note else "") + "\n" + "=" * 70, flush=True)

def ram_text():
    try:
        import psutil
        v = psutil.virtual_memory()
        return f"RAM {v.used / 1e9:.1f}/{v.total / 1e9:.0f} GB" + ("  ⚠ RAM HIGH" if v.percent > 85 else "")
    except Exception:
        return ""

ROLE = {"ROI-Crop": "REMEDY: classify only the tumour region found by the tumour map (box with context)",
        "ROI-Mask": "REMEDY: like ROI-Crop, but everything outside the found tumour is blacked out",
        "CTR-Net": "TESTED REMEDY (Run 3): tumour-mask head + mask-guided pooling + counterfactual rules",
        "CTR-noCF": "ABLATION: CTR-Net without the counterfactual rules",
        "CTR-noSeg": "ABLATION: counterfactual rules only (no mask head, no mask-guided pooling)",
        "CTR-MambaKAN": "ABLATION: CTR-Net + Mamba (Seq) + KAN pathways",
        "CNX-Plain": "BASELINE: ConvNeXt-T, same pipeline, no guidance",
        "MTL-Seg": "EXISTING METHOD: multi-task segmentation + classification",
        "RRR": "EXISTING METHOD: Right-for-the-Right-Reasons (input-gradient penalty outside the tumour)",
        "CAM-Guide": "EXISTING METHOD: class-activation-map penalty outside the tumour",
        "ROI-Loc": "SOLUTION 1: tumour appearance (crop) + tumour location relative to the brain (6 numbers)",
        "ROI-Crop-m06": "CONTEXT SWEEP: ROI-Crop with a wider box (margin 0.6)",
        "ROI-Crop-m15": "CONTEXT SWEEP: ROI-Crop with a much wider box (margin 1.5)",
        "CNX-Transplant": "SOLUTION 2: ordinary ConvNeXt trained with tumour transplants between classes",
        "CNX-Strip": "CONTROL: ordinary ConvNeXt on skull-stripped images (is the BraTS gain just 'no skull'?)",
        "ROI-m06-Transplant": "COMBINED: ROI crop with context (margin 0.6) + tumour transplant (context carries no label)"}

def role_of(name):
    if name in ROLE: return ROLE[name]
    if name == "TA-MTK": return "NEW PROPOSED model (tumour-aware: ConvNeXt-Tiny + Mamba + KAN + tumour mask head)"
    if name.startswith("TA-MTK-no"): return "ABLATION of the new proposed model"
    if name == "MTK": return "Original MTK (submitted version, fixed)"
    if name.startswith("MTK-no"): return "ABLATION (MTK with one pathway removed)"
    if name.startswith("CNN-Backbone"): return "ABLATION / baseline (plain CNN, no Mamba, no KAN)"
    return "BASELINE (ImageNet-pretrained)"

def hms(sec):
    sec = int(sec); return f"{sec // 3600}h {sec % 3600 // 60:02d}m" if sec >= 3600 else f"{sec // 60}m {sec % 60:02d}s"

def figshare_pid(path):
    m = re.match(r"^[gmpn]_(.+)_s_\d+$", Path(path).stem)
    if m is None:
        raise ValueError(f"Cannot parse patient id: {path}")
    return m.group(1)

ZIP_PREFIX = "mtk_run6"
SYNCED_CKPTS = set()        # checkpoints copied from attached inputs (already saved -> not zipped again)
ZIP_PART_GB = 1.8          # every zip stays below this size, so the browser download works

def make_artifact_zip(with_figures=True):
    """Small zips (each < 1.8 GB): one RESULTS zip (tables, records, predictions, figures) and
    MODELS part zips (checkpoints). Download all of them; attach all of them next run."""
    import zipfile
    for old in WORK.glob(f"{ZIP_PREFIX}_*.zip"):
        old.unlink()
    res = WORK / f"{ZIP_PREFIX}_results.zip"
    with zipfile.ZipFile(res, "w", zipfile.ZIP_DEFLATED) as z:
        for f in sorted(OUT_DIR.rglob("*")):
            if f.is_file() and f.suffix in (".json", ".npy", ".npz", ".csv", ".md"):
                z.write(f, f"outputs/{f.relative_to(OUT_DIR)}")
        if with_figures:
            for f in sorted(FIG_DIR.glob("*.png")):
                z.write(f, f"figures/{f.name}")
    parts, cur, size = [], [], 0
    for f in sorted(p for p in MODEL_DIR.glob("*.pt") if p.name not in SYNCED_CKPTS):   # only NEW models
        s = f.stat().st_size
        if cur and size + s > ZIP_PART_GB * 1e9:
            parts.append(cur); cur, size = [], 0
        cur.append(f); size += s
    if cur:
        parts.append(cur)
    for k, files in enumerate(parts, 1):
        with zipfile.ZipFile(WORK / f"{ZIP_PREFIX}_models_part{k}.zip", "w", zipfile.ZIP_STORED) as z:
            for f in files:
                z.write(f, f"models/{f.name}")
    print("Saved (download ALL of these files from the Output panel -> /kaggle/working):")
    for f in sorted(WORK.glob(f"{ZIP_PREFIX}_*.zip")):
        print(f"   {f.name:32s} {f.stat().st_size / 1e9:5.2f} GB")
    print(f"   ({len([p for p in MODEL_DIR.glob('*.pt') if p.name not in SYNCED_CKPTS])} new model checkpoints; "
          f"{len(SYNCED_CKPTS)} older ones are already in your attached datasets - keep attaching those)")

CKPT_RE = re.compile(r"^.+_s\d+_f\d+\.pt$")
KEEP_NAMES = ("lr_choice.json", "fold_stats_3class.json", "overlap_audit.csv", "split_3class.csv",
              "figshare_masks_224.npz", "image_props.csv", "plane_clusters.csv")

def sync_artifacts():
    """Copy checkpoints and saved results of earlier runs (zips or folders anywhere in /kaggle/input)."""
    import zipfile
    roots = []
    if KAGGLE_IN.exists():
        for zp in [z for pat in ("mtk_planA_*.zip", "mtk_run3_*.zip", "mtk_run4_*.zip", "mtk_run5_*.zip", "mtk_run6_*.zip") for z in KAGGLE_IN.rglob(pat)]:
            dst = (Path("/kaggle/tmp") if Path("/kaggle/tmp").exists() else Path("/tmp")) / "prev_artifacts" / zp.stem
            if not dst.exists():
                dst.mkdir(parents=True, exist_ok=True); zipfile.ZipFile(zp).extractall(dst)
            roots.append(dst)
        roots.append(KAGGLE_IN)
    n = 0
    for root in roots:
        for src in root.rglob("*"):
            if not src.is_file():
                continue
            if src.suffix == ".pt" and CKPT_RE.match(src.name):
                dst = MODEL_DIR / src.name; SYNCED_CKPTS.add(src.name)
            elif src.parent.name in ("runs", "oof", "preds", "xai", "reliance", "reliance2", "alarm", "fm") and src.parent.parent.name == "outputs":
                dst = OUT_DIR / src.parent.name / src.name
            elif src.name in KEEP_NAMES:
                dst = OUT_DIR / src.name
            else:
                continue
            if dst.exists() and (dst.is_symlink() or dst.stat().st_size > 0):
                continue
            dst.parent.mkdir(parents=True, exist_ok=True)
            if dst.exists():
                dst.unlink()                              # empty file from a full disk -> replace it
            if src.suffix == ".pt":
                dst.symlink_to(src)                       # LINK old models (saves ~10 GB; Kaggle allows ~20 GB)
            else:
                shutil.copy2(src, dst)
            n += 1
    print(f"Earlier results copied/linked: {n} files | model checkpoints available: {len(list(MODEL_DIR.glob('*.pt')))} | "
          f"disk free {shutil.disk_usage(WORK).free / 1e9:.1f} GB")

set_seed(SPLIT_SEED)
sync_artifacts()

## 0.4 Success criteria (written down BEFORE the results exist)
These thresholds were fixed before Run 5. Section 12 checks every one automatically and prints ✔ / ✘.
A ✘ is still a valid, reportable result - the criteria exist so that we cannot move the goal posts afterwards.

In [ ]:
# ─── 0.4 Pre-registered success criteria ───
CRITERIA = [
    ("S1a", "ROI-Loc", "CV macro-F1 >= 96.0% (wins back the accuracy ROI-Crop lost)"),
    ("S1b", "ROI-Loc", "tumour reliance >= 0.60 AND skull/position-only <= 36% (shortcut stays removed)"),
    ("S1c", "ROI-Loc", "BraTS glioma-vs-meningioma AUC >= ROI-Crop - 0.01 (generalisation kept)"),
    ("S2a", "CNX-Transplant", "tumour reliance >= 0.50 (CNX-Plain: 0.35) with the swap test"),
    ("S2b", "CNX-Transplant", "tumour reliance >= 0.45 with the SECOND hiding method (tissue from another patient)"),
    ("S2c", "CNX-Transplant", "CV macro-F1 >= 95.0% and pituitary still-correct-when-hidden < 80% (CNX-Plain: 98.7%)"),
    ("S3",  "CNX-Strip", "if BraTS AUC of CNX-Strip < ROI-Crop - 0.02 -> ROI's BraTS gain is NOT explained by the skull"),
    ("S4a", "Shortcut alarm", "on BraTS, alarm-based ranking has lower AURC than confidence alone (CNX-Plain)"),
    ("S4b", "Shortcut alarm", "separates tumour slices from tumour-free slices with AUROC >= 0.75 (CNX-Plain)"),
    ("S5",  "Metric check", "model ranking by reliance is stable: Spearman(swap, other-patient) >= 0.80"),
    # --- added before Run 6 ---
    ("S6a", "ROI-m06-Transplant", "CV macro-F1 >= 96.0% (context brings the accuracy back)"),
    ("S6b", "ROI-m06-Transplant", "tumour reliance >= 0.60 (the extra context does not become a shortcut)"),
    ("S6c", "ROI-m06-Transplant", "BraTS AUC >= ROI-Crop - 0.01"),
    ("S7",  "XAI (ROI-Loc)", "share of heat inside the tumour >= 2x what a random map gets inside the same crop"),
    ("S8",  "XAI faithfulness", "deleting ROI-Loc's red pixels lowers its confidence faster than deleting random pixels"),
]
print(pd.DataFrame(CRITERIA, columns=["ID", "Experiment", "Success if ..."]).to_string(index=False))

## 1. Data
The split reproduces the **original** patient-level split exactly (same seed, same 4-class stratification),
then removes the IXI no-tumour patients. Test patients and fold membership of every tumour patient are therefore
identical to the submitted paper.

In [ ]:
# ─── 1.1 Reproduce the original split, then keep the 3 tumour classes ───
records = []
for cls in CLASSES_4:
    folder = FIGSHARE_DIR / cls
    files = sorted(list(folder.glob("*.png")) + list(folder.glob("*.jpg")) + list(folder.glob("*.jpeg")))
    for f in files:
        m = re.match(rf"^{PREFIX[cls]}_(.+)_s_\d+$", f.stem)
        records.append({"filepath": str(f), "label4": cls, "label4_idx": CLASSES_4.index(cls),
                        "pid": m.group(1) if m else f.stem})
df_all = pd.DataFrame(records)
assert len(df_all), "No images found: check FIGSHARE_DIR"

df_pat = df_all.groupby("pid").agg(label4=("label4", "first"), label4_idx=("label4_idx", "first")).reset_index()
sss = StratifiedShuffleSplit(n_splits=1, test_size=TEST_FRACTION, random_state=SPLIT_SEED)
tv_idx, te_idx = next(sss.split(df_pat["pid"].values, df_pat["label4_idx"].values))
df_pat_tv = df_pat.iloc[tv_idx].reset_index(drop=True)
test_pids = set(df_pat.iloc[te_idx]["pid"])
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SPLIT_SEED)
val_fold_of = {}
for k, (_, vl) in enumerate(skf.split(df_pat_tv["pid"].values, df_pat_tv["label4_idx"].values), start=1):
    for p in df_pat_tv["pid"].values[vl]:
        val_fold_of[p] = k
df_all["role"] = np.where(df_all["pid"].isin(test_pids), "test", "dev")
df_all["val_fold"] = df_all["pid"].map(val_fold_of)

print(f"4-class reproduction: {len(df_all)} images, {df_all.pid.nunique()} patients, "
      f"test {(df_all.role=='test').sum()} images / {len(test_pids)} patients (original: 713 / 53)")
old_test_csv = OLD_ARTIFACT_DIR / "outputs" / "test_set_index.csv"
if old_test_csv.exists():
    old_pids = set(pd.read_csv(old_test_csv)["pid"].astype(str))
    print("  identical to the original test patients:", old_pids == {str(p) for p in test_pids})

# 3-class (main task)
df3 = df_all[df_all.label4 != "notumor"].copy()
df3["label"] = df3["label4_idx"].astype(int)          # glioma 0, meningioma 1, pituitary 2
assert set(df3.label) == {0, 1, 2}
FOLDS = {}
for k in range(1, N_FOLDS + 1):
    dev = df3[df3.role == "dev"]
    FOLDS[k] = {"train": list(zip(dev[dev.val_fold != k].filepath, dev[dev.val_fold != k].label)),
                "val":   list(zip(dev[dev.val_fold == k].filepath, dev[dev.val_fold == k].label))}
    tr_p = set(dev[dev.val_fold != k].pid); vl_p = set(dev[dev.val_fold == k].pid)
    assert not (tr_p & vl_p) and not (tr_p & test_pids) and not (vl_p & test_pids), "patient leakage"
TEST_ITEMS = list(zip(df3[df3.role == "test"].filepath, df3[df3.role == "test"].label))

df3.assign(split=np.where(df3.role == "test", "test", "fold" + df3.val_fold.fillna(0).astype(int).astype(str))) \
   [["filepath", "label4", "label", "pid", "split"]].to_csv(OUT_DIR / "split_3class.csv", index=False)
summary = df3.groupby(["role", "label4"]).agg(images=("filepath", "size"), patients=("pid", "nunique"))
print(summary)
for k in FOLDS:
    print(f"  fold {k}: train {len(FOLDS[k]['train'])} / val {len(FOLDS[k]['val'])} images")
print(f"  TEST: {len(TEST_ITEMS)} images, {df3[df3.role=='test'].pid.nunique()} patients")

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
for a, role, title in [(ax[0], "dev", "Development (5-fold CV)"), (ax[1], "test", "Held-out test")]:
    d = df3[df3.role == role].groupby("label4").agg(images=("filepath", "size"), patients=("pid", "nunique")).reindex(CLASSES)
    bars = a.bar(CLASSES, d.images, color=[CLASS_COLORS[c] for c in CLASSES])
    for b, n_p in zip(bars, d.patients):
        a.text(b.get_x() + b.get_width() / 2, b.get_height(), f"{n_p} pts", ha="center", va="bottom", fontsize=8)
    a.set(title=title, ylabel="Images")
fig.tight_layout(); fig.savefig(FIG_DIR / "fig_dataset.png"); plt.show()

In [ ]:
# ─── 1.2 Image properties per source (also used in the shortcut study) ───
def native_props(path):
    with Image.open(path) as im:
        return im.size[0], im.size[1], im.mode, Path(path).suffix.lower(), os.path.getsize(path)

props_csv = OUT_DIR / "image_props.csv"
if props_csv.exists() and len(pd.read_csv(props_csv)) == len(df_all):
    props = pd.read_csv(props_csv)                       # cached from an earlier run (reading headers takes ~10 min)
else:
    props = pd.DataFrame([dict(zip(["width", "height", "mode", "ext", "bytes"], native_props(p)), label4=l)
                          for p, l in tqdm(list(zip(df_all.filepath, df_all.label4)), desc="Reading image headers")])
    props.to_csv(props_csv, index=False)
prop_table = props.groupby("label4").agg(n=("width", "size"), width=("width", "median"), height=("height", "median"),
                                         ext=("ext", lambda s: ",".join(sorted(set(s)))),
                                         kB=("bytes", lambda s: round(s.median() / 1024, 1)))
print(prop_table)
prop_table.to_csv(OUT_DIR / "tables" / "image_properties_by_class.csv")

In [ ]:
# ─── 1.3 External datasets: lists (tumour classes only) ───
ALIASES = {"glioma": "glioma", "glioma_tumor": "glioma", "meningioma": "meningioma", "meningioma_tumor": "meningioma",
           "pituitary": "pituitary", "pituitary_tumor": "pituitary", "gliomas": "glioma", "meningiomas": "meningioma",
           "pituitary_tumour": "pituitary", "glioma_tumour": "glioma", "meningioma_tumour": "meningioma"}

def list_external(root):
    items = []
    root = Path(root)
    if not root.exists():
        return items
    for folder in sorted(root.iterdir()):
        cls = ALIASES.get(folder.name.lower().replace(" ", "_").replace("-", "_"))
        if folder.is_dir() and cls:
            for f in sorted(list(folder.glob("*.png")) + list(folder.glob("*.jpg")) + list(folder.glob("*.jpeg"))):
                items.append((str(f), CLASS2IDX[cls]))
    return items

def list_external_recursive(root):
    """Any sub-folder whose name is a tumour class (any depth, any case, .png/.jpg/.jpeg)."""
    items, root = [], Path(root)
    if not root.exists():
        return items
    for folder in sorted(d for d in root.rglob("*") if d.is_dir()):
        cls = ALIASES.get(folder.name.lower().strip().replace(" ", "_").replace("-", "_"))
        if cls:
            for f in sorted(folder.iterdir()):
                if f.suffix.lower() in (".png", ".jpg", ".jpeg"):
                    items.append((str(f), CLASS2IDX[cls]))
    return items

BRISC_ITEMS = list_external(EXT_DIR_BRISC)
BD_ITEMS = list_external_recursive(BDNEURO_DIR)
MSOUD_TRAIN = list_external_recursive(MSOUD_DIR / "Training") if (MSOUD_DIR / "Training").exists() else []
MSOUD_TEST = list_external_recursive(MSOUD_DIR / "Testing") if (MSOUD_DIR / "Testing").exists() else []
BFREE_ITEMS, BFREE_META = [], pd.DataFrame()
if (BRATS_FREE_DIR / "manifest_free.csv").exists():
    BFREE_META = pd.read_csv(BRATS_FREE_DIR / "manifest_free.csv")
    BFREE_META["path"] = [str(BRATS_FREE_DIR / f) for f in BFREE_META["file"]]
    BFREE_META = BFREE_META[BFREE_META.path.map(lambda q: Path(q).exists())].reset_index(drop=True)
    BFREE_ITEMS = list(zip(BFREE_META.path, BFREE_META.patient_tumour_type.map(CLASS2IDX)))   # label = patient's tumour type (reference only)
MEND_ITEMS = list_external(EXT_DIR_MENDELEY)

BRATS_ITEMS, BRATS_META = [], pd.DataFrame()
if (BRATS_DIR / "manifest.csv").exists():
    BRATS_META = pd.read_csv(BRATS_DIR / "manifest.csv")
    BRATS_META["path"] = [str(BRATS_DIR / f) for f in BRATS_META["file"]]
    BRATS_META["y"] = BRATS_META["label"].map(CLASS2IDX)
    BRATS_META = BRATS_META[BRATS_META.path.map(lambda p: Path(p).exists())]
    # drop slices with a tiny tumour core (< 100 pixels of 240x240): not a fair "tumour visible" test
    BRATS_META = BRATS_META[BRATS_META.tc_pixels >= BRATS_MIN_TC_PIXELS].reset_index(drop=True)
    BRATS_ITEMS = list(zip(BRATS_META.path, BRATS_META.y))

BRISC_MASKS = {}
if BRISC_MASK_DIR.exists():
    for m in BRISC_MASK_DIR.rglob("*"):
        if m.suffix.lower() in (".png", ".jpg", ".jpeg"):
            BRISC_MASKS[m.stem.replace("_mask", "")] = str(m)

print(f"BraTS tumour-free slices: {len(BFREE_ITEMS)} (from {BFREE_META.patient_id.nunique() if len(BFREE_META) else 0} patients)")
for name, items in [("BRISC", BRISC_ITEMS), ("Mendeley", MEND_ITEMS), ("BraTS", BRATS_ITEMS), ("BDNeuro", BD_ITEMS),
                    ("Masoud-train", MSOUD_TRAIN), ("Masoud-test", MSOUD_TEST)]:
    c = Counter(IDX2CLASS[y] for _, y in items)
    print(f"{name:9s}: {len(items):5d} tumour images {dict(c)}")
print(f"BRISC masks indexed: {len(BRISC_MASKS)}")

In [ ]:
# ─── 1.4 Duplicate / overlap audit (Reviewer 2) ───
# Perceptual hash (flip-invariant) finds candidates (Hamming <= 40 of 256 bits);
# a 64x64 pixel correlation (flip-invariant) confirms them: duplicate = correlation >= 0.95.
# Pairs with 0.85 <= r < 0.95 are shown for a manual check (adjacent slices / crops).
import imagehash

HAM_CAND, CORR_THR = 40, 0.95

def fingerprint(path):
    g = Image.open(path).convert("L")
    h1 = np.packbits(imagehash.phash(g.resize((256, 256)), hash_size=16).hash.ravel())
    h2 = np.packbits(imagehash.phash(g.transpose(Image.FLIP_LEFT_RIGHT).resize((256, 256)), hash_size=16).hash.ravel())
    small = np.asarray(g.resize((64, 64), Image.BILINEAR), np.float32).ravel()
    small = (small - small.mean()) / (small.std() + 1e-6)
    return h1, h2, small

POPCOUNT = np.array([bin(i).count("1") for i in range(256)], np.uint8)

def fingerprints(items, cache_name):
    path = OUT_DIR / "preds" / f"fp_{cache_name}.npz"
    if path.exists():
        z = np.load(path)
        if len(z["h"]) == len(items):
            print(f"  {cache_name}: fingerprints loaded from cache ({len(items)} images)")
            return z["h"], z["hf"], z["s"]
    try:
        with Pool(max(1, os.cpu_count() or 1)) as pool:      # all CPU cores
            fps = list(tqdm(pool.imap(fingerprint, [p for p, _ in items], chunksize=32), total=len(items),
                            desc=f"  Fingerprinting {cache_name}"))
    except Exception as e:                                   # fallback: one core
        print("  parallel failed, using one core:", e)
        fps = [fingerprint(p) for p, _ in tqdm(items, desc=f"  Fingerprinting {cache_name}")]
    h = np.stack([a for a, _, _ in fps]); hf = np.stack([b for _, b, _ in fps]); s = np.stack([c for _, _, c in fps])
    np.savez_compressed(path, h=h, hf=hf, s=s)
    return h, hf, s

def audit(src_items, src_fp, tgt_items, tgt_fp, src_name, tgt_name):
    hs, _, ss = src_fp; ht, htf, st = tgt_fp
    rows = []
    for j in tqdm(range(len(tgt_items)), desc=f"  Comparing {tgt_name} vs {src_name}"):
        ham = np.minimum(POPCOUNT[np.bitwise_xor(hs, ht[j])].sum(1), POPCOUNT[np.bitwise_xor(hs, htf[j])].sum(1))
        i = int(ham.argmin()); best = int(ham[i])
        corr = np.nan
        if best <= HAM_CAND:
            cands = np.where(ham <= HAM_CAND)[0]
            flip = st[j].reshape(64, 64)[:, ::-1].ravel()
            cs = np.maximum(ss[cands] @ st[j], ss[cands] @ flip) / st.shape[1]
            i = int(cands[cs.argmax()]); corr = float(cs.max())
        dup = best <= HAM_CAND and corr >= CORR_THR
        rows.append({"target": tgt_name, "source": src_name, "tgt_path": tgt_items[j][0],
                     "tgt_class": IDX2CLASS[tgt_items[j][1]], "min_hamming": best, "corr": corr,
                     "match_path": src_items[i][0], "duplicate": bool(dup)})
    return pd.DataFrame(rows)

banner("1.4 Duplicate audit", "First run: ~5-15 min (reads ~15,000 images). Later runs: seconds (cached).")
overlap_csv = OUT_DIR / "overlap_audit.csv"
if overlap_csv.exists() and overlap_csv.stat().st_size > 10 and set(pd.read_csv(overlap_csv).tgt_path) == {p for p, _ in BRISC_ITEMS + MEND_ITEMS + BD_ITEMS + MSOUD_TRAIN + MSOUD_TEST}:
    df_overlap = pd.read_csv(overlap_csv)
else:
    t0 = time.time()
    FIG_ITEMS_ALL = list(zip(df3.filepath, df3.label))          # all 3,064 Figshare tumour images
    fp_fig = fingerprints(FIG_ITEMS_ALL, "figshare")
    fp_brisc = fingerprints(BRISC_ITEMS, "brisc") if BRISC_ITEMS else None
    fp_mend = fingerprints(MEND_ITEMS, "mendeley") if MEND_ITEMS else None
    parts = []
    if BRISC_ITEMS: parts.append(audit(FIG_ITEMS_ALL, fp_fig, BRISC_ITEMS, fp_brisc, "Figshare", "BRISC"))
    if MEND_ITEMS:  parts.append(audit(FIG_ITEMS_ALL, fp_fig, MEND_ITEMS, fp_mend, "Figshare", "Mendeley"))
    if BRISC_ITEMS and MEND_ITEMS:
        parts.append(audit(BRISC_ITEMS, fp_brisc, MEND_ITEMS, fp_mend, "BRISC", "Mendeley"))
    if BD_ITEMS:                                              # the new hospital must not contain known images
        fp_bd = fingerprints(BD_ITEMS, "bdneuro")
        parts.append(audit(FIG_ITEMS_ALL, fp_fig, BD_ITEMS, fp_bd, "Figshare", "BDNeuro"))
        if BRISC_ITEMS: parts.append(audit(BRISC_ITEMS, fp_brisc, BD_ITEMS, fp_bd, "BRISC", "BDNeuro"))
        if MEND_ITEMS:  parts.append(audit(MEND_ITEMS, fp_mend, BD_ITEMS, fp_bd, "Mendeley", "BDNeuro"))
    if MSOUD_TRAIN and MSOUD_TEST:                            # the most-used Kaggle brain-tumour dataset
        fp_mtr = fingerprints(MSOUD_TRAIN, "msoud_train"); fp_mte = fingerprints(MSOUD_TEST, "msoud_test")
        parts.append(audit(FIG_ITEMS_ALL, fp_fig, MSOUD_TRAIN, fp_mtr, "Figshare", "Masoud-train"))
        parts.append(audit(FIG_ITEMS_ALL, fp_fig, MSOUD_TEST, fp_mte, "Figshare", "Masoud-test"))
        parts.append(audit(MSOUD_TRAIN, fp_mtr, MSOUD_TEST, fp_mte, "Masoud-train", "Masoud-test"))  # train/test leakage
    df_overlap = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    df_overlap.to_csv(overlap_csv, index=False)
    print(f"Audit done in {(time.time()-t0)/60:.1f} min")

if len(df_overlap):
    split_of = dict(zip(df3.filepath, np.where(df3.role == "test", "test", "train/val")))
    df_overlap["match_split"] = df_overlap.match_path.map(split_of).fillna("-")
    tab = (df_overlap.groupby(["target", "source", "tgt_class"]).duplicate.agg(["sum", "size"])
           .assign(pct=lambda d: (100 * d["sum"] / d["size"]).round(1)))
    print("Share of each dataset that is a near-copy of another dataset (pct = %):")
    print(tab); tab.to_csv(OUT_DIR / "tables" / "table_overlap.csv")
    print("\nFigshare split of matched images (BRISC/Mendeley duplicates):")
    print(df_overlap[df_overlap.duplicate & (df_overlap.source == "Figshare")]
          .groupby(["target", "match_split"]).size())

def clean_items(items, target):
    """Items of `target` that are NOT duplicates of Figshare (and, for Mendeley, not of BRISC)."""
    if not len(df_overlap):
        return list(items)
    bad = set(df_overlap[(df_overlap.target == target) & df_overlap.duplicate].tgt_path)
    return [(p, y) for p, y in items if p not in bad]

BRISC_CLEAN = clean_items(BRISC_ITEMS, "BRISC")
MEND_CLEAN = clean_items(MEND_ITEMS, "Mendeley")
print(f"\nBRISC: {len(BRISC_ITEMS)} -> {len(BRISC_CLEAN)} after deduplication")
print(f"Mendeley: {len(MEND_ITEMS)} -> {len(MEND_CLEAN)} after deduplication")
BD_CLEAN = clean_items(BD_ITEMS, "BDNeuro")
print(f"BDNeuro: {len(BD_ITEMS)} -> {len(BD_CLEAN)} after deduplication  "
      f"(class counts {dict(Counter(IDX2CLASS[y] for _, y in BD_CLEAN))})")
if MSOUD_TEST:
    mt = df_overlap[df_overlap.target == "Masoud-test"]
    for src in ["Figshare", "Masoud-train"]:
        d = mt[mt.source == src]
        print(f"Masoud TEST images that are near-copies of {src}: {int(d.duplicate.sum())} / {len(d)} "
              f"({100 * d.duplicate.mean():.1f}%)")
if BD_ITEMS and len(BD_CLEAN) < 0.8 * len(BD_ITEMS):
    print("  WARNING: BDNeuro contains many copies of public data -> it is evaluated on its unique images only.")

In [ ]:
# ─── 1.5 Visual check of the audit (send these pairs to reviewers / supplement) ───
def show_pairs(df, title, fname, n=8, seed=0):
    if len(df) == 0:
        print("no pairs for", title); return
    df = df.sample(min(n, len(df)), random_state=seed)
    fig, ax = plt.subplots(2, len(df), figsize=(1.6 * len(df), 3.4))
    ax = np.array(ax).reshape(2, -1)
    for c, r in enumerate(df.itertuples()):
        ax[0, c].imshow(Image.open(r.tgt_path).convert("L"), cmap="gray")
        ax[1, c].imshow(Image.open(r.match_path).convert("L"), cmap="gray")
        ax[0, c].set_title(f"{r.target}\nH={r.min_hamming} r={r.corr:.2f}", fontsize=6)
        ax[1, c].set_title(r.source, fontsize=6)
        for a in ax[:, c]: a.axis("off")
    fig.suptitle(title, fontsize=9); fig.tight_layout()
    fig.savefig(FIG_DIR / fname); plt.show()

def pair_sheet(df, title, fname, n=16, seed=0):
    """n random flagged pairs in 2 rows of n/2: each pair = dataset image (top) + its Figshare match (below)."""
    if len(df) == 0:
        print("  no pairs for", title); return
    df = df.sample(min(n, len(df)), random_state=seed); cols = min(8, len(df)); rows = 2 * int(np.ceil(len(df) / cols))
    fig, ax = plt.subplots(rows, cols, figsize=(1.55 * cols, 1.7 * rows), squeeze=False)
    for a in ax.ravel(): a.axis("off")
    for k, r in enumerate(df.itertuples()):
        rr, c = 2 * (k // cols), k % cols
        ax[rr, c].imshow(Image.open(r.tgt_path).convert("L"), cmap="gray")
        ax[rr + 1, c].imshow(Image.open(r.match_path).convert("L"), cmap="gray")
        ax[rr, c].set_title(f"{r.target} ({r.tgt_class[:4]})", fontsize=6)
        ax[rr + 1, c].set_title(f"{r.source}  r={r.corr:.3f}", fontsize=6)
    fig.suptitle(title, fontsize=9); fig.tight_layout(); fig.savefig(FIG_DIR / fname, bbox_inches="tight"); plt.show()

if len(df_overlap):
    for tgt in df_overlap.target.unique():
        src = "Figshare" if tgt != "Masoud-test" else "Masoud-train"
        d = df_overlap[(df_overlap.target == tgt) & (df_overlap.source == src) & df_overlap.duplicate]
        pair_sheet(d, f"{tgt}: 16 random images flagged as near-copies of {src} (pairs: top = {tgt}, below = match)",
                   f"fig_pairs_{tgt.lower().replace('-', '_')}.png")
    border = df_overlap[(~df_overlap.duplicate) & (df_overlap["corr"].between(0.85, CORR_THR))]
    pair_sheet(border, "Borderline pairs (0.85 <= r < 0.95) - NOT counted as copies (check visually)", "fig_pairs_borderline.png")

In [ ]:
# ─── 1.6 Tumour masks for every Figshare slice (from the ORIGINAL .mat files) ───
# The PNG dataset has no masks. Each PNG is matched to its .mat file by pixel content
# (correlation on 64x64 thumbnails, within the same tumour class), the orientation is detected
# automatically, and the mask is saved at 224x224. Cached: later runs take 1 second.
MASKS = {}                                          # PNG path -> bool (224, 224)
mask_cache = OUT_DIR / "figshare_masks_224.npz"
MAT_LABEL = {1: "meningioma", 2: "glioma", 3: "pituitary"}

def png224(path):                                   # same pixels as load_u8 (defined in 2.4)
    with Image.open(path) as im:
        return np.asarray(im.convert("L").resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC), np.uint8)

def read_mat(path):
    try:
        import h5py
        with h5py.File(path, "r") as h:
            cj = h["cjdata"]
            lab = int(np.array(cj["label"]).squeeze())
            pid = "".join(chr(int(c)) for c in np.array(cj["PID"]).ravel())
            return lab, pid, np.array(cj["image"]).T.astype(np.float32), np.array(cj["tumorMask"]).T > 0
    except OSError:                                 # old (non-HDF5) .mat version
        from scipy.io import loadmat
        cj = loadmat(path)["cjdata"][0, 0]
        return (int(cj["label"].squeeze()), str(cj["PID"].squeeze()),
                cj["image"].astype(np.float32), cj["tumorMask"] > 0)

def thumb(a, s=64):
    t = np.asarray(Image.fromarray(np.asarray(a, np.float32)).resize((s, s), Image.BILINEAR), np.float64).ravel()
    return (t - t.mean()) / (t.std() + 1e-8)

DIHEDRAL = [lambda a: a, np.fliplr, np.flipud, lambda a: np.rot90(a, 1), lambda a: np.rot90(a, 2),
            lambda a: np.rot90(a, 3), np.transpose, lambda a: np.rot90(a.T, 2)]

if not mask_cache.exists() and MASK_NPZ_INPUT is not None:
    shutil.copy2(MASK_NPZ_INPUT, mask_cache)              # uploaded mask file: skips ~25 min of .mat reading
if mask_cache.exists():
    z = np.load(mask_cache, allow_pickle=False)
    bits = np.unpackbits(z["bits"], axis=1)[:, :IMG_SIZE * IMG_SIZE].reshape(-1, IMG_SIZE, IMG_SIZE).astype(bool)
    stems = list(z["stems"])
    by_stem = {Path(p).stem: p for p in df3.filepath}
    MASKS = {by_stem[s]: b for s, b in zip(stems, bits) if s in by_stem}
    print(f"Tumour masks loaded from cache: {len(MASKS)} of {len(df3)} Figshare slices")
elif any(FIGSHARE_MAT_DIR.glob("brainTumorDataPublic_*")):
    banner("1.6 Matching PNG slices to the original .mat files (tumour masks)", "first run only: ~3-6 min")
    mats = sorted(FIGSHARE_MAT_DIR.glob("brainTumorDataPublic_*/*.mat"))
    mat_rows = []
    for f in tqdm(mats, desc="  Reading .mat files"):
        lab, pid, img, mk = read_mat(f)
        mat_rows.append((f.name, MAT_LABEL.get(lab), pid, img, mk))
    # 1) orientation: which of 8 flips/rotations makes the .mat images look like the PNGs?
    sample = df3.sample(min(60, len(df3)), random_state=0)
    png_t = {p: thumb(png224(p)) for p in sample.filepath}
    scores = []
    for op in DIHEDRAL:
        mt = np.stack([thumb(op(r[3])) for r in mat_rows[::5]])
        scores.append(np.mean([np.max(mt @ png_t[p]) / png_t[p].size for p in png_t]))
    best_op = int(np.argmax(scores)); op = DIHEDRAL[best_op]
    print(f"  orientation: transform #{best_op} (mean best correlation {scores[best_op]:.3f})")
    # 2) match every PNG to the best .mat of the same class
    n_ok, corrs, pid_agree = 0, [], []
    for cls in CLASSES:
        rows_c = [r for r in mat_rows if r[1] == cls]
        if not rows_c:
            continue
        M = np.stack([thumb(op(r[3])) for r in rows_c])
        sub = df3[df3.label4 == cls]
        Pt = np.stack([thumb(png224(p)) for p in tqdm(sub.filepath, desc=f"  Matching {cls}", leave=False)])
        C = Pt @ M.T / Pt.shape[1]
        best = C.argmax(1)
        for (p, pid), j, c in zip(zip(sub.filepath, sub.pid), best, C[np.arange(len(best)), best]):
            corrs.append(c)
            if c >= 0.95:
                mk = np.asarray(Image.fromarray(op(rows_c[j][4]).astype(np.uint8) * 255)
                                .resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)) > 127
                MASKS[p] = mk; n_ok += 1
                pid_agree.append(str(rows_c[j][2]).strip() == str(pid))
    corrs = np.array(corrs)
    print(f"  matched {n_ok} of {len(df3)} slices (correlation >= 0.95); median r = {np.median(corrs):.4f}; "
          f"patient ID agrees in {100 * np.mean(pid_agree):.1f}% of matches")
    stems = np.array([Path(p).stem for p in MASKS])
    bits = np.packbits(np.stack(list(MASKS.values())).reshape(len(MASKS), -1), axis=1)
    np.savez_compressed(mask_cache, stems=stems, bits=bits)
else:
    print("Figshare .mat files not found -> CTR-Net cannot be trained (add the 'figshare-original-mat' input).")

if MASKS:
    cover = pd.Series({c: np.mean([p in MASKS for p in df3[df3.label4 == c].filepath]) for c in CLASSES})
    print("  mask coverage per class (%):", (100 * cover).round(1).to_dict())
    area = pd.Series({c: np.mean([MASKS[p].mean() for p in df3[df3.label4 == c].filepath if p in MASKS]) for c in CLASSES})
    print("  mean tumour area (% of image):", (100 * area).round(2).to_dict())
    rng_m = np.random.default_rng(1)
    show = [p for c in CLASSES for p in rng_m.choice([q for q in df3[df3.label4 == c].filepath if q in MASKS], 2, replace=False)]
    fig, ax = plt.subplots(1, len(show), figsize=(1.7 * len(show), 2))
    for a, p in zip(ax, show):
        a.imshow(png224(p), cmap="gray"); a.contour(MASKS[p], levels=[0.5], colors="lime", linewidths=0.8)
        a.set_title(Path(p).parent.name, fontsize=7); a.axis("off")
    fig.suptitle("Check: expert tumour outline from the .mat file on the PNG slice (must sit on the tumour)", fontsize=8)
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig_mask_check.png"); plt.show()

In [ ]:
# ─── 1.7 Figure: how much of the "external" data is really Figshare? ───
if len(df_overlap):
    leak = (df_overlap.groupby(["target", "source", "tgt_class"]).duplicate.mean() * 100).reset_index()
    pairs_ = [(t, "Figshare") for t in ["BRISC", "Mendeley", "BDNeuro"] if (leak.target == t).any()]
    fig, ax = plt.subplots(1, len(pairs_), figsize=(4.5 * len(pairs_), 3.2), squeeze=False); ax = ax[0]
    for a, (tgt, src) in zip(ax, pairs_):
        d = leak[(leak.target == tgt) & (leak.source == src)].set_index("tgt_class").reindex(CLASSES)
        bars = a.bar(CLASSES, d.duplicate, color=[CLASS_COLORS[c] for c in CLASSES])
        for b, v in zip(bars, d.duplicate):
            a.text(b.get_x() + b.get_width() / 2, v + 1, f"{v:.0f}%", ha="center", fontsize=8)
        a.set(ylim=(0, 110), ylabel="% of images that are near-duplicates", title=f"{tgt} images found in {src}")
    n_all = {"BRISC": len(BRISC_ITEMS), "Mendeley": len(MEND_ITEMS), "BDNeuro": len(BD_ITEMS)}
    n_cln = {"BRISC": len(BRISC_CLEAN), "Mendeley": len(MEND_CLEAN), "BDNeuro": len(BD_CLEAN)}
    n_all = {k: v for k, v in n_all.items() if v}
    fig.suptitle("Cross-dataset duplication. Unique images left: " +
                 ", ".join(f"{k} {n_cln[k]}/{n_all[k]}" for k in n_all), fontsize=9)
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig_duplicate_leakage.png"); plt.show()

In [ ]:
# ─── 1.8 Tumour-hiding versions of every slice (used for CTR training AND for the tumour-reliance test) ───
# region  = expert mask + small margin
# inpaint = tumour region filled from the surrounding tissue          (used in CTR TRAINING)
# swap    = tumour region replaced by a same-shape patch of normal brain from the same slice (TEST ONLY:
#           a different hiding method than in training, so CTR cannot pass the test by learning our trick)
# random  = a same-shape region elsewhere in the brain filled (control: should change nothing)
# skull   = brain interior removed, only skull outline + slice position left
# tonly   = only the tumour region kept (supplementary)
import cv2

def brain_mask(u8):
    _, th = cv2.threshold(cv2.GaussianBlur(u8, (5, 5), 0), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8))
    n, lab, st, _ = cv2.connectedComponentsWithStats(th)
    if n <= 1:
        return th > 0
    m = (lab == 1 + np.argmax(st[1:, cv2.CC_STAT_AREA])).astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((15, 15), np.uint8))
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    filled = np.zeros_like(m); cv2.drawContours(filled, cnts, -1, 1, -1)
    return filled > 0

HIDE_KER = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))

def shifted_region(region, bm, rng, tries=200):
    """Offset (dy, dx) that moves `region` to a place inside the brain that does not touch the tumour."""
    ys, xs = np.where(region)
    if len(ys) == 0:
        return None
    near = cv2.dilate(region.astype(np.uint8), HIDE_KER, iterations=2).astype(bool)
    for _ in range(tries):
        dy, dx = (int(v) for v in rng.integers(-IMG_SIZE // 2, IMG_SIZE // 2, 2))
        y2, x2 = ys + dy, xs + dx
        if y2.min() < 0 or x2.min() < 0 or y2.max() >= IMG_SIZE or x2.max() >= IMG_SIZE:
            continue
        r = np.zeros_like(region); r[y2, x2] = True
        if not (r & near).any() and bm[r].mean() >= 0.95:
            return dy, dx, r
    return None

def hide_versions(img, mk, rng):
    reg = cv2.dilate(mk.astype(np.uint8), HIDE_KER).astype(bool)
    bm = brain_mask(img)
    inp = cv2.inpaint(img, reg.astype(np.uint8) * 255, 7, cv2.INPAINT_TELEA)
    sh = shifted_region(reg, bm, rng)
    if sh is not None:
        dy, dx, rr = sh
        ys, xs = np.where(reg)
        swap = img.copy(); swap[ys, xs] = img[ys + dy, xs + dx]
        rnd = cv2.inpaint(img, rr.astype(np.uint8) * 255, 7, cv2.INPAINT_TELEA)
    else:                                     # tumour too large to move: control not available for this slice
        swap, rnd = inp, img
    interior = cv2.erode(bm.astype(np.uint8), np.ones((17, 17), np.uint8)).astype(bool)
    skull = img.copy(); skull[interior | reg] = 0
    return {"region": reg, "inpaint": inp, "swap": swap, "random": rnd, "skull": skull,
            "tonly": np.where(reg, img, 0).astype(np.uint8), "has_control": sh is not None}

HIDE = {}
if MASKS:
    banner("1.8 Building tumour-hidden versions of every slice", f"{len(MASKS)} slices (~1-2 min)")
    rng_h = np.random.default_rng(SPLIT_SEED)
    for p in tqdm(sorted(MASKS), desc="  building", mininterval=5):
        HIDE[p] = hide_versions(png224(p), MASKS[p], rng_h)
    print(f"  done: {len(HIDE)} slices | random-spot control possible for "
          f"{100 * np.mean([v['has_control'] for v in HIDE.values()]):.1f}%")
    rng_s = np.random.default_rng(3)
    show = [rng_s.choice([p for p in df3[df3.label4 == c].filepath if p in HIDE and HIDE[p]["has_control"]]) for c in CLASSES]
    cols = [("Original", None), ("Tumour hidden\n(inpaint, training)", "inpaint"), ("Tumour hidden\n(swap, test)", "swap"),
            ("Random spot hidden\n(control)", "random"), ("Skull / position\nonly", "skull"), ("Tumour only", "tonly")]
    fig, ax = plt.subplots(3, len(cols), figsize=(2.1 * len(cols), 6.6))
    for r, p in enumerate(show):
        for c, (title, key) in enumerate(cols):
            ax[r, c].imshow(png224(p) if key is None else HIDE[p][key], cmap="gray", vmin=0, vmax=255)
            if key is None:
                ax[r, c].contour(MASKS[p], levels=[0.5], colors="lime", linewidths=0.7)
            ax[r, c].axis("off")
            if r == 0: ax[r, c].set_title(title, fontsize=8)
        ax[r, 0].text(-10, IMG_SIZE / 2, CLASSES[r], rotation=90, va="center", ha="right", fontsize=9)
    fig.suptitle("Tumour-hiding test: versions of the same slice (green = expert tumour outline)", fontsize=9)
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig_hiding_versions.png", bbox_inches="tight"); plt.show()

In [ ]:
# ─── 1.9 Slice plane (axial / coronal / sagittal) - semi-automatic labelling ───
# Figshare slices come in 3 viewing planes. A pretrained image model groups the slices into 12 visual clusters;
# the plane of each cluster was read from the Run 4 sheet (PLANE_BY_SIZE in cell 0.2).
# "mixed" clusters are left out of the plane analysis.
from sklearn.cluster import KMeans
PLANE_OF = {}
plane_csv = OUT_DIR / "plane_clusters.csv"
if plane_csv.exists() and len(pd.read_csv(plane_csv)) == len(df3):
    df_plane = pd.read_csv(plane_csv)
else:
    banner("1.9 Grouping slices by viewing plane", "pretrained image features + 12 clusters (~2 min)")
    fe = timm.create_model("convnext_tiny", pretrained=True, num_classes=0).to(DEVICE).eval()
    feats = []
    with torch.no_grad():
        arrs_all = [png224(p) for p in df3.filepath]
        for i in range(0, len(arrs_all), 64):
            x = torch.from_numpy(np.stack(arrs_all[i:i + 64])).to(DEVICE).float().div(255).unsqueeze(1).repeat(1, 3, 1, 1)
            m_ = torch.tensor(IMAGENET_MEAN, device=DEVICE).view(1, 3, 1, 1); s_ = torch.tensor(IMAGENET_STD, device=DEVICE).view(1, 3, 1, 1)
            feats.append(fe((x - m_) / s_).float().cpu().numpy())
    del fe
    feats = np.concatenate(feats); feats = feats / np.linalg.norm(feats, axis=1, keepdims=True)
    df_plane = pd.DataFrame({"filepath": df3.filepath.values, "cluster": KMeans(12, n_init=10, random_state=0).fit_predict(feats)})
    df_plane.to_csv(plane_csv, index=False)
rng_p = np.random.default_rng(0)
fig, ax = plt.subplots(12, 10, figsize=(10, 12.6))
for c in range(12):
    paths = df_plane[df_plane.cluster == c].filepath.values
    pick = rng_p.choice(paths, min(10, len(paths)), replace=False) if len(paths) else []
    for j in range(10):
        ax[c, j].axis("off")
        if j < len(pick):
            ax[c, j].imshow(png224(pick[j]), cmap="gray")
    ax[c, 0].text(-20, 112, f"cluster {c}\n(n={len(paths)})\n{PLANE_BY_SIZE.get(len(paths), '?')}", ha="right", va="center", fontsize=7)
fig.suptitle("Viewing-plane clusters (each row = one cluster; its plane is printed below the sheet)", fontsize=9)
fig.tight_layout(); fig.savefig(FIG_DIR / "plane_cluster_sheet.png", bbox_inches="tight"); plt.show()
sizes = df_plane.cluster.value_counts().to_dict()                     # cluster -> number of slices
PLANE_MAP = {c: PLANE_BY_SIZE[n] for c, n in sizes.items() if n in PLANE_BY_SIZE}
if len(PLANE_MAP) == 12:
    df_plane["plane"] = df_plane.cluster.map(PLANE_MAP)
    PLANE_OF = dict(zip(df_plane.filepath, df_plane.plane))
    print("✔ Clusters recognised (same sizes as in Run 4). Slices per plane:", df_plane.plane.value_counts().to_dict())
else:
    PLANE_MAP = {}
    print("⚠ The clusters differ from Run 4 (sizes " + str(sorted(sizes.values())) + "). The plane analysis (6.9) is "
          "skipped. Send me this sheet and I will update PLANE_BY_SIZE.")

## 2. Pre-processing, image cache and models

In [ ]:
# ─── 2.1 Normalisation statistics (scratch models: per-fold training images) ───
def compute_fold_stats(fold, max_per_class=200, seed=SPLIT_SEED):
    rng = random.Random(seed)
    by_cls = defaultdict(list)
    for p, y in FOLDS[fold]["train"]:
        by_cls[y].append(p)
    s = s2 = n = 0.0
    for y in sorted(by_cls):
        for p in tqdm(rng.sample(by_cls[y], min(max_per_class, len(by_cls[y]))), desc=f"  fold {fold} stats, class {y}", leave=False):
            a = np.asarray(Image.open(p).convert("L").resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC), np.float64) / 255.0
            s += a.sum(); s2 += (a ** 2).sum(); n += a.size
    mean = s / n
    return {"mean": [float(mean)] * 3, "std": [float(np.sqrt(max(s2 / n - mean ** 2, 1e-8)))] * 3}

fs_path = OUT_DIR / "fold_stats_3class.json"
FOLD_STATS = {int(k): v for k, v in (load_json(fs_path) or {}).items()}
if len(FOLD_STATS) != N_FOLDS:
    FOLD_STATS = {k: compute_fold_stats(k) for k in range(1, N_FOLDS + 1)}
    save_json(FOLD_STATS, fs_path)
for k, v in FOLD_STATS.items():
    print(f"fold {k}: mean {v['mean'][0]:.4f}  std {v['std'][0]:.4f}")

In [ ]:
# ─── 2.2 Model registry ───
MODEL_SPECS = {
    # --- NEW (Run 5): solutions and controls ---
    "ROI-Loc":        dict(kind="roi", crop="box", loc=True),        # SOLUTION 1: crop + location numbers
    "CNX-Transplant": dict(kind="guide", seg=False, pool=False, cf=False, guide=None, transplant=True),  # SOLUTION 2
    "CNX-Strip":      dict(kind="guide", seg=False, pool=False, cf=False, guide=None, strip=True),       # CONTROL
    "ROI-Crop-m06":   dict(kind="roi", crop="box", margin=0.6),      # context sweep
    "ROI-Crop-m15":   dict(kind="roi", crop="box", margin=1.5),      # context sweep
    "ROI-m06-Transplant": dict(kind="roi", crop="box", margin=0.6, transplant=True),   # Run 6: context + transplant
    # --- NEW (Run 4): structural remedy. Same backbone; the input is only the region found by MTL-Seg's tumour map ---
    "ROI-Crop":     dict(kind="roi", crop="box"),      # box around the found tumour (+20% context)
    "ROI-Mask":     dict(kind="roi", crop="mask"),     # same box, everything outside the found tumour blacked out
    # --- NEW (Run 3): all on a pretrained ConvNeXt-Tiny backbone, same GPU training pipeline ("guide" kind) ---
    "CTR-Net":      dict(kind="guide", seg=True,  pool=True,  cf=True,  guide=None),     # PROPOSED
    "CTR-noCF":     dict(kind="guide", seg=True,  pool=True,  cf=False, guide=None),
    "CTR-noSeg":    dict(kind="guide", seg=False, pool=False, cf=True,  guide=None),
    "CNX-Plain":    dict(kind="guide", seg=False, pool=False, cf=False, guide=None),
    "MTL-Seg":      dict(kind="guide", seg=True,  pool=False, cf=False, guide=None),
    "RRR":          dict(kind="guide", seg=False, pool=False, cf=False, guide="rrr"),
    "CAM-Guide":    dict(kind="guide", seg=False, pool=False, cf=False, guide="cam"),
    "CTR-MambaKAN": dict(kind="tamtk", use_seg=True, use_seq=True, use_kan=True, seg=True, pool=True, cf=True, guide=None),
    # --- Run 1 models (scratch CNN / MTK family and ImageNet baselines) ---
    "MTK":               dict(kind="mtk", use_cnn=True,  use_seq=True,  use_kan=True),
    "MTK-noSeq":         dict(kind="mtk", use_cnn=True,  use_seq=False, use_kan=True),
    "MTK-noKAN":         dict(kind="mtk", use_cnn=True,  use_seq=True,  use_kan=False),
    "MTK-noCNN":         dict(kind="mtk", use_cnn=False, use_seq=True,  use_kan=True),
    "CNN-Backbone":      dict(kind="mtk", use_cnn=True,  use_seq=False, use_kan=False),
    "CNN-Backbone-Wide": dict(kind="mtk", use_cnn=True,  use_seq=False, use_kan=False, cnn_hidden=1536),
    "ResNet-50":         dict(kind="timm", arch="resnet50"),
    "EfficientNet-B0":   dict(kind="timm", arch="efficientnet_b0"),
    "ConvNeXt-Tiny":     dict(kind="timm", arch="convnext_tiny"),
    "DeiT-Small":        dict(kind="timm", arch="deit_small_patch16_224"),
}
MODEL_ORDER = list(MODEL_SPECS)
PROPOSED = "ROI-Loc"     # pre-registered main solution (falls back to ROI-Crop if not trained yet, see 3.4)
ABLATIONS = ["MTK-noSeq", "MTK-noKAN", "MTK-noCNN", "CNN-Backbone", "CNN-Backbone-Wide"]
CTR_FAMILY = ["CTR-Net", "CTR-noCF", "CTR-noSeg", "CTR-MambaKAN", "MTL-Seg", "RRR", "CAM-Guide", "CNX-Plain"]
TA_ABLATIONS = ["CTR-noCF", "CTR-noSeg", "CTR-MambaKAN", "MTL-Seg", "RRR", "CAM-Guide", "CNX-Plain"]
BASELINES = ["CNN-Backbone", "CNN-Backbone-Wide", "ResNet-50", "EfficientNet-B0", "ConvNeXt-Tiny", "DeiT-Small"]
FAMILY_COLOR = {"proposed": "#d1495b", "solution": "#2a9d8f", "control": "#6c757d", "roi": "#f08a8f", "ctr-ablation": "#9b5de5", "guidance": "#edae49",
                "pretrained": "#30638e", "scratch": "#888888"}
FAMILY_LABEL = {"proposed": "proposed solution", "solution": "Run 5 solutions", "control": "control (skull-stripped)",
                "roi": "ROI variants", "ctr-ablation": "CTR (Run 3 remedy) + ablations",
                "guidance": "existing guidance methods", "pretrained": "pretrained baselines", "scratch": "scratch / MTK models"}

def family_of(name):
    if name == PROPOSED: return "proposed"
    if name in ("ROI-Loc", "CNX-Transplant", "ROI-m06-Transplant"): return "solution"
    if name == "CNX-Strip": return "control"
    if name.startswith("ROI"): return "roi"
    if name.startswith("CTR"): return "ctr-ablation"
    if name in ("MTL-Seg", "RRR", "CAM-Guide"): return "guidance"
    if MODEL_SPECS[name]["kind"] in ("timm", "guide", "tamtk"): return "pretrained"
    return "scratch"

def new_pipeline(name):
    """Run-3 models: GPU augmentation, tumour masks and hidden versions available in every batch."""
    return MODEL_SPECS[name]["kind"] in ("guide", "tamtk") and "seg" in MODEL_SPECS[name]

def is_pretrained(name):
    return MODEL_SPECS[name]["kind"] in ("timm", "tamtk", "guide", "roi")

def is_roi(name):
    return MODEL_SPECS[name]["kind"] == "roi"

def uses_masks(name):
    s = MODEL_SPECS[name]
    return bool(s.get("seg") or s.get("cf") or s.get("guide"))

def norm_for(name, fold):
    if is_pretrained(name):
        return IMAGENET_MEAN, IMAGENET_STD
    return FOLD_STATS[fold]["mean"], FOLD_STATS[fold]["std"]

In [ ]:
# ─── 2.3 MedTriMambaKAN v3 (fixed) ───
# Fixes vs the submitted v2:
#  * sequence pathway: the old CLS token (depthwise conv, kernel 5, two blocks) only
#    saw the first 4 tokens of the 14x14 grid (top-left corner). v3 uses a real
#    selective state-space scan (Mamba S6) in 4 directions over a 7x7 token grid
#    (all directions in one time loop) and mean-pools ALL tokens.
#  * KAN pathway: inputs are layer-normalised + tanh so they fall inside the
#    Gaussian RBF centres in [-1, 1] (otherwise the basis is ~0 and the layer is linear).
#  * pathways can be switched off -> ablations use exactly the same code.

class ConvBlock(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, 3, stride, 1, bias=False), nn.BatchNorm2d(cout), nn.GELU(),
            nn.Conv2d(cout, cout, 3, 1, 1, bias=False), nn.BatchNorm2d(cout), nn.GELU())
    def forward(self, x):
        return self.block(x)

class MultiDirSSM(nn.Module):
    """Mamba (S6) selective scan in plain PyTorch, K scan directions processed TOGETHER
    (separate weights per direction, one shared time loop) -> K times fewer sequential steps."""
    def __init__(self, d, k_dirs=4, d_state=16, d_conv=3):
        super().__init__()
        self.K, self.d, self.N, self.r, self.kc = k_dirs, d, d_state, max(1, math.ceil(d / 16)), d_conv
        def w(*shape, fan_in):
            return nn.Parameter(torch.empty(*shape).uniform_(-1, 1) / math.sqrt(fan_in))
        self.W_in = w(k_dirs, d, 2 * d, fan_in=d)
        self.conv_w = w(k_dirs * d, 1, d_conv, fan_in=d_conv)
        self.conv_b = nn.Parameter(torch.zeros(k_dirs * d))
        self.W_x = w(k_dirs, d, self.r + 2 * d_state, fan_in=d)
        self.W_dt = w(k_dirs, self.r, d, fan_in=self.r)
        dt = torch.exp(torch.rand(k_dirs, d) * (math.log(0.1) - math.log(1e-3)) + math.log(1e-3))
        self.b_dt = nn.Parameter(dt + torch.log(-torch.expm1(-dt)))          # standard Mamba dt init
        self.A_log = nn.Parameter(torch.log(torch.arange(1, d_state + 1, dtype=torch.float32)).repeat(k_dirs, d, 1))
        self.D = nn.Parameter(torch.ones(k_dirs, d))
        self.W_out = w(k_dirs, d, d, fan_in=d)

    def forward(self, xs):                                   # xs: (B, K, L, d)
        with torch.autocast(device_type=xs.device.type, enabled=False):
            xs = xs.float(); B, K, L, d = xs.shape
            u, z = torch.einsum("bkld,kde->bkle", xs, self.W_in).chunk(2, dim=-1)
            u = F.conv1d(u.permute(0, 1, 3, 2).reshape(B, K * d, L), self.conv_w, self.conv_b,
                         padding=self.kc - 1, groups=K * d)[..., :L]
            u = F.silu(u.reshape(B, K, d, L).permute(0, 1, 3, 2))
            dt, Bm, Cm = torch.einsum("bkld,kde->bkle", u, self.W_x).split([self.r, self.N, self.N], dim=-1)
            dt = F.softplus(torch.einsum("bklr,krd->bkld", dt, self.W_dt) + self.b_dt[None, :, None])
            dA = torch.exp(dt.unsqueeze(-1) * -torch.exp(self.A_log)[None, :, None])   # (B,K,L,d,N)
            dBu = dt.unsqueeze(-1) * Bm.unsqueeze(3) * u.unsqueeze(-1)
            h = xs.new_zeros(B, K, d, self.N); ys = []
            for t in range(L):
                h = dA[:, :, t] * h + dBu[:, :, t]
                ys.append((h * Cm[:, :, t].unsqueeze(2)).sum(-1))
            y = (torch.stack(ys, 2) + u * self.D[None, :, None]) * F.silu(z)
            return torch.einsum("bkld,kde->bkle", y, self.W_out)

class CrossScanBlock(nn.Module):
    """Pre-norm residual block scanning an HxW token grid in 4 orders (rows fwd/bwd, cols fwd/bwd)."""
    def __init__(self, d, hw):
        super().__init__()
        self.hw = hw
        self.norm = nn.LayerNorm(d)
        self.ssm = MultiDirSSM(d, k_dirs=4)
        self.ffn = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, 2 * d), nn.GELU(), nn.Linear(2 * d, d))

    def forward(self, x):                                    # x: (B, h*w, d)
        h = w = self.hw; B, L, d = x.shape
        xn = self.norm(x)
        col = xn.view(B, h, w, d).transpose(1, 2).reshape(B, L, d)
        ys = self.ssm(torch.stack([xn, xn.flip(1), col, col.flip(1)], dim=1))
        yc = ys[:, 2] + ys[:, 3].flip(1)
        y = ys[:, 0] + ys[:, 1].flip(1) + yc.view(B, w, h, d).transpose(1, 2).reshape(B, L, d)
        x = x + y / 4
        return x + self.ffn(x)

class GaussianKAN(nn.Module):
    """KAN-inspired layer: SiLU base + learnable weights on 5 Gaussian RBFs (centres in [-1, 1])."""
    def __init__(self, din, dout, n_basis=5, width=0.5):
        super().__init__()
        self.base = nn.Linear(din, dout, bias=False)
        self.spline = nn.Parameter(torch.randn(dout, din, n_basis) * 0.1)
        self.register_buffer("centres", torch.linspace(-1, 1, n_basis))
        self.width = width
        self.norm = nn.LayerNorm(dout)
    def forward(self, x):
        basis = torch.exp(-((x.unsqueeze(-1) - self.centres) ** 2) / self.width)
        return self.norm(self.base(F.silu(x)) + torch.einsum("big,oig->bo", basis, self.spline))

class MedTriMambaKAN(nn.Module):
    def __init__(self, num_classes=N_CLASSES, use_cnn=True, use_seq=True, use_kan=True,
                 cnn_hidden=None, fused_dim=128, seq_depth=1):
        super().__init__()
        assert use_cnn or use_seq or use_kan
        self.use = {"cnn": use_cnn, "seq": use_seq, "kan": use_kan}
        self.stem = nn.Sequential(nn.Conv2d(3, 32, 3, 1, 1, bias=False), nn.BatchNorm2d(32), nn.GELU())
        self.stage1, self.stage2 = ConvBlock(32, 64, 2), ConvBlock(64, 128, 2)
        self.stage3, self.stage4 = ConvBlock(128, 192, 2), ConvBlock(192, 256, 2)
        dims = {}
        if use_cnn:
            layers = ([nn.Linear(256, cnn_hidden), nn.GELU(), nn.Dropout(0.2), nn.Linear(cnn_hidden, 128)]
                      if cnn_hidden else [nn.Linear(256, 128)])
            self.cnn_head = nn.Sequential(*layers, nn.GELU(), nn.Dropout(0.2)); dims["cnn"] = 128
        if use_seq:
            # 14x14 -> 7x7 tokens (2x2 average pooling): 49 scan steps instead of 196
            self.seq_reduce = nn.Sequential(nn.Conv2d(256, 128, 1, bias=False), nn.BatchNorm2d(128), nn.AvgPool2d(2))
            self.seq_blocks = nn.Sequential(*[CrossScanBlock(128, 7) for _ in range(seq_depth)])
            self.seq_norm = nn.LayerNorm(128); dims["seq"] = 128
        if use_kan:
            self.kan_in = nn.LayerNorm(128)
            self.kan = nn.Sequential(GaussianKAN(128, 128), GaussianKAN(128, 64)); dims["kan"] = 64
        self.paths = list(dims)
        self.proj = nn.ModuleDict({k: nn.Linear(v, fused_dim) for k, v in dims.items()})
        if len(self.paths) > 1:
            self.gate = nn.Sequential(nn.Linear(fused_dim * len(self.paths), fused_dim), nn.GELU(),
                                      nn.Linear(fused_dim, len(self.paths)))
        self.fuse_norm, self.fuse_drop = nn.LayerNorm(fused_dim), nn.Dropout(0.25)
        if use_cnn:
            self.skip_alpha = nn.Parameter(torch.tensor(0.1))
        self.classifier = nn.Sequential(nn.Linear(fused_dim, 64), nn.GELU(), nn.Dropout(0.3), nn.Linear(64, num_classes))

    def cam_layer(self):
        return self.stage4.block[3]          # last conv of stage 4 (14x14)

    def forward(self, x, return_gate=False, return_embed=False):
        x2 = self.stage2(self.stage1(self.stem(x)))
        x4 = self.stage4(self.stage3(x2))
        feats = {}
        if self.use["cnn"]:
            feats["cnn"] = self.cnn_head(x4.mean((2, 3)))
        if self.use["seq"]:
            tok = self.seq_reduce(x4).flatten(2).transpose(1, 2)       # (B, 49, 128)
            feats["seq"] = self.seq_norm(self.seq_blocks(tok)).mean(1)
        if self.use["kan"]:
            feats["kan"] = self.kan(torch.tanh(self.kan_in(x2.mean((2, 3)))))
        p = [self.proj[k](feats[k]) for k in self.paths]
        if len(p) > 1:
            g = torch.softmax(self.gate(torch.cat(p, 1)), dim=1)
            fused = sum(g[:, i:i + 1] * p[i] for i in range(len(p)))
        else:
            g = torch.ones(x.shape[0], 1, device=x.device); fused = p[0]
        fused = self.fuse_drop(self.fuse_norm(fused))
        if self.use["cnn"]:
            fused = fused + self.skip_alpha * feats["cnn"]
        if return_embed:
            return fused
        logits = self.classifier(fused)
        return (logits, g) if return_gate else logits

class TumourAwareMTK(nn.Module):
    """TA-MTK: pretrained ConvNeXt-Tiny features -> three pathways (CNN, Seq = 4-direction Mamba scan, KAN)
    fused by a softmax gate (same design as MTK), plus a light tumour-mask decoder.
    The predicted tumour map is used for MASK-GUIDED POOLING: every pathway pools
    0.5 x (global average) + 0.5 x (average inside the predicted tumour), so the class decision
    is pushed to come from the tumour region. Without the mask head (use_seg=False) all pooling is global."""
    def __init__(self, num_classes=N_CLASSES, use_seg=True, use_seq=True, use_kan=True, pretrained=True, fused_dim=128):
        super().__init__()
        self.use_seg = use_seg
        self.backbone = timm.create_model("convnext_tiny", pretrained=pretrained, features_only=True, out_indices=(1, 2, 3))
        c1, c2, c3 = self.backbone.feature_info.channels()          # 192 @28x28, 384 @14x14, 768 @7x7
        self.cam_id = nn.Identity()                                  # Grad-CAM++ hook (last stage, 7x7, as ConvNeXt-Tiny)
        self.cam_id2 = nn.Identity()                                 # 14x14 features (LayerCAM explanations)
        if use_seg:
            self.lat3, self.lat2, self.lat1 = nn.Conv2d(c3, 128, 1), nn.Conv2d(c2, 128, 1), nn.Conv2d(c1, 128, 1)
            self.seg_head = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.GELU(),
                                          nn.Conv2d(64, 1, 1))
        dims = {"cnn": 128}
        self.cnn_head = nn.Sequential(nn.Linear(c3, 128), nn.GELU(), nn.Dropout(0.2))
        if use_seq:
            self.seq_reduce = nn.Sequential(nn.Conv2d(c3, 128, 1, bias=False), nn.BatchNorm2d(128))
            self.seq_blocks = CrossScanBlock(128, 7)
            self.seq_norm = nn.LayerNorm(128); dims["seq"] = 128
        if use_kan:
            self.kan_in = nn.Sequential(nn.Linear(c2, 128), nn.LayerNorm(128))
            self.kan = nn.Sequential(GaussianKAN(128, 128), GaussianKAN(128, 64)); dims["kan"] = 64
        self.paths = list(dims)
        self.proj = nn.ModuleDict({k: nn.Linear(v, fused_dim) for k, v in dims.items()})
        if len(self.paths) > 1:
            self.gate = nn.Sequential(nn.Linear(fused_dim * len(self.paths), fused_dim), nn.GELU(),
                                      nn.Linear(fused_dim, len(self.paths)))
        self.fuse_norm, self.fuse_drop = nn.LayerNorm(fused_dim), nn.Dropout(0.25)
        self.skip_alpha = nn.Parameter(torch.tensor(0.1))
        self.classifier = nn.Sequential(nn.Linear(fused_dim, 64), nn.GELU(), nn.Dropout(0.3), nn.Linear(64, num_classes))

    def cam_layer(self):
        return self.cam_id

    def param_groups(self, lr):
        bb = list(self.backbone.parameters()); ids = {id(q) for q in bb}
        heads = [q for q in self.parameters() if id(q) not in ids]
        return [{"params": bb, "base_lr": lr}, {"params": heads, "base_lr": lr * HEAD_LR_MULT}]

    @staticmethod
    def _pool(f, w):                      # f (B,C,H,W), w (B,1,H,W) in [0,1] or None
        g = f.mean((2, 3))
        if w is None:
            return g
        return 0.5 * g + 0.5 * (f * w).sum((2, 3)) / (w.sum((2, 3)) + 1e-4)

    def forward(self, x, return_gate=False, return_seg=False, return_embed=False):
        f1, f2, f3 = self.backbone(x)
        f2 = self.cam_id2(f2); f3 = self.cam_id(f3)
        seg = w3 = w2 = None
        if self.use_seg:
            d = self.lat3(f3)
            d = self.lat2(f2) + F.interpolate(d, size=f2.shape[-2:], mode="bilinear", align_corners=False)
            d = self.lat1(f1) + F.interpolate(d, size=f1.shape[-2:], mode="bilinear", align_corners=False)
            seg = self.seg_head(d)                                    # (B,1,28,28) tumour logits
            prob = torch.sigmoid(seg.float()).to(f3.dtype)
            w3, w2 = F.adaptive_avg_pool2d(prob, f3.shape[-2:]), F.adaptive_avg_pool2d(prob, f2.shape[-2:])
        feats = {"cnn": self.cnn_head(self._pool(f3, w3))}
        if "seq" in self.paths:
            t = self.seq_reduce(f3); B, C, H, W = t.shape
            t = self.seq_norm(self.seq_blocks(t.flatten(2).transpose(1, 2))).transpose(1, 2).reshape(B, C, H, W)
            feats["seq"] = self._pool(t, w3)
        if "kan" in self.paths:
            feats["kan"] = self.kan(torch.tanh(self.kan_in(self._pool(f2, w2))))
        p = [self.proj[k](feats[k]) for k in self.paths]
        if len(p) > 1:
            g = torch.softmax(self.gate(torch.cat(p, 1)), dim=1)
            fused = sum(g[:, i:i + 1] * p[i] for i in range(len(p)))
        else:
            g = torch.ones(x.shape[0], 1, device=x.device); fused = p[0]
        fused = self.fuse_drop(self.fuse_norm(fused)) + self.skip_alpha * feats["cnn"]
        if return_embed:
            return fused
        logits = self.classifier(fused)
        if return_seg:
            return logits, seg
        return (logits, g) if return_gate else logits

class GuidedConvNeXt(nn.Module):
    """ConvNeXt-Tiny backbone + linear head, with OPTIONAL tumour-mask decoder (seg) and
    mask-guided pooling (pool). Used for CTR-Net, its ablations and the guidance baselines."""
    def __init__(self, num_classes=N_CLASSES, seg=False, pool=False, pretrained=True):
        super().__init__()
        self.use_seg, self.use_pool = seg, pool and seg
        self.backbone = timm.create_model("convnext_tiny", pretrained=pretrained, features_only=True, out_indices=(1, 2, 3))
        c1, c2, c3 = self.backbone.feature_info.channels()           # 192 @28, 384 @14, 768 @7
        self.cam_id = nn.Identity()                                   # 7x7 features  (CAM-Guide penalty)
        self.cam_id2 = nn.Identity()                                  # 14x14 features (LayerCAM explanations)
        if seg:
            self.lat3, self.lat2, self.lat1 = nn.Conv2d(c3, 128, 1), nn.Conv2d(c2, 128, 1), nn.Conv2d(c1, 128, 1)
            self.seg_head = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.GELU(),
                                          nn.Conv2d(64, 1, 1))
        self.head_norm = nn.LayerNorm(c3)
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(c3, num_classes))

    def cam_layer(self):
        return self.cam_id2

    def param_groups(self, lr):
        bb = list(self.backbone.parameters()); ids = {id(q) for q in bb}
        return [{"params": bb, "base_lr": lr},
                {"params": [q for q in self.parameters() if id(q) not in ids], "base_lr": lr * HEAD_LR_MULT}]

    def forward(self, x, return_seg=False, return_feat=False, return_embed=False):
        f1, f2, f3 = self.backbone(x)
        f2 = self.cam_id2(f2); f3 = self.cam_id(f3)
        seg = None
        if self.use_seg:
            d = self.lat3(f3)
            d = self.lat2(f2) + F.interpolate(d, size=f2.shape[-2:], mode="bilinear", align_corners=False)
            d = self.lat1(f1) + F.interpolate(d, size=f1.shape[-2:], mode="bilinear", align_corners=False)
            seg = self.seg_head(d)                                    # (B,1,28,28) tumour logits
        g = f3.mean((2, 3))
        if self.use_pool:                                             # 0.5 x global + 0.5 x inside predicted tumour
            w = F.adaptive_avg_pool2d(torch.sigmoid(seg.float()).to(f3.dtype), f3.shape[-2:])
            g = 0.5 * g + 0.5 * (f3 * w).sum((2, 3)) / (w.sum((2, 3)) + 1e-4)
        emb = self.head_norm(g)
        if return_embed:
            return emb
        logits = self.head(emb)
        if return_feat:
            return logits, f3
        if return_seg:
            return logits, seg
        return logits

N_LOC = 6   # location numbers of ROI-Loc (see loc_feats in 2.4)

class RoiLocNet(nn.Module):
    """SOLUTION 1. Two inputs that a radiologist would also use, and nothing else:
    (a) what the tumour looks like = ConvNeXt-Tiny on the ROI crop, (b) WHERE it is relative to the brain =
    6 numbers (vertical position, distance from the midline, distance from the centre, relative area, width, height).
    The whole slice (skull, slice position, plane) is never seen, so it cannot be used as a shortcut."""
    def __init__(self, num_classes=N_CLASSES, pretrained=True):
        super().__init__()
        self.cnn = GuidedConvNeXt(num_classes, seg=False, pool=False, pretrained=pretrained)
        self.loc = nn.Sequential(nn.Linear(N_LOC, 32), nn.GELU(), nn.Linear(32, 32), nn.GELU())
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(768 + 32, num_classes))
        self.loc_override = None          # set to a (N_LOC,) tensor to replace every location (knock-out test)
    def param_groups(self, lr):
        bb = list(self.cnn.backbone.parameters()); ids = {id(q) for q in bb}
        return [{"params": bb, "base_lr": lr},
                {"params": [q for q in self.parameters() if id(q) not in ids], "base_lr": lr * HEAD_LR_MULT}]
    def cam_layer(self):
        return self.cnn.cam_id2
    def forward(self, x, loc=None, return_embed=False):
        emb = self.cnn(x, return_embed=True)
        if loc is None:
            loc = torch.zeros(x.shape[0], N_LOC, device=x.device, dtype=emb.dtype)
        if self.loc_override is not None:
            loc = self.loc_override.to(x.device, emb.dtype).expand(x.shape[0], -1)
        z = torch.cat([emb, self.loc(loc.to(emb.dtype))], 1)
        return z if return_embed else self.head(z)

def build_model(name, pretrained=True, num_classes=N_CLASSES):
    spec = MODEL_SPECS[name]
    if spec["kind"] == "roi" and spec.get("loc"):
        return RoiLocNet(num_classes, pretrained=pretrained)
    if spec["kind"] == "guide":
        return GuidedConvNeXt(num_classes, spec["seg"], spec["pool"], pretrained=pretrained)
    if spec["kind"] == "roi":
        return GuidedConvNeXt(num_classes, seg=False, pool=False, pretrained=pretrained)
    if spec["kind"] == "mtk":
        return MedTriMambaKAN(num_classes, spec["use_cnn"], spec["use_seq"], spec["use_kan"], spec.get("cnn_hidden"))
    if spec["kind"] == "tamtk":
        return TumourAwareMTK(num_classes, spec["use_seg"], spec["use_seq"], spec["use_kan"], pretrained=pretrained)
    return timm.create_model(spec["arch"], pretrained=pretrained, num_classes=num_classes)

def embed(model, x):
    """Penultimate feature vector (used for the t-SNE figure)."""
    m = model.module if isinstance(model, nn.DataParallel) else model
    if isinstance(m, (MedTriMambaKAN, TumourAwareMTK, GuidedConvNeXt)):
        return m(x, return_embed=True)
    return m.forward_head(m.forward_features(x), pre_logits=True)

def cam_target_layer(model, name):
    m = model.module if isinstance(model, nn.DataParallel) else model
    # all explanation layers have a 14x14 grid (stride 16) -> sharper and comparable across models
    if MODEL_SPECS[name]["kind"] in ("mtk", "guide", "roi"):
        return m.cam_layer()
    if MODEL_SPECS[name]["kind"] == "tamtk":
        return m.cam_id2
    arch = MODEL_SPECS[name]["arch"]
    if arch.startswith("resnet"):       return m.layer3[-1]
    if arch.startswith("efficientnet"): return m.blocks[4]
    if arch.startswith("convnext"):     return m.stages[2]
    return None                          # DeiT: token maps -> not used for heat maps

# Parameter / FLOP / latency table
banner("2.3 Model size / speed table", "parameters, compute (GMACs) and time per image, ~1 min")
rows = []
for name in MODEL_ORDER:
    m = build_model(name, pretrained=False).to(DEVICE).eval()
    n_par = sum(p.numel() for p in m.parameters())
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    with torch.no_grad():
        for _ in range(3): m(x)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(10): m(x)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
    gflops = np.nan
    try:
        from torch.utils.flop_counter import FlopCounterMode
        with FlopCounterMode(display=False) as fc, torch.no_grad():
            m(x)
        gflops = fc.get_total_flops() / 2e9          # multiply-adds -> GMACs
    except Exception:
        pass
    rows.append({"Model": name, "Params (M)": round(n_par / 1e6, 3), "GMACs": round(gflops, 2),
                 "ms/image (bs=1)": round((time.perf_counter() - t0) / 10 * 1000, 1)})
    del m
df_complexity = pd.DataFrame(rows)
print(df_complexity.to_string(index=False))
df_complexity.to_csv(OUT_DIR / "tables" / "table_complexity.csv", index=False)
if DEVICE.type == "cuda": torch.cuda.empty_cache()

In [ ]:
# ─── 2.4 Datasets, training transforms, and a fast evaluation cache ───
def train_transform(mean, std):
    return T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE), interpolation=T.InterpolationMode.BICUBIC),
        T.Grayscale(num_output_channels=3),
        T.RandomHorizontalFlip(0.5),
        T.RandomRotation(15),
        T.RandomAffine(0, translate=(0.05, 0.05), scale=(0.9, 1.1), shear=5),
        T.ColorJitter(brightness=0.2, contrast=0.2),
        T.RandomAutocontrast(p=0.3),
        T.GaussianBlur(3, sigma=(0.1, 1.0)),
        T.ToTensor(), T.Normalize(mean, std)])


_U8_CACHE = {}
def load_u8(path):
    """Grayscale, bicubic 224x224, uint8 - identical to Resize -> Grayscale -> ToTensor (x255)."""
    a = _U8_CACHE.get(path)
    if a is None:
        with Image.open(path) as im:
            a = np.asarray(im.convert("L").resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC), np.uint8)
        _U8_CACHE[path] = a
    return a

def load_mask(path):
    return np.asarray(Image.open(path).convert("L").resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)) > 127

class ImageList(Dataset):
    """Images are read from disk ONCE (224x224 uint8, ~50 KB each) and kept in RAM.
    Same pixels as before (grayscale -> bicubic 224), but no disk reading every epoch."""
    def __init__(self, items, tf):
        self.items, self.tf = list(items), tf
        self.arrs = [load_u8(p) for p, _ in self.items]
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        return self.tf(Image.fromarray(self.arrs[i], mode="L")), int(self.items[i][1])

from torchvision.transforms import v2 as T2
from torchvision import tv_tensors

class ImageMaskList(Dataset):
    """Same augmentation as train_transform, but flips / rotation / affine are applied to the
    image AND its tumour mask together; brightness / contrast / blur only to the image."""
    def __init__(self, items, mean, std):
        self.items = list(items)
        self.arrs = [load_u8(p) for p, _ in self.items]
        self.masks = [MASKS.get(p) for p, _ in self.items]
        self.geo = T2.Compose([T2.RandomHorizontalFlip(0.5), T2.RandomRotation(15),
                               T2.RandomAffine(0, translate=(0.05, 0.05), scale=(0.9, 1.1), shear=5)])
        self.photo = T2.Compose([T2.ColorJitter(brightness=0.2, contrast=0.2), T2.RandomAutocontrast(p=0.3),
                                 T2.GaussianBlur(3, sigma=(0.1, 1.0))])
        self.mean = torch.tensor(mean, dtype=torch.float32).view(3, 1, 1)
        self.std = torch.tensor(std, dtype=torch.float32).view(3, 1, 1)
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        img = tv_tensors.Image(torch.from_numpy(self.arrs[i])[None])
        has = self.masks[i] is not None
        mk = tv_tensors.Mask(torch.from_numpy(self.masks[i].astype(np.uint8))[None] if has
                             else torch.zeros(1, IMG_SIZE, IMG_SIZE, dtype=torch.uint8))
        img, mk = self.geo(img, mk)
        img = self.photo(img)
        x = (img.float().div(255).repeat(3, 1, 1) - self.mean) / self.std
        return x, mk.float(), torch.tensor(float(has)), int(self.items[i][1])

class GuidedSet(Dataset):
    """Run-3 pipeline: returns uint8 (5, H, W) = [image, tumour hidden (inpaint), random spot hidden,
    tumour mask, tumour region (mask + margin)], has_mask, label. All augmentation happens on the GPU,
    identically for the 5 channels (so the hidden versions differ from the image ONLY in the hidden region)."""
    def __init__(self, items, strip=False):
        self.items, self.strip = list(items), strip
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        p, y = self.items[i]
        img = load_u8(p); h = HIDE.get(p)
        if self.strip:                      # CNX-Strip: the model only ever sees skull-stripped images
            img = skull_strip(img); h = None
        if h is not None:
            st = np.stack([img, h["inpaint"], h["random"], MASKS[p].astype(np.uint8) * 255, h["region"].astype(np.uint8) * 255])
            has = 1.0
        else:
            z = np.zeros_like(img); st = np.stack([img, img, img, z, z]); has = 0.0
        return torch.from_numpy(st), torch.tensor(has), int(y)

def gpu_augment(b):
    """b: uint8 (B,5,H,W) on GPU -> images (B,3,H,W) in [0,1] [orig, hidden, random-hidden], masks (B,2,H,W).
    Same augmentation family as train_transform: flip, rotation 15, translate 5%, scale 0.9-1.1, shear 5,
    brightness/contrast 0.2, autocontrast p=0.3, Gaussian blur sigma 0.1-1.0."""
    x = b.float() / 255; B = x.shape[0]; dev = x.device
    U = lambda lo, hi: torch.empty(B, device=dev).uniform_(lo, hi)
    ang, sh = torch.deg2rad(U(-15, 15)), torch.deg2rad(U(-5, 5))
    sc, tx, ty = U(0.9, 1.1), U(-0.1, 0.1), U(-0.1, 0.1)
    flip = torch.where(torch.rand(B, device=dev) < 0.5, -1.0, 1.0)
    cos, sin = torch.cos(ang), torch.sin(ang)
    theta = torch.zeros(B, 2, 3, device=dev)
    theta[:, 0, 0] = cos / sc * flip; theta[:, 0, 1] = (-sin + torch.tan(sh) * cos) / sc
    theta[:, 1, 0] = sin / sc * flip; theta[:, 1, 1] = (cos + torch.tan(sh) * sin) / sc
    theta[:, 0, 2], theta[:, 1, 2] = tx, ty
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    imgs = F.grid_sample(x[:, :3], grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    masks = (F.grid_sample(x[:, 3:], grid, mode="nearest", padding_mode="zeros", align_corners=False) > 0.5).float()
    bri, con = U(0.8, 1.2).view(B, 1, 1, 1), U(0.8, 1.2).view(B, 1, 1, 1)
    imgs = imgs * bri
    mu = imgs[:, :1].mean((2, 3), keepdim=True)
    imgs = ((imgs - mu) * con + mu).clamp(0, 1)
    ac = (torch.rand(B, device=dev) < 0.3).view(B, 1, 1, 1)
    lo = imgs[:, :1].amin((2, 3), keepdim=True); hi = imgs[:, :1].amax((2, 3), keepdim=True)
    imgs = torch.where(ac, ((imgs - lo) / (hi - lo + 1e-6)).clamp(0, 1), imgs)
    sig = U(0.1, 1.0)
    k1 = torch.exp(-(torch.tensor([-1.0, 0.0, 1.0], device=dev)[None] ** 2) / (2 * sig[:, None] ** 2))
    k1 = k1 / k1.sum(1, keepdim=True); k2 = (k1[:, :, None] * k1[:, None, :])          # (B,3,3)
    k2 = k2.repeat_interleave(3, 0)[:, None]                                            # (B*3,1,3,3)
    imgs = F.conv2d(F.pad(imgs.reshape(1, B * 3, *imgs.shape[-2:]), (1, 1, 1, 1), mode="reflect"),
                    k2, groups=B * 3).reshape(B, 3, *imgs.shape[-2:])
    return imgs, masks

def norm3(x1, mean, std):
    """(B,1,H,W) in [0,1] -> normalised 3-channel input."""
    m = torch.tensor([float(v) for v in mean], device=x1.device).view(1, 3, 1, 1)
    s = torch.tensor([float(v) for v in std], device=x1.device).view(1, 3, 1, 1)
    return (x1.repeat(1, 3, 1, 1) - m) / s

# ─── ROI remedy: crop the region of the (predicted or expert) tumour ───
def box_from_mask(mask, min_pixels=ROI_MIN_PIXELS):
    """Largest connected region of a binary mask -> (x, y, w, h, component) or None (= no tumour found)."""
    m = np.asarray(mask, np.uint8)
    if m.sum() < min_pixels:
        return None
    n, lab, st, _ = cv2.connectedComponentsWithStats(m)
    k = 1 + int(np.argmax(st[1:, cv2.CC_STAT_AREA]))
    if st[k, cv2.CC_STAT_AREA] < min_pixels:
        return None
    x, y, w, h = (int(v) for v in st[k, :4])
    return x, y, w, h, lab == k

def crop_roi(img, box, mode, margin=ROI_MARGIN, shift=(0.0, 0.0)):
    """Square crop around the box (+margin on each side), resized to 224. mode 'mask': outside the region = 0."""
    x, y, w, h, comp = box
    if mode == "mask":
        keep = cv2.dilate(comp.astype(np.uint8), np.ones((7, 7), np.uint8)).astype(bool)
        img = np.where(keep, img, 0).astype(np.uint8)
    s = max(max(w, h) * (1 + 2 * margin), 32)
    cx, cy = x + w / 2 + shift[0] * s, y + h / 2 + shift[1] * s
    im = Image.fromarray(img).crop((int(cx - s / 2), int(cy - s / 2), int(cx + s / 2), int(cy + s / 2)))
    return np.asarray(im.resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC), np.uint8)

class RoiSet(Dataset):
    """Training crops from the EXPERT mask with random margin / shift (the model must cope with an imperfect
    box at test time, where the box comes from the predicted tumour map). margin = the model's box margin
    (training draws it from 0.5x-1.75x around that value, as in Run 4: 0.2 -> 0.1-0.35).
    loc=True (ROI-Loc): also returns the 6 location numbers (+ small noise, as predicted maps are imperfect)."""
    def __init__(self, items, tf, mode, margin=ROI_MARGIN, loc=False, transplant=False):
        self.items, self.tf, self.mode, self.margin, self.loc = list(items), tf, mode, margin, loc
        self.arrs = [load_u8(p) for p, _ in self.items]
        self.boxes = [box_from_mask(MASKS[p]) if p in MASKS else None for p, _ in self.items]
        self.locs = [loc_feats(a, MASKS[p]) if (loc and p in MASKS) else np.zeros(N_LOC, np.float32)
                     for a, (p, _) in zip(self.arrs, self.items)]
        self.transplant = transplant
        if transplant:     # Run 6: the context around the crop comes from a patient of ANOTHER class
            self.ok = [i for i, (p, _) in enumerate(self.items) if p in HIDE]
            self.by_cls = {k: [i for i in self.ok if self.items[i][1] == k] for k in range(N_CLASSES)}
            self.bm = {self.items[i][0]: brain_bm(self.arrs[i], self.items[i][0]) for i in self.ok}
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        img, b = self.arrs[i], self.boxes[i]
        r = np.random.default_rng()
        p, y = self.items[i]
        if self.transplant and p in HIDE and r.random() < TRANSPLANT_PROB:
            others = [k for k in range(N_CLASSES) if k != y and self.by_cls[k]]
            k = int(r.choice(others)); j = int(r.choice(self.by_cls[k])); q = self.items[j][0]
            res = transplant(img, HIDE[p]["inpaint"], self.bm[p], self.arrs[j], MASKS[q], self.bm[q], r)
            bt = box_from_mask(res[1]) if res is not None else None
            if bt is not None:
                crop = crop_roi(res[0], bt, self.mode, margin=r.uniform(0.5 * self.margin, 1.75 * self.margin),
                                shift=tuple(r.uniform(-0.1, 0.1, 2)))
                return self.tf(Image.fromarray(crop, mode="L")), k
        if b is not None:
            img = crop_roi(img, b, self.mode, margin=r.uniform(0.5 * self.margin, 1.75 * self.margin),
                           shift=tuple(r.uniform(-0.1, 0.1, 2)))
        x = self.tf(Image.fromarray(img, mode="L"))
        if self.loc:
            l = self.locs[i] + r.normal(0, 0.03, N_LOC).astype(np.float32)
            return x, torch.from_numpy(l.astype(np.float32)), int(self.items[i][1])
        return x, int(self.items[i][1])

# ─── Run 5 helpers: location numbers, skull stripping, tumour transplant ───
_BM_CACHE = {}
def brain_bm(img, key=None):
    """Head/brain mask (Otsu, cell 1.8), cached by path when a key is given."""
    if key is not None and key in _BM_CACHE:
        return _BM_CACHE[key]
    bm = brain_mask(img)
    if key is not None:
        _BM_CACHE[key] = bm
    return bm

def loc_feats(img, mask, bm=None):
    """6 numbers describing WHERE the tumour is relative to the brain outline (no image content):
    vertical position (-1 top .. +1 bottom), distance from the left-right midline (0..1), distance from the brain
    centre, log relative area, relative width, relative height. Left/right is deliberately NOT used."""
    mk = np.asarray(mask, bool)
    bm = brain_mask(img) if bm is None else bm
    if not mk.any() or not bm.any():
        return np.zeros(N_LOC, np.float32)
    ys, xs = np.where(mk); bys, bxs = np.where(bm)
    by0, bx0 = bys.min(), bxs.min(); bh = max(bys.max() - by0, 1); bw = max(bxs.max() - bx0, 1)
    vy = 2 * (ys.mean() - by0) / bh - 1; vx = 2 * (xs.mean() - bx0) / bw - 1
    return np.array([vy, abs(vx), math.sqrt(vy ** 2 + vx ** 2),
                     math.log(mk.sum() / max(bm.sum(), 1) + 1e-4) / 5 + 1,
                     (xs.max() - xs.min() + 1) / bw, (ys.max() - ys.min() + 1) / bh], np.float32)

def skull_strip(img):
    """Approximate skull stripping for 2D slices: keep only the head mask eroded by STRIP_ERODE pixels
    (removes scalp + skull ring). Applied to every input of CNX-Strip EXCEPT BraTS (already skull-stripped)."""
    bm = brain_mask(img)
    keep = cv2.erode(bm.astype(np.uint8), np.ones((STRIP_ERODE, STRIP_ERODE), np.uint8)).astype(bool)
    return np.where(keep, img, 0).astype(np.uint8)

def transplant(rec_img, rec_hidden, rec_bm, don_img, don_mask, don_bm, rng):
    """Paste the DONOR's tumour into the RECIPIENT slice whose own tumour was removed (inpainted).
    The tumour keeps its own relative position in the brain (location is a legitimate cue), but everything
    around it - skull, slice level, plane - now comes from a patient of ANOTHER class.
    Returns (image, mask) or None if the tumour does not fit."""
    ys, xs = np.where(don_mask); bys, bxs = np.where(don_bm); rys, rxs = np.where(rec_bm)
    if len(ys) == 0 or len(rys) == 0:
        return None
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    pad = 4; y0, x0 = max(y0 - pad, 0), max(x0 - pad, 0); y1, x1 = min(y1 + pad, IMG_SIZE), min(x1 + pad, IMG_SIZE)
    patch = don_img[y0:y1, x0:x1].astype(np.float32)
    alpha = cv2.GaussianBlur(cv2.dilate(don_mask.astype(np.uint8), np.ones((5, 5), np.uint8)).astype(np.float32),
                             (7, 7), 2)[y0:y1, x0:x1]
    scale = float(np.clip((np.ptp(rys) + 1) / (np.ptp(bys) + 1), 0.7, 1.4))
    h, w = max(int(round((y1 - y0) * scale)), 3), max(int(round((x1 - x0) * scale)), 3)
    patch = cv2.resize(patch, (w, h), interpolation=cv2.INTER_LINEAR)
    alpha = cv2.resize(alpha, (w, h), interpolation=cv2.INTER_LINEAR)
    med_d = np.median(don_img[don_bm & ~don_mask]) + 1e-3; med_r = np.median(rec_img[rec_bm]) + 1e-3
    patch = patch * float(np.clip(med_r / med_d, 0.6, 1.6))              # brightness matched to the recipient
    ry = (ys.mean() - bys.min()) / max(np.ptp(bys), 1); rx = (xs.mean() - bxs.min()) / max(np.ptp(bxs), 1)
    if rng.random() < 0.5:
        rx = 1 - rx                                                         # either hemisphere
    cy = rys.min() + ry * np.ptp(rys) + rng.normal(0, 3); cx = rxs.min() + rx * np.ptp(rxs) + rng.normal(0, 3)
    ty, tx = int(round(cy - h / 2)), int(round(cx - w / 2))
    if ty < 0 or tx < 0 or ty + h > IMG_SIZE or tx + w > IMG_SIZE:
        return None
    if rec_bm[ty:ty + h, tx:tx + w][alpha > 0.5].mean() < 0.9:              # tumour must land inside the head
        return None
    out = rec_hidden.astype(np.float32).copy()
    region = out[ty:ty + h, tx:tx + w]
    out[ty:ty + h, tx:tx + w] = alpha * patch + (1 - alpha) * region
    mk = np.zeros((IMG_SIZE, IMG_SIZE), bool); mk[ty:ty + h, tx:tx + w] = alpha > 0.5
    return np.clip(out, 0, 255).astype(np.uint8), mk

class TransplantSet(GuidedSet):
    """SOLUTION 2. With probability TRANSPLANT_PROB the image becomes: recipient slice (its own tumour removed
    by inpainting) + a tumour from a training patient of ANOTHER class; the label is the donor's class.
    The rest of the slice therefore carries no information about the label. NOTE: inpainting is used here only to
    remove the recipient's tumour; the reliance TEST uses 'swap' and 'tissue from another patient' instead, so the
    model cannot pass the test by recognising the training operation (the CTR-Net mistake of Run 3)."""
    def __init__(self, items):
        super().__init__(items)
        self.ok = [i for i, (p, _) in enumerate(self.items) if p in HIDE]
        self.by_cls = {k: [i for i in self.ok if self.items[i][1] == k] for k in range(N_CLASSES)}
        self.bm = {self.items[i][0]: brain_bm(load_u8(self.items[i][0]), self.items[i][0]) for i in self.ok}
        self.n_ok = self.n_try = 0
    def __getitem__(self, i):
        p, y = self.items[i]
        rng = np.random.default_rng()
        if p in HIDE and rng.random() < TRANSPLANT_PROB:
            others = [k for k in range(N_CLASSES) if k != y and self.by_cls[k]]
            k = int(rng.choice(others)); j = int(rng.choice(self.by_cls[k])); q = self.items[j][0]
            res = transplant(load_u8(p), HIDE[p]["inpaint"], self.bm[p], load_u8(q), MASKS[q], self.bm[q], rng)
            if res is not None:
                img, mk = res
                z = np.zeros_like(img)
                st = np.stack([img, img, img, mk.astype(np.uint8) * 255, z])
                return torch.from_numpy(st), torch.tensor(0.0), k
        return super().__getitem__(i)

def to_batch(arrs, mean, std, device=DEVICE):
    x = torch.from_numpy(np.stack(arrs)).to(device).float().div_(255).unsqueeze(1).repeat(1, 3, 1, 1)
    m = torch.tensor([float(v) for v in mean], dtype=torch.float32, device=device).view(1, 3, 1, 1)
    s = torch.tensor([float(v) for v in std], dtype=torch.float32, device=device).view(1, 3, 1, 1)
    return (x - m) / s

def class_weights(items):
    c = np.bincount([y for _, y in items], minlength=N_CLASSES).astype(float)
    w = 1.0 / np.maximum(c, 1); w = w / w.sum() * N_CLASSES
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)

# ─── Fast metrics (used everywhere) ───
def hard(P):
    """Predicted class per row; -1 = ABSTAINED (an ROI model that found no tumour outputs exactly equal
    probabilities - this must not be counted as 'glioma', which np.argmax would do)."""
    P = np.asarray(P); pred = P.argmax(1)
    pred[(P.max(1) - P.min(1)) < 1e-6] = -1
    return pred

def macro_f1(y, p, n=N_CLASSES, labels=None):
    """Macro-F1; abstained predictions (-1) count as a miss for their true class (no false positive elsewhere)."""
    labels = range(n) if labels is None else labels
    y, p = np.asarray(y), np.asarray(p); ok = p >= 0
    cm = np.bincount(y[ok] * n + p[ok], minlength=n * n).reshape(n, n)
    f = []
    for k in labels:
        tp = cm[k, k]; fp = cm[:, k].sum() - tp; fn = cm[k, :].sum() - tp + int(((y == k) & ~ok).sum())
        f.append(0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn))
    return float(np.mean(f))

def macro_auc(y, probs):
    y = np.asarray(y); present = np.unique(y)
    if len(present) == 2:                     # e.g. deduplicated BRISC has no glioma: AUC of the 2 present classes
        a, b = present
        try:
            return float(roc_auc_score(y == b, probs[:, b] / (probs[:, a] + probs[:, b] + 1e-12)))
        except ValueError:
            return float("nan")
    try:
        return float(roc_auc_score(np.asarray(y), probs, multi_class="ovr", labels=list(range(N_CLASSES))))
    except ValueError:
        return float("nan")

def per_class_recall(y, p):
    y, p = np.asarray(y), np.asarray(p)
    return {IDX2CLASS[k]: (float((p[y == k] == k).mean()) if (y == k).any() else float("nan")) for k in range(N_CLASSES)}

def softmax_np(z):
    z = z - z.max(1, keepdims=True); e = np.exp(z)
    return e / e.sum(1, keepdims=True)

### 2.5 Run 5 inputs at a glance (check that the new inputs look sensible before trusting any result)

In [ ]:
# ─── 2.5 What the new solutions actually see ───
if MASKS:
    rng_v = np.random.default_rng(7)
    tr_items = FOLDS[1]["train"]
    by_c = {k: [p for p, y in tr_items if y == k and p in HIDE] for k in range(N_CLASSES)}
    # (a) tumour transplant examples
    rows_ = []
    for rec_c, don_c in [(2, 0), (0, 1), (1, 2), (2, 1)]:
        for _ in range(50):
            p, q = rng_v.choice(by_c[rec_c]), rng_v.choice(by_c[don_c])
            res = transplant(load_u8(p), HIDE[p]["inpaint"], brain_mask(load_u8(p)), load_u8(q), MASKS[q], brain_mask(load_u8(q)), rng_v)
            if res is not None:
                rows_.append((p, q, rec_c, don_c, res)); break
    if rows_:
        fig, ax = plt.subplots(len(rows_), 3, figsize=(7.2, 2.45 * len(rows_)), squeeze=False)
        for r, (p, q, rc, dc, (img, mk)) in enumerate(rows_):
            ax[r, 0].imshow(load_u8(p), cmap="gray"); ax[r, 0].contour(MASKS[p], levels=[0.5], colors="lime", linewidths=0.7)
            ax[r, 1].imshow(load_u8(q), cmap="gray"); ax[r, 1].contour(MASKS[q], levels=[0.5], colors="orange", linewidths=0.7)
            ax[r, 2].imshow(img, cmap="gray"); ax[r, 2].contour(mk, levels=[0.5], colors="orange", linewidths=0.7)
            ax[r, 0].set_title(f"recipient: {CLASSES[rc]} patient", fontsize=7)
            ax[r, 1].set_title(f"donor tumour: {CLASSES[dc]}", fontsize=7)
            ax[r, 2].set_title(f"training image -> label '{CLASSES[dc]}'", fontsize=7, color=CLASS_COLORS[CLASSES[dc]])
            for a in ax[r]: a.axis("off")
        fig.suptitle("Solution 2 - tumour transplant: the surroundings come from a patient of ANOTHER class,\n"
                     "so only the tumour itself tells the label (green = own tumour, removed; orange = donor tumour)", fontsize=8)
        fig.tight_layout(); fig.savefig(FIG_DIR / "fig_transplant_examples.png", bbox_inches="tight"); plt.show()
    # (b) skull stripping (control experiment)
    ex = [rng_v.choice(by_c[k]) for k in range(N_CLASSES)] + ([BRATS_ITEMS[0][0]] if BRATS_ITEMS else [])
    fig, ax = plt.subplots(2, len(ex), figsize=(2.1 * len(ex), 4.4), squeeze=False)
    for c, p in enumerate(ex):
        a = load_u8(p); is_b = c == len(ex) - 1 and BRATS_ITEMS
        ax[0, c].imshow(a, cmap="gray"); ax[1, c].imshow(a if is_b else skull_strip(a), cmap="gray")
        ax[0, c].set_title("BraTS (already stripped)" if is_b else f"Figshare {CLASSES[c]}", fontsize=7)
        for a_ in ax[:, c]: a_.axis("off")
    ax[1, 0].text(-12, IMG_SIZE / 2, "what CNX-Strip sees", rotation=90, va="center", ha="right", fontsize=7)
    fig.suptitle("Control - skull stripping: does ROI win on BraTS only because BraTS has no skull?", fontsize=8)
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig_strip_examples.png", bbox_inches="tight"); plt.show()
    # (c) ROI-Loc input: crop + 6 location numbers
    names6 = ["vertical pos.", "dist. midline", "dist. centre", "rel. area", "rel. width", "rel. height"]
    fig, ax = plt.subplots(2, 3, figsize=(7.5, 5.2))
    for c in range(N_CLASSES):
        p = rng_v.choice(by_c[c]); a = load_u8(p); bm = brain_mask(a); b = box_from_mask(MASKS[p]); lf = loc_feats(a, MASKS[p], bm)
        ax[0, c].imshow(a, cmap="gray"); ax[0, c].contour(bm, levels=[0.5], colors="cyan", linewidths=0.6)
        ax[0, c].contour(MASKS[p], levels=[0.5], colors="lime", linewidths=0.8)
        ax[0, c].set_title(f"{CLASSES[c]}: brain outline (cyan) + tumour", fontsize=7)
        ax[1, c].imshow(crop_roi(a, b, "box"), cmap="gray")
        ax[1, c].set_title("crop + " + ", ".join(f"{v:+.2f}" for v in lf[:3]) + "\n" + ", ".join(f"{v:.2f}" for v in lf[3:]), fontsize=6)
        for a_ in ax[:, c]: a_.axis("off")
    fig.suptitle("Solution 1 - ROI-Loc sees ONLY the tumour crop + 6 location numbers\n(" + ", ".join(names6) + ")", fontsize=8)
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig_roiloc_inputs.png", bbox_inches="tight"); plt.show()

## 3. Training (identical protocol for every model)
AdamW, 3-epoch warm-up + cosine decay, max 80 epochs, early stopping on validation macro-F1 (patience 15),
class-weighted cross-entropy with label smoothing 0.1, MixUp/CutMix (p = 0.5), AMP, gradient clipping.
No SWA and no test-time augmentation anywhere. Each (model, seed, fold) is saved and skipped if it already exists.

In [ ]:
# ─── 3.1 Training engine ───
def run_paths(name, seed, fold):
    tag = f"{name}_s{seed}_f{fold}"
    return MODEL_DIR / f"{tag}.pt", OUT_DIR / "runs" / f"{tag}.json", OUT_DIR / "oof" / f"{tag}_logits.npy"

def run_done(name, seed, fold):
    ckpt, rec, oof = run_paths(name, seed, fold)
    return ckpt.exists() and rec.exists() and oof.exists()

def get_lr(name):
    choice = load_json(OUT_DIR / "lr_choice.json", {})
    name = LR_INHERIT.get(name, name)
    return choice.get(name, DEFAULT_LR["pretrained" if is_pretrained(name) else "scratch"])

def lr_factor(epoch, total, warmup=WARMUP_EPOCHS, floor=0.01):
    if epoch < warmup:
        return (epoch + 1) / warmup
    t = (epoch - warmup) / max(1, total - warmup)
    return floor + (1 - floor) * 0.5 * (1 + math.cos(math.pi * t))

def mix_batch(x, y, m=None):
    """MixUp / CutMix. If a tumour mask m is given, it is mixed exactly like the image.
    Returns x, y_a, y_b, lam, m, idx (idx = None when no mixing was applied)."""
    if np.random.rand() >= MIX_PROB:
        return x, y, y, 1.0, m, None
    idx = torch.randperm(x.size(0), device=x.device)
    if np.random.rand() < 0.5:                                    # MixUp
        lam = float(np.random.beta(0.4, 0.4))
        m = None if m is None else lam * m + (1 - lam) * m[idx]
        return lam * x + (1 - lam) * x[idx], y, y[idx], lam, m, idx
    lam = float(np.random.beta(1.0, 1.0))                         # CutMix
    H, W = x.shape[2:]
    ch, cw = int(H * math.sqrt(1 - lam)), int(W * math.sqrt(1 - lam))
    cy, cx = np.random.randint(H), np.random.randint(W)
    y1, y2, x1, x2 = max(cy - ch // 2, 0), min(cy + ch // 2, H), max(cx - cw // 2, 0), min(cx + cw // 2, W)
    x = x.clone(); x[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    if m is not None:
        m = m.clone(); m[:, :, y1:y2, x1:x2] = m[idx, :, y1:y2, x1:x2]
    return x, y, y[idx], 1 - (y2 - y1) * (x2 - x1) / (H * W), m, idx

def seg_loss_fn(seg_logit, mask, has):
    """BCE + soft Dice between the 28x28 tumour logits and the (area-downsampled) expert mask.
    Slices without a mask (has = 0) do not contribute."""
    tgt = F.adaptive_avg_pool2d(mask, seg_logit.shape[-2:])
    z = seg_logit.float()
    bce = F.binary_cross_entropy_with_logits(z, tgt, reduction="none").mean((1, 2, 3))
    pr = torch.sigmoid(z)
    dice = 1 - (2 * (pr * tgt).sum((1, 2, 3)) + 1) / (pr.sum((1, 2, 3)) + tgt.sum((1, 2, 3)) + 1)
    return ((bce + dice) * has).sum() / has.sum().clamp(min=1)

@torch.no_grad()
def predict_logits(model, arrs, mean, std, bs=128):
    model.eval(); out = []
    for i in range(0, len(arrs), bs):
        xb = to_batch(arrs[i:i + bs], mean, std)
        with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
            out.append(model(xb).float().cpu().numpy())
    return np.concatenate(out)

_SEG = {"key": None, "model": None}

def roi_seg_maps(arrs, fold, seed=0):
    """Tumour-probability maps (224x224) from ROI_SEG_SOURCE's model of the SAME fold (never saw this fold's
    validation patients). Separate model cache, so it does not disturb load_checkpoint's cache."""
    ckpt = run_paths(ROI_SEG_SOURCE, seed, fold)[0]
    if _SEG["key"] != ckpt:
        if _SEG["model"] is None:
            _SEG["model"] = build_model(ROI_SEG_SOURCE, pretrained=False).to(DEVICE)
        _SEG["model"].load_state_dict(torch.load(ckpt, map_location=DEVICE), strict=True); _SEG["model"].eval()
        _SEG["key"] = ckpt
    mean, std = norm_for(ROI_SEG_SOURCE, fold); out = []
    with torch.no_grad():
        for i in range(0, len(arrs), 64):
            with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
                _, sl = _SEG["model"](to_batch(arrs[i:i + 64], mean, std), return_seg=True)
            out.append(torch.sigmoid(F.interpolate(sl.float(), (IMG_SIZE, IMG_SIZE), mode="bilinear",
                                                   align_corners=False))[:, 0].cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, IMG_SIZE, IMG_SIZE), np.float32)

def roi_inputs(name, fold, arrs, maps=None):
    """(crops, found, locs): crop around the predicted tumour; found = False -> nothing to classify.
    locs = the 6 location numbers of the found tumour (ROI-Loc only, otherwise zeros)."""
    spec = MODEL_SPECS[name]
    maps = roi_seg_maps(arrs, fold) if maps is None else maps
    crops, found, locs = [], [], []
    for a, mp in zip(arrs, maps):
        b = box_from_mask(mp >= 0.5)
        found.append(b is not None)
        crops.append(crop_roi(a, b, spec["crop"], margin=spec.get("margin", ROI_MARGIN)) if b is not None else a)
        locs.append(loc_feats(a, b[4]) if (b is not None and spec.get("loc")) else np.zeros(N_LOC, np.float32))
    return crops, np.array(found, bool), np.stack(locs) if locs else np.zeros((0, N_LOC), np.float32)

@torch.no_grad()
def predict_logits_loc(model, arrs, locs, mean, std, bs=128):
    model.eval(); out = []
    for i in range(0, len(arrs), bs):
        xb = to_batch(arrs[i:i + bs], mean, std); lb = torch.from_numpy(np.asarray(locs[i:i + bs], np.float32)).to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
            out.append(model(xb, loc=lb).float().cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, N_CLASSES), np.float32)

STRIP_SKIP_SETS = {"brats", "bfree"}      # already skull-stripped -> CNX-Strip uses them as they are
_CUR_SET = {"name": None}                 # set by the evaluation loops (5.1, 8.5) to the current dataset

def predict_any(name, fold, model, arrs, mean, std, maps=None):
    """Logits for any model. ROI models: when no tumour is found the logits are 0 (= uniform = 'unsure').
    CNX-Strip: inputs are skull-stripped first (except BraTS, which already is)."""
    spec = MODEL_SPECS[name]
    if spec.get("strip"):
        if _CUR_SET["name"] not in STRIP_SKIP_SETS:
            arrs = [skull_strip(a) for a in arrs]
        return predict_logits(model, arrs, mean, std)
    if not is_roi(name):
        return predict_logits(model, arrs, mean, std)
    crops, found, locs = roi_inputs(name, fold, arrs, maps)
    z = np.zeros((len(arrs), N_CLASSES), np.float32)
    if found.any():
        sel = [c for c, f in zip(crops, found) if f]
        z[found] = (predict_logits_loc(model, sel, locs[found], mean, std) if spec.get("loc")
                    else predict_logits(model, sel, mean, std))
    return z

def guide_penalty(model, x, region, y, kind):
    """Share of the model's evidence that lies OUTSIDE the tumour region (0 = all evidence inside).
    rrr: squared input gradients of the log-probabilities (Ross et al. 2017, normalised);
    cam: class-activation map of the true class on the last 7x7 features (CAM / attention guidance)."""
    with torch.autocast(device_type=DEVICE.type, enabled=False):
        if kind == "rrr":
            x = x.float().detach().requires_grad_(True)
            lp = F.log_softmax(model(x).float(), 1).sum()
            g = torch.autograd.grad(lp, x, create_graph=True)[0].pow(2).sum(1, keepdim=True)
            return ((g * (1 - region)).sum((1, 2, 3)) / (g.sum((1, 2, 3)) + 1e-12)).mean()
        logits, f3 = model(x.float(), return_feat=True)
        g = torch.autograd.grad(logits.float().gather(1, y[:, None]).sum(), f3, create_graph=True)[0]
        cam = F.relu((g.mean((2, 3), keepdim=True) * f3).sum(1, keepdim=True))
        r = F.adaptive_avg_pool2d(region, cam.shape[-2:])
        return ((cam * (1 - r)).sum((1, 2, 3)) / (cam.sum((1, 2, 3)) + 1e-6)).mean()

def close_loader(loader):
    """Stop the DataLoader worker processes. Without this, every finished model left 4 workers
    alive in RAM, which crashed the kernel ("allocate more memory than available")."""
    it = getattr(loader, "_iterator", None)
    if it is not None and hasattr(it, "_shutdown_workers"):
        it._shutdown_workers()
    loader._iterator = None

def train_one(name, seed, fold, lr=None, max_epochs=NUM_EPOCHS, patience=PATIENCE, save=True, show_every=10):
    lr = lr or get_lr(name)
    set_seed(seed * 100 + fold)
    mean, std = norm_for(name, fold)
    train_items, val_items = FOLDS[fold]["train"], FOLDS[fold]["val"]
    newp = new_pipeline(name) and bool(MASKS)
    spec = MODEL_SPECS[name]
    use_loc = bool(spec.get("loc"))
    if newp:                                      # Run-3 models: GPU augmentation + masks + hidden versions
        seg, cf, guide = bool(spec.get("seg")), bool(spec.get("cf")), spec.get("guide")
        ds = (TransplantSet(train_items) if spec.get("transplant")                       # Run 5, solution 2
              else GuidedSet(train_items, strip=bool(spec.get("strip"))))               # Run 5, skull control
    elif is_roi(name):                            # Run-4 remedy: train on expert-mask crops
        seg, cf, guide = False, False, None
        ds = RoiSet(train_items, train_transform(mean, std), spec["crop"],
                    margin=spec.get("margin", ROI_MARGIN), loc=use_loc, transplant=bool(spec.get("transplant")))
    else:
        seg, cf, guide = uses_masks(name) and bool(MASKS) and spec.get("use_seg", False), False, None
        ds = ImageMaskList(train_items, mean, std) if seg else ImageList(train_items, train_transform(mean, std))
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=NUM_WORKERS, pin_memory=True, drop_last=True, persistent_workers=True)
    val_raw = [load_u8(p) for p, _ in val_items]
    val_locs = None
    if is_roi(name):                              # early stopping on expert-mask crops (no test information)
        mg = spec.get("margin", ROI_MARGIN)
        val_arrs = [crop_roi(a, box_from_mask(MASKS[p]), spec["crop"], margin=mg) if p in MASKS and box_from_mask(MASKS[p]) else a
                    for a, (p, _) in zip(val_raw, val_items)]
        if use_loc:
            val_locs = np.stack([loc_feats(a, MASKS[p]) if p in MASKS else np.zeros(N_LOC, np.float32)
                                 for a, (p, _) in zip(val_raw, val_items)])
    elif spec.get("strip"):
        val_arrs = [skull_strip(a) for a in val_raw]
    else:
        val_arrs = val_raw
    val_y = np.array([y for _, y in val_items])
    model = build_model(name, pretrained=True).to(DEVICE)
    if is_pretrained(name) and USE_DP_FOR_PRETRAINED:
        model = nn.DataParallel(model)
    groups = model.param_groups(lr) if hasattr(model, "param_groups") else [{"params": list(model.parameters()), "base_lr": lr}]
    opt = torch.optim.AdamW(groups, lr=lr, weight_decay=WEIGHT_DECAY)
    scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")
    w = class_weights(train_items)
    ce = lambda out, t: F.cross_entropy(out, t, weight=w, label_smoothing=LABEL_SMOOTHING)
    best = {"f1": -1, "epoch": -1, "state": None}
    hist = {"train_loss": [], "val_f1": [], "val_acc": [], "lr": []}
    wait, t_start = 0, time.time()
    try:
      for epoch in range(max_epochs):
          for g in opt.param_groups:
              g["lr"] = g["base_lr"] * lr_factor(epoch, max_epochs)
          model.train(); tot = 0.0; t_ep = time.time()
          for batch in loader:
              if newp:
                  st, has, yb = (t.to(DEVICE, non_blocking=True) for t in batch)
                  imgs, masks = gpu_augment(st)
                  x0, xh, xr = (norm3(imgs[:, k:k + 1], mean, std) for k in range(3))
                  mb, reg = (masks[:, :1] if seg else None), masks[:, 1:2]
                  hb = has
                  xb = x0
              elif seg:
                  xb, mb, hb, yb = (t.to(DEVICE, non_blocking=True) for t in batch)
              elif use_loc:
                  xb, lb, yb = (t.to(DEVICE, non_blocking=True) for t in batch); mb = None
              else:
                  (xb, yb), mb = (t.to(DEVICE, non_blocking=True) for t in batch), None
              xb, ya, yb2, lam, mb, idx = mix_batch(xb, yb, mb)
              if use_loc and idx is not None:
                  lb = lam * lb + (1 - lam) * lb[idx]       # location numbers are mixed like the images
              opt.zero_grad(set_to_none=True)
              with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
                  if seg:
                      out, seg_logit = model(xb, return_seg=True)
                  elif use_loc:
                      out = model(xb, loc=lb)
                  else:
                      out = model(xb)
                  loss = lam * ce(out, ya) + (1 - lam) * ce(out, yb2) if lam < 1 else ce(out, ya)
                  if seg:
                      hb_m = hb * hb[idx] if idx is not None else hb
                      loss = loss + SEG_LOSS_WEIGHT * seg_loss_fn(seg_logit, mb, hb_m)
                  if cf:                          # counterfactual rules on half of the batch (unmixed images)
                      sel = torch.nonzero(has[: xb.shape[0] // 2] > 0).squeeze(1)
                      if len(sel):
                          lh = model(xh[sel]).float()
                          loss = loss + CF_HIDE_WEIGHT * (-F.log_softmax(lh, 1).mean(1)).mean()   # -> uniform
                          loss = loss + CF_KEEP_WEIGHT * ce(model(xr[sel]), yb[sel])              # -> same label
              if guide:                           # RRR / CAM-Guide penalty (fp32, double backprop, few images)
                  sel = torch.nonzero(has > 0).squeeze(1)[:GUIDE_BATCH]
                  if len(sel):
                      loss = loss + GUIDE_WEIGHT * guide_penalty(model, x0[sel], reg[sel], yb[sel], guide)
              scaler.scale(loss).backward()
              scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
              scaler.step(opt); scaler.update()
              tot += loss.item()
          logits = (predict_logits_loc(model, val_arrs, val_locs, mean, std) if use_loc
                    else predict_logits(model, val_arrs, mean, std))
          pred = logits.argmax(1)
          f1, acc = macro_f1(val_y, pred), float((pred == val_y).mean())   # (validation crops always contain a tumour)
          hist["train_loss"].append(tot / len(loader)); hist["val_f1"].append(f1)
          hist["val_acc"].append(acc); hist["lr"].append(opt.param_groups[0]["lr"])
          improved = f1 > best["f1"] + 1e-4
          if improved:
              m = model.module if isinstance(model, nn.DataParallel) else model
              best = {"f1": f1, "epoch": epoch, "state": {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}}
              wait = 0
          else:
              wait += 1
          if epoch == 0:
              print(f"    1 epoch = {time.time() - t_start:.0f}s  ->  max {(time.time() - t_start) * max_epochs / 60:.0f} min "
                    f"for this {'test' if not save else 'job'} (stops earlier if no progress for {patience} epochs)", flush=True)
          if show_every and ((epoch + 1) % show_every == 0):
              print(f"    epoch {epoch + 1:2d}: val F1 now {100 * f1:4.1f}% | best so far {100 * best['f1']:4.1f}% "
                    f"| {ram_text()}", flush=True)
          if wait >= patience:
              print(f"    stopped early at epoch {epoch + 1} (no progress for {patience} epochs)", flush=True)
              break
    finally:
        close_loader(loader)
    sec_per_epoch = (time.time() - t_start) / (epoch + 1)
    m = model.module if isinstance(model, nn.DataParallel) else model
    m.load_state_dict(best["state"])
    # final out-of-fold scores END-TO-END (ROI models: box from the predicted tumour map, not the expert mask)
    _CUR_SET["name"] = "figshare"
    logits = predict_any(name, fold, m, val_raw, mean, std)
    pred = hard(softmax_np(logits))           # -1 = no tumour found (abstained)
    rec = {"model": name, "seed": seed, "fold": fold, "lr": lr, "uses_masks": seg, "counterfactual": cf,
           "guide": guide, "best_epoch": best["epoch"] + 1,
           "epochs_run": epoch + 1, "sec_per_epoch": round(sec_per_epoch, 1),
           "val_f1": macro_f1(val_y, pred), "val_acc": float((pred == val_y).mean()),
           "val_auc": macro_auc(val_y, softmax_np(logits)), "n_val": len(val_y),
           "n_params": sum(p.numel() for p in m.parameters()), "history": hist}
    if save:
        ckpt, rec_path, oof_path = run_paths(name, seed, fold)
        # half precision on disk (2x smaller zips); weights are cast back to float when loaded
        torch.save({k: (v.half() if v.is_floating_point() else v) for k, v in best["state"].items()}, ckpt)
        rec["ckpt"], rec["sha256"] = ckpt.name, sha256_file(ckpt)
        rec["val_paths"], rec["val_labels"] = [p for p, _ in val_items], val_y.tolist()
        np.save(oof_path, logits)
        save_json(rec, rec_path)
    del model, m, opt, loader
    gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return rec

In [ ]:
# ─── 3.2 (Session 1 only) Short learning-rate search on fold 1 ───
# Same budget for every model: max 30 epochs, patience 8, fold 1, seed 0. Chosen LR = best val macro-F1.
if RUN_TRAINING and RUN_LR_SEARCH:
    choice = load_json(OUT_DIR / "lr_choice.json", {})
    for name in LR_SEARCH_MODELS:
        if name in choice:
            continue
        if (time.time() - SESSION_START) / 3600 > MAX_TRAIN_HOURS:
            print("Time budget reached during LR search - it continues in the next run."); break
        grid = LR_GRID["pretrained" if is_pretrained(name) else "scratch"]
        k = LR_SEARCH_MODELS.index(name) + 1
        banner(f"STEP 1/2  Picking the learning rate for {name}  (model {k} of {len(LR_SEARCH_MODELS)})",
               f"short test runs on fold 1, nothing saved | time used {hms(time.time() - SESSION_START)}")
        scores = {}
        for i, lr in enumerate(grid, 1):
            print(f"  Test {i}/{len(grid)}: learning rate {lr} ...", flush=True)
            scores[lr] = train_one(name, 0, 1, lr=lr, max_epochs=30, patience=8, save=False, show_every=0)["val_f1"]
            print(f"  Test {i}/{len(grid)}: learning rate {lr} -> score (val F1) {100 * scores[lr]:.1f}%", flush=True)
        choice[name] = max(scores, key=scores.get)
        print(f"\n  ✔ {name} will be trained with learning rate {choice[name]}  "
              f"(" + ", ".join(f"{l}: {100 * v:.1f}%" for l, v in scores.items()) + ")")
        save_json(choice, OUT_DIR / "lr_choice.json")
if RUN_TRAINING:
    print("\nLearning rates (ablations reuse their parent's LR):")
    for n in MODEL_ORDER:
        print(f"   {n:18s} {get_lr(n)}" + ("" if n in load_json(OUT_DIR / "lr_choice.json", {}) or n in LR_INHERIT
                                            else "   (not tested yet)"))

In [ ]:
# ─── 3.3 Train the queue (skips finished runs; stops before the Kaggle time limit) ───
lr_done = all(m in load_json(OUT_DIR / "lr_choice.json", {}) for m in LR_SEARCH_MODELS) or not RUN_LR_SEARCH
if RUN_TRAINING and lr_done:
    jobs = [(n, s, f) for n, seeds in TRAIN_QUEUE for s in seeds for f in range(1, N_FOLDS + 1)]
    n_done0 = sum(run_done(*j) for j in jobs)
    banner("STEP 2 of 2 - TRAINING THE REAL MODELS (saved, used in the paper)",
           f"{n_done0} of {len(jobs)} jobs already finished. 1 job = 1 model x 1 seed x 1 fold (max {NUM_EPOCHS} epochs).\n"
           f"  Training stops by itself after {MAX_TRAIN_HOURS} h; the rest continues in the next run.\n"
           f"  elapsed {hms(time.time() - SESSION_START)} | {ram_text()}")
    for name, seeds in TRAIN_QUEUE:
        for seed in seeds:
            for fold in range(1, N_FOLDS + 1):
                if run_done(name, seed, fold):
                    continue                                   # already finished in an earlier run
                if (time.time() - SESSION_START) / 3600 > MAX_TRAIN_HOURS:
                    print("Time budget reached - remaining runs left for the next session."); break
                n_done = sum(run_done(*j) for j in jobs)
                print("\n" + "-" * 70 + f"\nJOB {n_done + 1}/{len(jobs)} | {name} | seed {seed} | fold {fold} of {N_FOLDS} "
                      f"| lr {get_lr(name)}\n  {role_of(name)} | elapsed {hms(time.time() - SESSION_START)}\n" + "-" * 70,
                      flush=True)
                r = train_one(name, seed, fold)
                print(f"  ✔ JOB {n_done + 1} SAVED: val F1 {100 * r['val_f1']:.1f}% | val acc {100 * r['val_acc']:.1f}% "
                      f"| best epoch {r['best_epoch']} of {r['epochs_run']} run")

In [ ]:
# ─── 3.4 Which runs are complete? ───
todo = [(n, s, f) for n, seeds in TRAIN_PLAN for s in seeds for f in range(1, N_FOLDS + 1) if not run_done(n, s, f)]
total = sum(len(seeds) * N_FOLDS for _, seeds in TRAIN_PLAN)
print("=" * 70)
if not lr_done:
    print("STATUS: learning-rate search not finished -> run the notebook again.")
elif todo:
    print(f"STATUS: {total - len(todo)} of {total} training runs done -> run the notebook again.")
    print(f"        next up: {todo[0][0]} seed {todo[0][1]} fold {todo[0][2]}")
else:
    print("STATUS: ALL TRAINING DONE. The tables below are final.")
print("=" * 70)
def complete_seeds(name):
    return [s for s in range(10) if all(run_done(name, s, f) for f in range(1, N_FOLDS + 1))]

READY = {n: complete_seeds(n) for n in MODEL_ORDER}
READY = {n: s for n, s in READY.items() if s}
plan_seeds = defaultdict(set)
for n, seeds in TRAIN_PLAN:
    plan_seeds[n] |= set(seeds)
print(f"\n{'Model':15s} {'seed 0':>7s} {'seed 1':>7s} {'seed 2':>7s}   role")
for n in MODEL_ORDER:
    if n not in plan_seeds:
        continue
    cells = []
    for s in range(3):
        if s not in plan_seeds[n]:
            cells.append("  -  ")
        else:
            k = sum(run_done(n, s, f) for f in range(1, N_FOLDS + 1))
            cells.append("  ✔  " if k == N_FOLDS else f" {k}/5 ")
    print(f"{n:15s} {cells[0]:>7s} {cells[1]:>7s} {cells[2]:>7s}   {role_of(n)[:60]}")
print("(✔ = all 5 folds trained; only models with ✔ appear in the tables below)")

# The proposed model (pre-registered): ROI-Loc; until it is trained, ROI-Crop (then CTR-Net, MTK)
PROPOSED = next((n for n in ["ROI-Loc", "ROI-Crop", "CTR-Net", "MTK"] if n in READY), "MTK")
print(f"Main model in all comparisons: {PROPOSED}  ({role_of(PROPOSED)})")
XAI_MODELS = [n for n in ["CNX-Transplant", "CTR-Net", "CTR-noCF", "MTL-Seg", "RRR", "CAM-Guide", "CNX-Plain", "MTK", "ResNet-50"] if n in READY]
if RUN_XAI == "auto":
    main_needed = ["CTR-Net", "CTR-noCF", "MTL-Seg", "RRR", "CAM-Guide", "CNX-Plain"]
    RUN_XAI = all(n in READY for n in main_needed)
    print("Explainability (section 9) will run now (cached per model, so later runs are fast)." if RUN_XAI else
          "Explainability (section 9) waits until CTR-Net and the comparison methods are trained.")

In [ ]:
# ─── 3.5 Save the artifact ZIP now (so training is safe even if a later cell fails) ───
make_artifact_zip()

In [ ]:
# ─── 3.6 ⬇ DOWNLOAD NOW (training is finished; everything below only makes tables and figures) ───
from IPython.display import FileLink, display, HTML
def show_downloads():
    zips = sorted(WORK.glob(f"{ZIP_PREFIX}_*.zip"))
    display(HTML("<div style='padding:10px;border:3px solid #d1495b;border-radius:8px;font-size:15px'>"
                 "<b>⬇ Download these files NOW</b> (click each link; keep this tab open until all are saved).<br>"
                 "Then upload them as ONE new Kaggle dataset (e.g. <code>mtk-run5-artifacts</code>) and attach it next run, "
                 "together with <code>mtk-run3-part2</code> and <code>mtk_planA_artifacts</code>.</div>"))
    for z in zips:
        display(FileLink(str(z.relative_to(WORK)), result_html_prefix=f"{z.stat().st_size / 1e9:.2f} GB &nbsp; "))
    if not zips:
        print("No zip found - run cell 3.5 again.")
show_downloads()

## 4. Cross-validation results (development patients)

In [ ]:
# ─── 4.1 CV table: mean ± SD over all (seed, fold) runs ───
_REC = {}
def rec_of(name, s, f):
    """Training record with the validation metrics RECOMPUTED from the saved out-of-fold logits, so that
    'no tumour found' (ROI models) counts as abstained, not as glioma (fix of Run 6)."""
    key = (name, s, f)
    if key not in _REC:
        _, rec_path, oof_path = run_paths(name, s, f)
        r = dict(load_json(rec_path)); z = np.load(oof_path); y = np.array(r["val_labels"]); P = softmax_np(z)
        pr = hard(P)
        r.update(val_f1=macro_f1(y, pr), val_acc=float((pr == y).mean()), val_auc=macro_auc(y, P),
                 val_abstain=float((pr < 0).mean()))
        _REC[key] = r
    return _REC[key]

def run_records(name):
    return [rec_of(name, s, f) for s in READY.get(name, []) for f in range(1, N_FOLDS + 1)]

cv_rows = []
for name in MODEL_ORDER:
    recs = run_records(name)
    if not recs:
        continue
    f1s, accs, aucs = [r["val_f1"] for r in recs], [r["val_acc"] for r in recs], [r["val_auc"] for r in recs]
    cv_rows.append({"Model": name, "Seeds": len(READY[name]), "Runs": len(recs),
                    "Macro-F1 (%)": f"{100*np.mean(f1s):.2f} ± {100*np.std(f1s, ddof=1):.2f}",
                    "Accuracy (%)": f"{100*np.mean(accs):.2f} ± {100*np.std(accs, ddof=1):.2f}",
                    "Macro-AUC": f"{np.mean(aucs):.4f} ± {np.std(aucs, ddof=1):.4f}",
                    "Best epoch (median)": int(np.median([r["best_epoch"] for r in recs])),
                    "Abstained (%)": round(100 * np.mean([r["val_abstain"] for r in recs]), 1),
                    "_f1": np.mean(f1s)})
df_cv = pd.DataFrame(cv_rows)
if len(df_cv):
    print(df_cv.drop(columns="_f1").to_string(index=False))
    df_cv.drop(columns="_f1").to_csv(OUT_DIR / "tables" / "table_cv.csv", index=False)

In [ ]:
# ─── 4.2 Corrected resampled t-test (Nadeau & Bengio) vs the proposed model on matched (seed, fold) runs ───
def corrected_ttest(a, b, ratio):
    d = np.asarray(a) - np.asarray(b); J = len(d)
    if J < 3 or d.std(ddof=1) == 0:
        return np.nan, np.nan
    t = d.mean() / math.sqrt((1 / J + ratio) * d.var(ddof=1))
    return float(t), float(2 * sps.t.sf(abs(t), J - 1))

ratio = np.mean([len(FOLDS[k]["val"]) / len(FOLDS[k]["train"]) for k in FOLDS])
cv_stat_rows = []
if PROPOSED in READY:
    for name in READY:
        if name == PROPOSED:
            continue
        pairs = [(s, f) for s in set(READY[name]) & set(READY[PROPOSED]) for f in range(1, N_FOLDS + 1)]
        a = [rec_of(PROPOSED, s, f)["val_f1"] for s, f in pairs]
        b = [rec_of(name, s, f)["val_f1"] for s, f in pairs]
        t, p = corrected_ttest(a, b, ratio)
        cv_stat_rows.append({"Comparison": f"{PROPOSED} vs {name}", "Runs": len(pairs),
                             "ΔF1 (pp)": round(100 * (np.mean(a) - np.mean(b)), 2), "t": t, "p": p})
df_cv_stats = pd.DataFrame(cv_stat_rows)
if len(df_cv_stats):
    ok_ = df_cv_stats.p.notna()
    holm = np.full(len(df_cv_stats), np.nan); pv = df_cv_stats.p[ok_].values; o = np.argsort(pv); m_ = len(pv)
    adj = np.maximum.accumulate([min(1, (m_ - i) * pv[j]) for i, j in enumerate(o)]) if m_ else []
    tmp = np.empty(m_); tmp[o] = adj; holm[ok_.values] = tmp
    df_cv_stats["p (Holm)"] = np.round(holm, 4); df_cv_stats["t"] = df_cv_stats.t.round(2); df_cv_stats["p"] = df_cv_stats.p.round(4)
    print(df_cv_stats.to_string(index=False))
    df_cv_stats.to_csv(OUT_DIR / "tables" / "table_cv_ttest.csv", index=False)

In [ ]:
# ─── 4.3 Learning curves (proposed vs its closest baselines, seed 0) ───
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
for name, col in [(PROPOSED, "#d1495b"), ("CNX-Plain" if PROPOSED == "CTR-Net" else "CNN-Backbone", "#30638e"),
                  ("CTR-noCF" if PROPOSED == "CTR-Net" else "MTK-noSeq", "#888888")]:
    if name not in READY:
        continue
    for f in range(1, N_FOLDS + 1):
        h = load_json(run_paths(name, READY[name][0], f)[1])["history"]
        ax[0].plot(h["train_loss"], color=col, alpha=.5, lw=1, label=name if f == 1 else None)
        ax[1].plot(h["val_f1"], color=col, alpha=.5, lw=1, label=name if f == 1 else None)
ax[0].set(title="Training loss", xlabel="Epoch"); ax[1].set(title="Validation macro-F1", xlabel="Epoch")
ax[1].legend(frameon=False); fig.tight_layout(); fig.savefig(FIG_DIR / "learning_curves.png"); plt.show()

## 5. Inference on all evaluation sets (cached)
Every (model, seed, fold) checkpoint is applied to: internal test, BRISC, Mendeley, BraTS.
Final predictions = mean softmax over the 5 fold models (and over seeds). No best-fold selection.

In [ ]:
# ─── 5.1 Predictions (only the images that are actually used; cached) ───
EVAL_SETS = {"test": TEST_ITEMS, "brisc": BRISC_CLEAN, "mendeley": MEND_CLEAN,
             "brats": BRATS_ITEMS, "bdneuro": BD_CLEAN, "bfree": BFREE_ITEMS}      # unique images only
EVAL_SETS = {k: v for k, v in EVAL_SETS.items() if len(v)}

def pred_path(name, seed, fold, set_name):
    return OUT_DIR / "preds" / f"{name}_s{seed}_f{fold}_{set_name}.npy"

_ARCH = {"name": None, "model": None}
_HASH_OK = set()

def free_mem():
    gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    try:
        import ctypes; ctypes.CDLL("libc.so.6").malloc_trim(0)      # give freed RAM back to the system
    except Exception:
        pass

def load_checkpoint(name, seed, fold):
    """The architecture is built once per model name and re-used for every fold / seed."""
    ckpt, rec_path, _ = run_paths(name, seed, fold)
    if ckpt not in _HASH_OK:
        rec = load_json(rec_path)
        if rec.get("sha256") and sha256_file(ckpt) != rec["sha256"]:
            raise RuntimeError(f"Checkpoint hash mismatch: {ckpt}")
        _HASH_OK.add(ckpt)
    if _ARCH["name"] != name:
        _ARCH["model"] = None; free_mem()
        _ARCH["name"], _ARCH["model"] = name, build_model(name, pretrained=False).to(DEVICE)
    m = _ARCH["model"]
    m.load_state_dict(torch.load(ckpt, map_location=DEVICE), strict=True)
    m.zero_grad(set_to_none=True)
    return m.eval()

def cached_ok(path, n):
    try:
        return path.exists() and np.load(path, mmap_mode="r").shape[0] == n
    except Exception:                      # half-written file from a stopped run -> redo it
        return False

if RUN_EVALUATION:
    banner("5.1 Predictions on all evaluation sets",
           "  ".join(f"{k}: {len(v)}" for k, v in EVAL_SETS.items()))
    ARRS = {s: [load_u8(p) for p, _ in items] for s, items in EVAL_SETS.items()}   # read images once
    jobs = [(n, s, f) for n in READY for s in READY[n] for f in range(1, N_FOLDS + 1)]
    todo_jobs = [j for j in jobs if not all(cached_ok(pred_path(*j, s), len(EVAL_SETS[s])) for s in EVAL_SETS)]
    print(f"{len(jobs)} fold-models in total | {len(jobs) - len(todo_jobs)} already cached | {len(todo_jobs)} to predict now\n")
    t_all = time.time()
    for k, (name, seed, fold) in enumerate(todo_jobs, 1):
        t0 = time.time()
        sets = [s for s in EVAL_SETS if not cached_ok(pred_path(name, seed, fold, s), len(EVAL_SETS[s]))]
        model = load_checkpoint(name, seed, fold); mean, std = norm_for(name, fold)
        for s in sets:
            _CUR_SET["name"] = s
            np.save(pred_path(name, seed, fold, s), predict_any(name, fold, model, ARRS[s], mean, std))
        _CUR_SET["name"] = None
        left = (time.time() - t_all) / k * (len(todo_jobs) - k)
        print(f"  [{k:3d}/{len(todo_jobs)}] {name:14s} seed {seed} fold {fold} | {time.time() - t0:4.0f}s "
              f"| about {left / 60:.0f} min left", flush=True)
    free_mem()
    print(f"\n✔ All predictions ready ({(time.time() - t_all) / 60:.1f} min)")

In [ ]:
# ─── 5.2 Temperature scaling (fit per fold model on its own validation fold) ───
def fit_temperature(logits, y):
    grid = np.exp(np.linspace(np.log(0.25), np.log(8), 300))
    nll = [-np.log(softmax_np(logits / t)[np.arange(len(y)), y] + 1e-12).mean() for t in grid]
    return float(grid[int(np.argmin(nll))])

TEMPS = {}
for name in READY:
    for s in READY[name]:
        for f in range(1, N_FOLDS + 1):
            _, rec_path, oof_path = run_paths(name, s, f)
            TEMPS[(name, s, f)] = fit_temperature(np.load(oof_path), np.array(load_json(rec_path)["val_labels"]))

def ensemble_probs(name, set_name, calibrated=False, seeds=None):
    """Mean softmax of the 5 fold models. FAIR COMPARISON: by default only the FIRST seed is used for
    every model (models with 3 seeds are not given a bigger ensemble). seeds="all" uses every seed."""
    seeds = READY[name][:1] if seeds is None else (READY[name] if seeds == "all" else seeds)
    ps = []
    for s in seeds:
        for f in range(1, N_FOLDS + 1):
            z = np.load(pred_path(name, s, f, set_name))
            ps.append(softmax_np(z / (TEMPS[(name, s, f)] if calibrated else 1.0)))
    return np.mean(ps, axis=0)

def labels_of(set_name):
    return np.array([y for _, y in EVAL_SETS[set_name]])

def paths_of(set_name):
    return [p for p, _ in EVAL_SETS[set_name]]

## 5.3 Tumour-reliance test: does the model still answer when the tumour is hidden?
Every Figshare slice is predicted by the fold model that never trained on it (test slices: mean of the 5 fold models),
in 6 versions. **Main measure = "hidden (swap)"**, a hiding method that CTR-Net never saw during training.
**Tumour-reliance score** = (accuracy drop when the tumour is hidden) / (accuracy above guessing):
0 = the tumour is not needed at all, 1 = without the tumour the model is at guessing level (what a doctor would be).

**Run 6 fix - "unsure" answers:** an ROI model that finds no tumour outputs exactly equal probabilities. These cases are
now counted as **abstained** (shown in their own column). For balanced accuracy an abstained answer counts as 1/3
correct (= a random guess); for accuracy / F1 / recall it counts as a miss. (Before Run 6, numpy's argmax silently
turned every "unsure" into "glioma".)

In [ ]:
# ─── 5.3 Tumour-reliance test (cached per model) ───
REL_CONDS = {"original": None, "hidden (swap)": "swap", "hidden (inpaint)": "inpaint",
             "random spot hidden": "random", "skull / position only": "skull", "tumour only": "tonly"}
REL_DF = df3[df3.filepath.isin(HIDE)].reset_index(drop=True) if HIDE else pd.DataFrame()
REL_Y = REL_DF.label.values if len(REL_DF) else np.array([])
REL_CTRL = np.array([HIDE[p]["has_control"] for p in REL_DF.filepath]) if len(REL_DF) else np.array([])

def bal_acc(y, pred):
    """Balanced accuracy. An ABSTAINED answer (-1, ROI model found no tumour) counts as 1/3 correct = the expected
    value of a random guess, so 'unsure' is neither rewarded (old argmax bug: always glioma) nor punished below chance."""
    pred = np.asarray(pred)
    return float(np.mean([((pred[y == k] == k) + (pred[y == k] == -1) / N_CLASSES).mean()
                          for k in range(N_CLASSES) if (y == k).any()]))

def reliance_probs(name, seed=0):
    """(condition -> (N,3) probabilities) for all REL_DF slices, out-of-fold for development slices."""
    out = {c: np.zeros((len(REL_DF), N_CLASSES), np.float32) for c in REL_CONDS}
    test_idx = np.where(REL_DF.role == "test")[0]
    for f in range(1, N_FOLDS + 1):
        idx = np.concatenate([np.where((REL_DF.role == "dev") & (REL_DF.val_fold == f))[0], test_idx])
        cache = OUT_DIR / "reliance" / f"{name}_s{seed}_f{f}.npz"
        z = np.load(cache) if cache.exists() else None
        if z is None or z["original"].shape[0] != len(idx):
            m = load_checkpoint(name, seed, f); mean, std = norm_for(name, f)
            res = {}
            for c, key in REL_CONDS.items():
                arrs = [load_u8(p) if key is None else HIDE[p][key] for p in REL_DF.filepath.values[idx]]
                res[c] = softmax_np(predict_any(name, f, m, arrs, mean, std))
            cache.parent.mkdir(exist_ok=True); np.savez_compressed(cache, **res); z = res
            del m; gc.collect()
        n_dev = len(idx) - len(test_idx)
        for c in REL_CONDS:
            out[c][idx[:n_dev]] = z[c][:n_dev]
            out[c][test_idx] += z[c][n_dev:] / N_FOLDS
    return out

REL, rel_rows, rel_cls_rows = {}, [], []
if RUN_EVALUATION and RUN_RELIANCE and len(REL_DF):
    banner("5.3 Tumour-reliance test", f"{len(READY)} models x 6 versions of {len(REL_DF)} slices "
           "(first time ~1 min per model, then cached)")
    for name in READY:
        REL[name] = reliance_probs(name, READY[name][0])
        ba = {c: bal_acc(REL_Y, hard(REL[name][c])) for c in REL_CONDS}
        o, h, hi_ = (bal_acc(REL_Y[REL_CTRL], hard(REL[name][c][REL_CTRL]))
                     for c in ["original", "hidden (swap)", "random spot hidden"])
        score = lambda hid: (float(np.clip((ba["original"] - hid) / (ba["original"] - 1 / 3), 0, 1))
                             if ba["original"] > 0.45 else np.nan)       # undefined for a model near guessing
        ent = -(REL[name]["hidden (swap)"] * np.log(REL[name]["hidden (swap)"] + 1e-12)).sum(1).mean() / np.log(3)
        rel_rows.append({"Model": name, **{c: round(100 * v, 1) for c, v in ba.items()},
                         "Control drop (pts)": round(100 * (o - hi_), 1),
                         "Reliance (swap)": round(score(ba["hidden (swap)"]), 2),
                         "Reliance (inpaint)": round(score(ba["hidden (inpaint)"]), 2),
                         "Uncertainty when hidden (0-1)": round(float(ent), 2)})
        pr = hard(REL[name]["hidden (swap)"])
        rel_cls_rows.append({"Model": name, **{f"{c} still correct (%)": round(100 * (pr[REL_Y == k] == k).mean(), 1)
                                               for k, c in enumerate(CLASSES)},
                             "abstained when hidden (%)": round(100 * (pr < 0).mean(), 1)})
    df_rel = pd.DataFrame(rel_rows).sort_values("Reliance (swap)", ascending=False)
    RELIANCE = dict(zip(df_rel.Model, df_rel["Reliance (swap)"]))
    print("Balanced accuracy (%) per version  |  guessing = 33.3%")
    print(df_rel[["Model", "original", "hidden (swap)", "hidden (inpaint)", "random spot hidden",
                  "skull / position only", "Reliance (swap)", "Uncertainty when hidden (0-1)"]].to_string(index=False))
    print("\nHow to read: a model that really uses the tumour falls towards 33% when the tumour is hidden "
          "(reliance close to 1) and becomes uncertain (uncertainty close to 1).")
    print("Control: hiding a random spot of the same size changes accuracy by at most "
          f"{df_rel['Control drop (pts)'].abs().max():.1f} points -> the test is fair.")
    df_rel.to_csv(OUT_DIR / "tables" / "table_reliance.csv", index=False)
    df_relc = pd.DataFrame(rel_cls_rows); df_relc.to_csv(OUT_DIR / "tables" / "table_reliance_by_class.csv", index=False)
    print("\nWith the tumour hidden (swap), share of slices still classified correctly, per class:")
    print(df_relc.set_index("Model").loc[df_rel.Model].to_string())

    order = list(df_rel.Model)
    fig, ax = plt.subplots(figsize=(max(8, 0.75 * len(order)), 3.8))
    xs = np.arange(len(order)); w = 0.22
    for k, (c, col) in enumerate([("original", "#30638e"), ("hidden (swap)", "#d1495b"),
                                  ("random spot hidden", "#bbbbbb"), ("skull / position only", "#edae49")]):
        ax.bar(xs + (k - 1.5) * w, df_rel.set_index("Model").loc[order, c], w, color=col, label=c)
    ax.axhline(100 / 3, color="k", ls="--", lw=1); ax.text(len(order) - 0.5, 35, "guessing", fontsize=7, ha="right")
    ax.set_xticks(xs); ax.set_xticklabels([f"{n}\nR={RELIANCE[n]:.2f}" for n in order], fontsize=7)
    ax.set(ylabel="Balanced accuracy (%)", ylim=(0, 100),
           title="Tumour-hiding test (R = tumour-reliance score; higher = decides from the tumour)")
    ax.legend(frameon=False, fontsize=7, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.2))
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig_reliance_test.png", bbox_inches="tight"); plt.show()
else:
    RELIANCE = {}

## 5.4 Is the tumour-reliance score robust? (metric check, experiment 5)
The main test hides the expert tumour outline + a 9-pixel ring with tissue from the same slice ("swap").
Here the same test is repeated with **bigger hidden areas** (25 and 41 pixel rings: these also hide the swelling /
oedema around a tumour) and with a **second hiding method**: the area is filled with tissue from the SAME place in a
slice of ANOTHER patient (same viewing-plane cluster, brightness matched). If the ranking of the models stays the
same, the score measures a property of the model, not of our hiding trick.

In [ ]:
# ─── 5.4 Metric check: bigger hidden areas + a second hiding method (cached per model) ───
from scipy.stats import spearmanr
REL2_KEYS = [f"swap {k}" for k in METRIC_SIZES] + ["other patient 9"]
REL2, rel2_rows = {}, []
if RUN_EVALUATION and RUN_RELIANCE and len(REL_DF):
    banner("5.4 Metric check", "building the extra hidden versions once (~3-5 min), then ~1 min per model")
    PID_OF = dict(zip(df3.filepath, df3.pid))
    CLUSTER_OF = dict(zip(df_plane.filepath, df_plane.cluster)) if len(df_plane) else {}
    CL_PATHS = defaultdict(list)
    for p_, c_ in CLUSTER_OF.items():
        if p_ in MASKS: CL_PATHS[c_].append(p_)
    rng5 = np.random.default_rng(5)

    def swap_hide(img, reg, bm):
        sh = shifted_region(reg, bm, rng5)
        if sh is None:
            return None
        dy, dx, _ = sh; ys, xs = np.where(reg); out = img.copy(); out[ys, xs] = img[ys + dy, xs + dx]
        return out

    def other_patient_hide(p, img, reg, bm):
        cands = CL_PATHS.get(CLUSTER_OF.get(p), [])
        for _ in range(40):
            if not cands:
                return None
            q = cands[rng5.integers(len(cands))]
            if PID_OF[q] == PID_OF[p]:
                continue
            if (cv2.dilate(MASKS[q].astype(np.uint8), HIDE_KER, iterations=2).astype(bool) & reg).any():
                continue
            aq = load_u8(q); bq = brain_bm(aq, q)
            if bq[reg].mean() < 0.95:
                continue
            k_ = (np.median(img[bm]) + 1e-3) / (np.median(aq[bq]) + 1e-3)
            out = img.copy(); out[reg] = np.clip(aq[reg].astype(np.float32) * k_, 0, 255).astype(np.uint8)
            return out
        return None

    VAR = {k: [None] * len(REL_DF) for k in REL2_KEYS}
    for i, p in enumerate(tqdm(REL_DF.filepath.values, desc="  building", mininterval=10)):
        img = load_u8(p); bm = brain_bm(img, p)
        for k in METRIC_SIZES:
            reg = cv2.dilate(MASKS[p].astype(np.uint8), cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))).astype(bool)
            VAR[f"swap {k}"][i] = HIDE[p]["swap"] if (k == 9 and HIDE[p]["has_control"]) else (swap_hide(img, reg, bm) if k != 9 else None)
            if k == 9:
                VAR["other patient 9"][i] = other_patient_hide(p, img, reg, bm)
    AVAIL = np.array([all(VAR[k][i] is not None for k in REL2_KEYS) for i in range(len(REL_DF))])
    print(f"  slices where every hiding version could be made: {AVAIL.sum()} of {len(REL_DF)} "
          f"({100 * AVAIL.mean():.1f}%; big tumours cannot be moved/filled) -> all numbers below use these slices")

    def reliance2_probs(name, seed=0):
        out = {k: np.zeros((len(REL_DF), N_CLASSES), np.float32) for k in REL2_KEYS}
        test_idx = np.where((REL_DF.role == "test").values & AVAIL)[0]
        for f in range(1, N_FOLDS + 1):
            idx = np.concatenate([np.where((REL_DF.role == "dev").values & (REL_DF.val_fold == f).values & AVAIL)[0], test_idx])
            cache = OUT_DIR / "reliance2" / f"{name}_s{seed}_f{f}.npz"
            z = np.load(cache) if cache.exists() else None
            if z is None or z[REL2_KEYS[0]].shape[0] != len(idx):
                m = load_checkpoint(name, seed, f); mean, std = norm_for(name, f); res = {}
                for k in REL2_KEYS:
                    res[k] = softmax_np(predict_any(name, f, m, [VAR[k][i] for i in idx], mean, std))
                cache.parent.mkdir(exist_ok=True); np.savez_compressed(cache, **res); z = res
            n_dev = len(idx) - len(test_idx)
            for k in REL2_KEYS:
                out[k][idx[:n_dev]] = z[k][:n_dev]; out[k][test_idx] += z[k][n_dev:] / N_FOLDS
        return out

    yA = REL_Y[AVAIL]
    for name in READY:
        REL2[name] = reliance2_probs(name, READY[name][0])
        o = bal_acc(yA, hard(REL[name]["original"][AVAIL]))
        row = {"Model": name, "original": round(100 * o, 1)}
        for k in REL2_KEYS:
            h = bal_acc(yA, hard(REL2[name][k][AVAIL]))
            row[f"BA {k}"] = round(100 * h, 1)
            row[f"R {k}"] = round(float(np.clip((o - h) / (o - 1 / 3), 0, 1)), 2) if o > 0.45 else np.nan
        pr = hard(REL2[name]["swap 41"][AVAIL])
        row["glioma still correct, swap 41 (%)"] = round(100 * (pr[yA == 0] == 0).mean(), 1)
        row["abstained, swap 41 (%)"] = round(100 * (pr < 0).mean(), 1)
        rel2_rows.append(row)
    df_rel2 = pd.DataFrame(rel2_rows).sort_values("R swap 9", ascending=False)
    print("Tumour-reliance score R with different hidden areas and a second hiding method (higher = uses the tumour):")
    print(df_rel2[["Model", "original"] + [f"R {k}" for k in REL2_KEYS] + ["glioma still correct, swap 41 (%)"]].to_string(index=False))
    df_rel2.to_csv(OUT_DIR / "tables" / "table_metric_check.csv", index=False)
    ok_ = df_rel2[["R swap 9", "R other patient 9"]].dropna()
    RHO_METRIC = spearmanr(ok_["R swap 9"], ok_["R other patient 9"])[0] if len(ok_) >= 4 else np.nan
    print(f"\nRanking stability: Spearman(swap, other patient) = {RHO_METRIC:.2f}  (criterion S5: >= 0.80)")

    key_models = [n for n in [PROPOSED, "ROI-m06-Transplant", "ROI-Crop-m06", "ROI-Crop", "CNX-Transplant", "CNX-Strip", "CNX-Plain", "CTR-Net", "MTL-Seg",
                              "CAM-Guide", "MTK", "EfficientNet-B0"] if n in REL2]
    key_models = list(dict.fromkeys(key_models))
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    d2 = df_rel2.set_index("Model")
    for n in key_models:
        ax[0].plot(METRIC_SIZES, [d2.loc[n, f"R swap {k}"] for k in METRIC_SIZES], "-o", ms=4,
                   lw=2 if n == PROPOSED else 1.2, color=FAMILY_COLOR[family_of(n)], label=n)
    ax[0].set(xlabel="Size of the hidden ring around the tumour (pixels)", ylabel="Tumour-reliance score R",
              ylim=(0, 1.02), xticks=METRIC_SIZES, title="Hiding more of the surroundings (incl. oedema)")
    ax[0].legend(frameon=False, fontsize=6, ncol=2)
    for n in d2.index:
        ax[1].scatter(d2.loc[n, "R swap 9"], d2.loc[n, "R other patient 9"], color=FAMILY_COLOR[family_of(n)], s=28, zorder=3)
        ax[1].annotate(n, (d2.loc[n, "R swap 9"], d2.loc[n, "R other patient 9"]), fontsize=5.5, xytext=(2, 2), textcoords="offset points")
    ax[1].plot([0, 1], [0, 1], color="#999999", ls="--", lw=1)
    ax[1].set(xlabel="R, hidden with tissue from the same slice (main test)", ylabel="R, hidden with another patient's tissue",
              xlim=(0, 1), ylim=(0, 1), title=f"Two hiding methods agree? Spearman ρ = {RHO_METRIC:.2f}")
    fig.suptitle("Metric check: is the tumour-reliance score a property of the MODEL (robust) or of the hiding trick?", fontsize=9)
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig_metric_check.png", bbox_inches="tight"); plt.show()
else:
    df_rel2, RHO_METRIC = pd.DataFrame(), np.nan

In [ ]:
# ─── 5.5 Why do ROI models still name GLIOMAS when the tumour is hidden? (look at what the tumour map finds) ───
ROI_REF = next((n for n in [PROPOSED, "ROI-Crop"] if n in REL and is_roi(n)), None)
if RUN_EVALUATION and ROI_REF and len(REL_DF):
    gi = np.where((REL_Y == 0) & (REL_DF.role == "dev").values & REL_CTRL)[0]
    found_any, inside, outside, rows_g = 0, [], [], []
    rng_g = np.random.default_rng(11); show_g = set(rng_g.choice(gi, min(8, len(gi)), replace=False).tolist())
    maps_show = {}
    for f in range(1, N_FOLDS + 1):
        ii = [i for i in gi if REL_DF.val_fold.values[i] == f]
        if not ii:
            continue
        maps = roi_seg_maps([HIDE[REL_DF.filepath.values[i]]["swap"] for i in ii], f)
        for i, mp in zip(ii, maps):
            b = box_from_mask(mp >= 0.5); reg = HIDE[REL_DF.filepath.values[i]]["region"]
            if b is not None:
                found_any += 1; comp = b[4]
                inside.append((comp & reg).sum() / comp.sum())
                ring = cv2.dilate(reg.astype(np.uint8), cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (41, 41))).astype(bool) & ~reg
                outside.append((comp & ring).sum() / comp.sum())
            if i in show_g:
                maps_show[i] = (mp, b)
    pr_h = hard(REL[ROI_REF]["hidden (swap)"])
    print(f"Glioma slices with the tumour hidden (swap), development folds: {len(gi)}")
    print(f"  the tumour map still finds 'a tumour' in {100 * found_any / max(len(gi), 1):.1f}% of them")
    if inside:
        print(f"  of the found region: {100 * np.mean(inside):.1f}% lies inside the hidden area, "
              f"{100 * np.mean(outside):.1f}% in the 20-pixel ring AROUND it (oedema / mass effect), rest elsewhere")
    print(f"  {ROI_REF} still says 'glioma' for {100 * (pr_h[gi] == 0).mean():.1f}% of them")
    if maps_show:
        ks = sorted(maps_show); fig, ax = plt.subplots(2, len(ks), figsize=(1.9 * len(ks), 4.2), squeeze=False)
        for c, i in enumerate(ks):
            p = REL_DF.filepath.values[i]; mp, b = maps_show[i]
            ax[0, c].imshow(load_u8(p), cmap="gray"); ax[0, c].contour(MASKS[p], levels=[0.5], colors="lime", linewidths=0.7)
            ax[1, c].imshow(HIDE[p]["swap"], cmap="gray")
            ax[1, c].contour(HIDE[p]["region"], levels=[0.5], colors="lime", linewidths=0.6, linestyles="--")
            if b is not None:
                ax[1, c].contour(mp >= 0.5, levels=[0.5], colors="red", linewidths=0.7)
                x, y, w, h, _ = b; s_ = max(w, h) * 1.4
                ax[1, c].add_patch(plt.Rectangle((x + w / 2 - s_ / 2, y + h / 2 - s_ / 2), s_, s_, fill=False, ec="cyan", lw=0.9))
            ax[1, c].set_xlim(0, IMG_SIZE); ax[1, c].set_ylim(IMG_SIZE, 0)
            ax[1, c].set_title(f"→ {IDX2CLASS[int(pr_h[i])][:6]} ({REL[ROI_REF]['hidden (swap)'][i].max():.2f})", fontsize=6,
                               color="green" if pr_h[i] == 0 else "red")
            for a in ax[:, c]: a.axis("off")
        ax[0, 0].set_title("original (green = tumour)", fontsize=6, loc="left")
        fig.suptitle(f"Glioma with the tumour hidden: dashed green = hidden area, red = what the tumour map still finds, "
                     f"cyan = {ROI_REF}'s crop", fontsize=7.5)
        fig.tight_layout(); fig.savefig(FIG_DIR / "fig_glioma_hidden_roi.png", bbox_inches="tight"); plt.show()

## 6. Internal held-out test (35 patients, never used for training or selection)

In [ ]:
# ─── 6.1 Image- and patient-level metrics + patient-clustered bootstrap CIs ───
TEST_Y = labels_of("test") if "test" in EVAL_SETS else np.array([])
TEST_PID = np.array([figshare_pid(p) for p in paths_of("test")]) if len(TEST_Y) else np.array([])
UPID = np.unique(TEST_PID)
PID_IDX = {u: np.where(TEST_PID == u)[0] for u in UPID}

def patient_vote(probs, pid=TEST_PID, y=TEST_Y):
    pids = np.unique(pid); py, pp, pm = [], [], []
    for u in pids:
        idx = np.where(pid == u)[0]
        h_ = hard(probs[idx]); mp = probs[idx].mean(0)
        if (h_ >= 0).any():
            votes = np.bincount(h_[h_ >= 0], minlength=N_CLASSES)
            tied = np.flatnonzero(votes == votes.max()); vote = int(tied[np.argmax(mp[tied])])
        else:
            vote = -1                                   # every slice of this patient abstained
        py.append(y[idx][0]); pp.append(vote); pm.append(mp)
    return np.array(py), np.array(pp), np.array(pm)

def cluster_boot(fn, n=N_BOOT, seed=SPLIT_SEED):
    rng = np.random.default_rng(seed); vals = []
    for _ in range(n):
        idx = np.concatenate([PID_IDX[u] for u in rng.choice(UPID, len(UPID), replace=True)])
        vals.append(fn(idx))
    return np.nanpercentile(vals, [2.5, 97.5])

TEST_PROBS = {}
test_rows = []
if RUN_EVALUATION and len(TEST_Y):
    for name in READY:
        P = ensemble_probs(name, "test"); TEST_PROBS[name] = P
        pred = hard(P)
        py, pp, pm = patient_vote(P)
        acc_ci = cluster_boot(lambda i: (pred[i] == TEST_Y[i]).mean())
        f1_ci = cluster_boot(lambda i: macro_f1(TEST_Y[i], pred[i]))
        per_seed = [macro_f1(TEST_Y, hard(ensemble_probs(name, "test", seeds=[s]))) for s in READY[name]]
        f1_all = macro_f1(TEST_Y, hard(ensemble_probs(name, "test", seeds="all")))
        test_rows.append({
            "Model": name, "Seeds": len(READY[name]),
            "Image acc (%)": f"{100*(pred==TEST_Y).mean():.2f} [{100*acc_ci[0]:.1f}, {100*acc_ci[1]:.1f}]",
            "Image macro-F1 (%)": f"{100*macro_f1(TEST_Y, pred):.2f} [{100*f1_ci[0]:.1f}, {100*f1_ci[1]:.1f}]",
            "Image macro-AUC": round(macro_auc(TEST_Y, P), 4),
            "Macro-F1 per seed (%)": "; ".join(f"{100*v:.2f}" for v in per_seed),
            "Macro-F1 all-seed ensemble (%)": round(100 * f1_all, 2) if len(per_seed) > 1 else "",
            "Patients correct": f"{int((pp==py).sum())}/{len(py)}",
            "Patient macro-F1 (%)": round(100 * macro_f1(py, pp), 2),
            "Patient macro-AUC": round(macro_auc(py, pm), 4)})
    df_test = pd.DataFrame(test_rows)
    print(df_test.to_string(index=False))
    df_test.to_csv(OUT_DIR / "tables" / "table_test.csv", index=False)
    print("\n[ ] = 95% CI from 2,000 patient-level (cluster) bootstrap resamples. All main columns use seed 0 only "
          "(same ensemble size for every model); extra seeds are shown separately.")

In [ ]:
# ─── 6.2 Paired comparisons vs MTK (patient-clustered bootstrap + sign-flip permutation, Holm) ───
def paired_compare(a, b, metric):
    pa, pb = hard(TEST_PROBS[a]), hard(TEST_PROBS[b])
    f = (lambda y, p: (p == y).mean()) if metric == "acc" else macro_f1
    obs = f(TEST_Y, pa) - f(TEST_Y, pb)
    ci = cluster_boot(lambda i: f(TEST_Y[i], pa[i]) - f(TEST_Y[i], pb[i]))
    d = np.array([(pa[PID_IDX[u]] == TEST_Y[PID_IDX[u]]).mean() - (pb[PID_IDX[u]] == TEST_Y[PID_IDX[u]]).mean()
                  for u in UPID])
    rng = np.random.default_rng(0)
    perm = np.array([(d * rng.choice([-1, 1], len(d))).mean() for _ in range(10000)])
    return obs, ci, float((np.abs(perm) >= abs(d.mean()) - 1e-12).mean())

stat_rows = []
if RUN_EVALUATION and PROPOSED in TEST_PROBS:
    for other in TEST_PROBS:
        if other == PROPOSED:
            continue
        for metric in ["acc", "f1"]:
            obs, ci, p = paired_compare(PROPOSED, other, metric)
            stat_rows.append({"Comparison": f"{PROPOSED} − {other}", "Metric": metric,
                              "Δ (pp)": round(100 * obs, 2), "95% CI": f"[{100*ci[0]:.1f}, {100*ci[1]:.1f}]", "p": p})
    df_stats = pd.DataFrame(stat_rows)
    if len(df_stats):
        for metric in ["acc", "f1"]:
            m = df_stats.Metric == metric
            p = df_stats.loc[m, "p"].values; order = np.argsort(p); adj = np.empty_like(p)
            adj[order] = np.minimum(1, np.maximum.accumulate((len(p) - np.arange(len(p))) * p[order]))
            df_stats.loc[m, "p (Holm)"] = adj
        print(df_stats.to_string(index=False))
        df_stats.to_csv(OUT_DIR / "tables" / "table_test_paired.csv", index=False)

In [ ]:
# ─── 6.3 Class-wise table + figure for the proposed model (image and patient level) ───
if RUN_EVALUATION and PROPOSED in TEST_PROBS:
    P = TEST_PROBS[PROPOSED]; pred = hard(P); py, pp, pm = patient_vote(P)
    rows = []
    for level, yy, pr, pb in [("Image", TEST_Y, pred, P), ("Patient", py, pp, pm)]:
        for k, c in enumerate(CLASSES):
            tp = ((pr == k) & (yy == k)).sum()
            rows.append({"Level": level, "Class": c, "n": int((yy == k).sum()),
                         "Precision (%)": round(100 * tp / max((pr == k).sum(), 1), 1),
                         "Recall (%)": round(100 * tp / max((yy == k).sum(), 1), 1),
                         "AUC": round(roc_auc_score(yy == k, pb[:, k]), 4)})
    df_cls = pd.DataFrame(rows); print(df_cls.to_string(index=False))
    df_cls.to_csv(OUT_DIR / "tables" / "table_test_classwise.csv", index=False)

    fig, ax = plt.subplots(2, 2, figsize=(7, 6.5))
    for c, (lvl, yy, pr, pb) in enumerate([("Image", TEST_Y, pred, P), ("Patient", py, pp, pm)]):
        cm = confusion_matrix(yy, pr, labels=range(N_CLASSES))
        ax[0, c].imshow(cm / cm.sum(1, keepdims=True), cmap="Blues", vmin=0, vmax=1)
        for i in range(N_CLASSES):
            for j in range(N_CLASSES):
                ax[0, c].text(j, i, cm[i, j], ha="center", va="center", fontsize=8)
        ax[0, c].set(xticks=range(3), yticks=range(3), xticklabels=CLASSES, yticklabels=CLASSES,
                     title=f"{lvl} level (n={len(yy)})", xlabel="Predicted", ylabel="True")
        for k, cl in enumerate(CLASSES):
            fpr, tpr, _ = roc_curve(yy == k, pb[:, k])
            ax[1, c].plot(fpr, tpr, color=CLASS_COLORS[cl], label=f"{cl} ({roc_auc_score(yy == k, pb[:, k]):.3f})")
        ax[1, c].plot([0, 1], [0, 1], ":", color="grey"); ax[1, c].legend(frameon=False, fontsize=7)
        ax[1, c].set(xlabel="False positive rate", ylabel="True positive rate")
    fig.suptitle(f"{PROPOSED} on the internal test set", fontsize=9)
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig_test_proposed.png"); plt.show()

In [ ]:
# ─── 6.3b Confusion matrices + ROC curves of the main models (internal test, image and patient level) ───
CMP_MODELS = list(dict.fromkeys(n for n in [PROPOSED, "CNX-Plain", "CTR-Net", "MTL-Seg", "MTK"] if n in TEST_PROBS))[:5]
CMP_MODELS = list(dict.fromkeys(CMP_MODELS))
if RUN_EVALUATION and CMP_MODELS:
    fig, ax = plt.subplots(2, len(CMP_MODELS), figsize=(2.6 * len(CMP_MODELS), 5.2), squeeze=False)
    for c, n in enumerate(CMP_MODELS):
        P = TEST_PROBS[n]; py, pp, _ = patient_vote(P)
        for r, (yy, pr, lvl) in enumerate([(TEST_Y, hard(P), "image"), (py, pp, "patient")]):
            cm = confusion_matrix(yy, pr, labels=range(N_CLASSES))
            ax[r, c].imshow(cm / np.maximum(cm.sum(1, keepdims=True), 1), cmap="Blues", vmin=0, vmax=1)
            for i in range(N_CLASSES):
                for j in range(N_CLASSES):
                    ax[r, c].text(j, i, cm[i, j], ha="center", va="center", fontsize=8,
                                  color="white" if cm[i, j] > cm.sum(1)[i] * 0.6 else "black")
            ax[r, c].set(xticks=range(3), yticks=range(3), xticklabels=[k[:4] for k in CLASSES],
                         yticklabels=[k[:4] for k in CLASSES] if c == 0 else [],
                         title=f"{n}\n{lvl} level (F1 {100 * macro_f1(yy, pr):.1f}%)")
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig_confusion_models.png"); plt.show()

    fig, ax = plt.subplots(1, N_CLASSES, figsize=(3.2 * N_CLASSES, 3), sharey=True)
    for k, cl in enumerate(CLASSES):
        for n in CMP_MODELS:
            P = TEST_PROBS[n]; fpr, tpr, _ = roc_curve(TEST_Y == k, P[:, k])
            ax[k].plot(fpr, tpr, lw=1.6 if n == PROPOSED else 1, label=f"{n} ({roc_auc_score(TEST_Y == k, P[:, k]):.3f})")
        ax[k].plot([0, 1], [0, 1], ":", color="grey")
        ax[k].set(title=f"{cl} vs rest (internal test)", xlabel="False positive rate"); ax[k].legend(frameon=False, fontsize=6)
    ax[0].set_ylabel("True positive rate"); fig.tight_layout(); fig.savefig(FIG_DIR / "fig_roc_test.png"); plt.show()

In [ ]:
# ─── 6.4 Ablation evidence: gate weights and prediction agreement ───
if RUN_EVALUATION and len(TEST_Y):
    agree_rows = []
    if PROPOSED in TEST_PROBS:
        _, pp_ref, _ = patient_vote(TEST_PROBS[PROPOSED])
        for other in (["ROI-Mask", "CTR-Net", "CNX-Plain", "MTL-Seg", "RRR", "CAM-Guide"] if PROPOSED != "MTK" else ABLATIONS):
            if other in TEST_PROBS:
                _, pp_o, _ = patient_vote(TEST_PROBS[other])
                agree_rows.append({"Variant": other,
                                   "Slices with different label": int((hard(TEST_PROBS[PROPOSED]) != hard(TEST_PROBS[other])).sum()),
                                   "Patients with different label": int((pp_ref != pp_o).sum())})
        if agree_rows:
            print(pd.DataFrame(agree_rows).to_string(index=False))
            pd.DataFrame(agree_rows).to_csv(OUT_DIR / "tables" / "table_ablation_agreement.csv", index=False)

    gate_rows = []
    for name in [n for n in READY if MODEL_SPECS[n]["kind"] in ("mtk", "tamtk")]:
        probe = build_model(name, pretrained=False)
        if len(probe.paths) < 2:
            continue
        gates = []
        print(f"  gate weights: {name} ({len(READY[name][:1]) * N_FOLDS} fold models)", flush=True)
        for s in READY[name][:1]:
            for f in range(1, N_FOLDS + 1):
                m = load_checkpoint(name, s, f); mean, std = norm_for(name, f); arrs = [load_u8(p) for p in paths_of("test")]
                with torch.no_grad():
                    gates.append(np.concatenate([m(to_batch(arrs[i:i + 128], mean, std), return_gate=True)[1].float().cpu().numpy()
                                                 for i in range(0, len(arrs), 128)]))
                del m
        g = np.mean(gates, 0)
        for k, c in enumerate(CLASSES):
            gate_rows.append({"Model": name, "Class": c, **{f"w_{p}": round(float(g[TEST_Y == k, i].mean()), 3)
                                                             for i, p in enumerate(probe.paths)}})
    if gate_rows:
        print(pd.DataFrame(gate_rows).to_string(index=False))
        pd.DataFrame(gate_rows).to_csv(OUT_DIR / "tables" / "table_gate_weights.csv", index=False)

In [ ]:
# ─── 6.5 Ablation summary: table + figure (CV folds, internal test, external) ───
ABL_SHOW = [n for n in ["ROI-Loc", "ROI-m06-Transplant", "ROI-Crop", "ROI-Crop-m06", "ROI-Crop-m15", "ROI-Mask", "CNX-Transplant", "CNX-Strip", "CTR-Net", "CTR-noCF", "CTR-noSeg", "CTR-MambaKAN", "MTL-Seg", "RRR", "CAM-Guide",
                         "CNX-Plain", "ConvNeXt-Tiny", "MTK", "MTK-noSeq", "MTK-noKAN", "MTK-noCNN", "CNN-Backbone"]
            if n in READY]
if RUN_EVALUATION and len(ABL_SHOW) >= 2:
    params = dict(zip(df_complexity.Model, df_complexity["Params (M)"]))
    rows = []
    for n in ABL_SHOW:
        f1s = [r["val_f1"] for r in run_records(n)]
        row = {"Variant": n, "Params (M)": params.get(n), "CV macro-F1 (%)": round(100 * np.mean(f1s), 2),
               "CV SD": round(100 * np.std(f1s, ddof=1), 2),
               "Test macro-F1 (%)": round(100 * macro_f1(TEST_Y, hard(TEST_PROBS[n])), 2)}
        if "mendeley" in EVAL_SETS:
            keep = np.isin(paths_of("mendeley"), [p for p, _ in MEND_CLEAN])
            row["Mendeley dedup macro-F1 (%)"] = round(100 * macro_f1(labels_of("mendeley")[keep],
                                                                      hard(ensemble_probs(n, "mendeley")[keep])), 2)
        if "brats" in EVAL_SETS:
            yb_ = labels_of("brats"); Pb = ensemble_probs(n, "brats")
            row["BraTS glioma-vs-meningioma AUC"] = round(roc_auc_score(yb_ == 0, Pb[:, 0] / (Pb[:, 0] + Pb[:, 1] + 1e-12)), 4)
        if "bdneuro" in EVAL_SETS and BD_CLEAN:
            keep = np.isin(paths_of("bdneuro"), [p for p, _ in BD_CLEAN])
            row["BDNeuro bal. acc (%)"] = round(100 * bal_acc(labels_of("bdneuro")[keep],
                                                              hard(ensemble_probs(n, "bdneuro")[keep])), 2)
        row["Tumour reliance"] = RELIANCE.get(n, np.nan)
        rows.append(row)
    df_abl = pd.DataFrame(rows)
    print("What each part contributes (seed 0 everywhere; higher is better in every column):")
    print(df_abl.to_string(index=False))
    df_abl.to_csv(OUT_DIR / "tables" / "table_ablation.csv", index=False)

    metrics = [("Test macro-F1 (%)", "Internal test macro-F1 (%)", 1),
               ("BraTS glioma-vs-meningioma AUC", "BraTS AUC (new hospital)", 1),
               ("BDNeuro bal. acc (%)", "BDNeuro balanced acc. (%)", 1),
               ("Tumour reliance", "Tumour reliance (0-1)", 1)]
    metrics = [m for m in metrics if m[0] in df_abl and df_abl[m[0]].notna().any()]
    show = df_abl.set_index("Variant")
    fig, ax = plt.subplots(1, len(metrics), figsize=(3.2 * len(metrics), 0.32 * len(show) + 1.2), sharey=True, squeeze=False)
    for a, (col, title, _) in zip(ax[0], metrics):
        vals = show[col].astype(float)
        a.barh(range(len(show)), vals, color=[FAMILY_COLOR[family_of(n)] for n in show.index])
        lo = np.nanmin(vals) - 0.15 * (np.nanmax(vals) - np.nanmin(vals) + 1e-6)
        a.set(xlim=(max(0, lo), np.nanmax(vals) * 1.03), title=title)
        for i, v in enumerate(vals):
            if np.isfinite(v):
                a.text(v, i, f" {v:.2f}" if v < 2 else f" {v:.1f}", va="center", fontsize=6)
    ax[0][0].set_yticks(range(len(show))); ax[0][0].set_yticklabels(show.index, fontsize=7); ax[0][0].invert_yaxis()
    handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in FAMILY_COLOR.values()]
    fig.legend(handles, ["proposed", "CTR ablations", "existing guidance methods", "pretrained baselines", "scratch models"],
               frameon=False, fontsize=7, ncol=5, loc="lower center", bbox_to_anchor=(0.5, -0.06))
    fig.suptitle("Ablation and comparison (seed 0)", fontsize=9)
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig_ablation.png", bbox_inches="tight"); plt.show()

In [ ]:
# ─── 6.6 Error analysis: which test patients are misclassified, and by which models ───
if RUN_EVALUATION and TEST_PROBS:
    err = {}
    for n in TEST_PROBS:
        py, pp, pm = patient_vote(TEST_PROBS[n])
        err[n] = dict(zip(np.unique(TEST_PID), pp))
    true_p = dict(zip(np.unique(TEST_PID), patient_vote(next(iter(TEST_PROBS.values())))[0]))
    rows = []
    for u in np.unique(TEST_PID):
        wrong = [n for n in err if err[n][u] != true_p[u]]
        if wrong:
            idx = PID_IDX[u]
            rows.append({"Patient": u, "True": IDX2CLASS[true_p[u]], "Slices": len(idx),
                         "Wrong in models": f"{len(wrong)}/{len(err)}",
                         f"{PROPOSED} predicts": IDX2CLASS[err[PROPOSED][u]] if PROPOSED in err else "-",
                         "Models wrong": ", ".join(wrong)})
    df_err = pd.DataFrame(rows).sort_values("Wrong in models", ascending=False) if rows else pd.DataFrame()
    print("Misclassified test patients (patient-level vote):" if rows else "No misclassified test patients.")
    if rows:
        print(df_err.to_string(index=False))
        df_err.to_csv(OUT_DIR / "tables" / "table_error_patients.csv", index=False)
    if PROPOSED in TEST_PROBS:
        cm = confusion_matrix(TEST_Y, hard(TEST_PROBS[PROPOSED]), labels=range(N_CLASSES))
        pairs = [(IDX2CLASS[i], IDX2CLASS[j], int(cm[i, j])) for i in range(3) for j in range(3) if i != j and cm[i, j]]
        print(f"\n{PROPOSED} image-level confusions (true -> predicted):",
              sorted(pairs, key=lambda t: -t[2]))

In [ ]:
# ─── 6.7 Does the tumour-mask head find the tumour? (Dice, internal test + BraTS) ───
def seg_probs(name, items, seed=None):
    """Mean predicted tumour probability map (224x224) over the 5 fold models."""
    seed = READY[name][0] if seed is None else seed
    out = np.zeros((len(items), IMG_SIZE, IMG_SIZE), np.float32)
    for f in range(1, N_FOLDS + 1):
        m = load_checkpoint(name, seed, f); mean, std = norm_for(name, f)
        arrs = [load_u8(p) for p, _ in items]
        with torch.no_grad():
            for i in range(0, len(arrs), 64):
                with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
                    _, sl = m(to_batch(arrs[i:i + 64], mean, std), return_seg=True)
                pr = torch.sigmoid(F.interpolate(sl.float(), (IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False))
                out[i:i + 64] += pr[:, 0].cpu().numpy() / N_FOLDS
        del m; gc.collect()
    return out

def dice(a, b):
    return float(2 * (a & b).sum() / (a.sum() + b.sum() + 1e-8))

SEG_MAPS = {}
seg_rows = []
SEG_MODEL = (ROI_SEG_SOURCE if is_roi(PROPOSED) and ROI_SEG_SOURCE in READY
             else PROPOSED if MODEL_SPECS[PROPOSED].get("seg") else None)
if RUN_EVALUATION and SEG_MODEL and MASKS:
    banner(f"6.7 Does {SEG_MODEL} find the tumour by itself? (Dice of its predicted tumour map)",
           f"this tumour map is what {PROPOSED} uses to decide where to look" if is_roi(PROPOSED) else "~2-4 min")
    sets = {"Internal test": [(p, y) for p, y in TEST_ITEMS if p in MASKS]}
    if len(BRATS_META):
        sets["BraTS (tumour core)"] = list(zip(BRATS_META.path, BRATS_META.y))
    for set_name, items in sets.items():
        maps = seg_probs(SEG_MODEL, items); SEG_MAPS[set_name] = (items, maps)
        for (p, y), mp in zip(items, maps):
            gt = MASKS[p] if set_name == "Internal test" else load_mask(str(BRATS_DIR / Path(p).parent.parent.name
                                                                          / "masks_tc" / Path(p).name))
            if gt.sum() == 0:
                continue
            yx = np.unravel_index(mp.argmax(), mp.shape)
            seg_rows.append({"Set": set_name, "Class": IDX2CLASS[y], "Dice": dice(mp >= 0.5, gt),
                             "Peak inside tumour": float(gt[yx]), "Tumour area (%)": 100 * float(gt.mean())})
    df_seg = pd.DataFrame(seg_rows)
    tab = df_seg.groupby(["Set", "Class"]).agg(n=("Dice", "size"), Dice=("Dice", "mean"),
                                               Dice_median=("Dice", "median"),
                                               Peak_inside=("Peak inside tumour", "mean")).round(3)
    print(tab.to_string()); tab.to_csv(OUT_DIR / "tables" / "table_segmentation_head.csv")
    print("Note: the mask head is an auxiliary output (28x28), trained only on Figshare; BraTS is a different "
          "scanner/protocol and is skull-stripped.")

In [ ]:
# ─── 6.8 Accuracy vs model size ───
if RUN_EVALUATION and TEST_PROBS:
    params = dict(zip(df_complexity.Model, df_complexity["Params (M)"]))
    fig, ax = plt.subplots(1, 2 if "brats" in EVAL_SETS else 1, figsize=(9, 3.4), squeeze=False)
    for n in TEST_PROBS:
        col = "#d1495b" if n == PROPOSED else ("#30638e" if is_pretrained(n) else "#888888")
        f1 = 100 * macro_f1(TEST_Y, hard(TEST_PROBS[n]))
        ax[0, 0].scatter(params[n], f1, color=col, s=40); ax[0, 0].annotate(n, (params[n], f1), fontsize=6, xytext=(3, 2), textcoords="offset points")
        if "brats" in EVAL_SETS:
            yb_ = labels_of("brats"); Pb = ensemble_probs(n, "brats")
            auc = roc_auc_score(yb_ == 0, Pb[:, 0] / (Pb[:, 0] + Pb[:, 1] + 1e-12))
            ax[0, 1].scatter(params[n], auc, color=col, s=40); ax[0, 1].annotate(n, (params[n], auc), fontsize=6, xytext=(3, 2), textcoords="offset points")
    ax[0, 0].set(xscale="log", xlabel="Parameters (M, log)", ylabel="Internal test macro-F1 (%)", title="Accuracy vs size")
    if "brats" in EVAL_SETS:
        ax[0, 1].set(xscale="log", xlabel="Parameters (M, log)", ylabel="AUC", title="External BraTS glioma-vs-meningioma")
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig_accuracy_vs_size.png"); plt.show()

In [ ]:
# ─── 6.9 Is the label given away by the slice PLANE or by the tumour LOCATION? ───
# location-only = only the expert tumour outline's position/shape (no image). Tumour location is a REAL clinical
# cue (pituitary tumours sit at the sella, meningiomas touch the brain surface), so a high score here is legitimate.
# plane-only    = only the viewing plane of the slice (axial/coronal/sagittal). This says nothing about the tumour;
# a high score means the dataset was assembled in a way that leaks the label.
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_predict

def location_features(p):
    mk = MASKS[p]; img = load_u8(p); bm = brain_mask(img)
    ys, xs = np.where(mk); bys, bxs = np.where(bm) if bm.any() else (ys, xs)
    h = max(bys.max() - bys.min(), 1); w = max(bxs.max() - bxs.min(), 1)
    cy, cx = (ys.mean() - bys.min()) / h, (xs.mean() - bxs.min()) / w
    edge = cv2.distanceTransform(bm.astype(np.uint8), cv2.DIST_L2, 3)
    return [cy, cx, abs(cx - 0.5), mk.mean() / max(bm.mean(), 1e-3), float(edge[mk].min()) / max(h, w),
            (ys.max() - ys.min() + 1) / h, (xs.max() - xs.min() + 1) / w]

if RUN_EVALUATION and MASKS:
    d = df3[df3.filepath.isin(MASKS)].reset_index(drop=True)
    Xl = np.array([location_features(p) for p in d.filepath]); yl = d.label.values
    prl = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)), Xl, yl,
                            groups=d.pid.values, cv=GroupKFold(5))
    rows = [{"Information used (no deep model)": "Tumour location + size only (from expert outline)",
             "Balanced acc (%)": round(100 * bal_acc(yl, prl), 1), "Legitimate clinical cue?": "yes"}]
    if PLANE_OF:
        pl = d.filepath.map(PLANE_OF).values; okp = pl != "mixed"
        tab = pd.crosstab(pd.Series(pl[okp], name="plane"), pd.Series(np.array(CLASSES)[yl[okp]], name="class"))
        print("How many slices of each tumour type come from each viewing plane:")
        print(tab.to_string()); tab.to_csv(OUT_DIR / "tables" / "table_plane_by_class.csv")
        Xp = pd.get_dummies(pd.Series(pl)).values.astype(float)
        prp = cross_val_predict(LogisticRegression(max_iter=5000), Xp[okp], yl[okp], groups=d.pid.values[okp], cv=GroupKFold(5))
        rows.append({"Information used (no deep model)": "Viewing plane only (axial / coronal / sagittal)",
                     "Balanced acc (%)": round(100 * bal_acc(yl[okp], prp), 1), "Legitimate clinical cue?": "no"})
        if RELIANCE:
            REL_PLANE = np.array([PLANE_OF.get(p, "mixed") for p in REL_DF.filepath])
            pr_rows = []
            for n in [m for m in [PROPOSED, "ROI-m06-Transplant", "ROI-Crop", "CNX-Transplant", "CNX-Plain", "CTR-Net", "MTL-Seg", "MTK"] if m in REL]:
                for c in ["original", "hidden (swap)"]:
                    pred = hard(REL[n][c])
                    pr_rows.append({"Model": n, "Version": c, **{pl_: round(100 * bal_acc(REL_Y[REL_PLANE == pl_], pred[REL_PLANE == pl_]), 1)
                                                                 for pl_ in ["axial", "coronal", "sagittal"] if (REL_PLANE == pl_).any()}})
            df_pp = pd.DataFrame(pr_rows)
            print("\nBalanced accuracy (%) per viewing plane, with the tumour visible and hidden:")
            print(df_pp.to_string(index=False)); df_pp.to_csv(OUT_DIR / "tables" / "table_accuracy_by_plane.csv", index=False)
        fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
        tabp = (tab.T / tab.sum(1)).T * 100
        left = np.zeros(len(tabp))
        for c in CLASSES:
            if c in tabp:
                ax[0].barh(tabp.index, tabp[c], left=left, color=CLASS_COLORS[c], label=c); left += tabp[c].values
        ax[0].set(xlabel="% of slices of this plane", title="Tumour type per viewing plane (Figshare)")
        ax[0].legend(frameon=False, fontsize=7, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.22))
        if RELIANCE and len(df_pp):
            dh = df_pp[df_pp.Version == "hidden (swap)"].set_index("Model")[[c for c in ["axial", "coronal", "sagittal"] if c in df_pp]]
            dh.T.plot.bar(ax=ax[1], rot=0, width=0.8, color=[FAMILY_COLOR[family_of(n)] for n in dh.index])
            ax[1].axhline(100 / 3, color="k", ls="--", lw=1)
            ax[1].set(ylabel="Balanced accuracy with tumour HIDDEN (%)", ylim=(0, 100), xlabel="",
                      title="Without the tumour, which planes still give the answer?")
            ax[1].legend(frameon=False, fontsize=6)
        fig.tight_layout(); fig.savefig(FIG_DIR / "fig_plane_analysis.png", bbox_inches="tight"); plt.show()
    rows.append({"Information used (no deep model)": "Brain outline / position only (Figshare, quick check)",
                 "Balanced acc (%)": 67.0, "Legitimate clinical cue?": "no"})
    df_loc = pd.DataFrame(rows); print("\nHow much do simple cues alone reveal the tumour type? (guessing = 33.3%)")
    print(df_loc.to_string(index=False)); df_loc.to_csv(OUT_DIR / "tables" / "table_location_plane_baselines.csv", index=False)
    if not PLANE_OF:
        print("(plane rows appear after PLANE_MAP is filled in cell 0.2)")

## 7. External evaluation (no fine-tuning)
BRISC and Mendeley: 3 tumour classes, on **all** images and on **deduplicated** images (no near-duplicate of Figshare;
for Mendeley also none of BRISC). No patient IDs → image level only.
BraTS-GLI / BraTS-MEN: one tumour type each → class recall (image and patient level) and glioma-vs-meningioma AUC.

In [ ]:
# ─── 7.1 BRISC + Mendeley ───
ext_rows = []
if RUN_EVALUATION:
    for set_name, clean in [("brisc", BRISC_CLEAN), ("mendeley", MEND_CLEAN)]:
        if set_name not in EVAL_SETS:
            continue
        y_all = labels_of(set_name)
        keep = np.isin(paths_of(set_name), [p for p, _ in clean])
        for name in READY:
            P = ensemble_probs(name, set_name)
            for subset, mask in [("all", np.ones(len(y_all), bool)), ("deduplicated", keep)]:
                y, pr = y_all[mask], P[mask]
                rec = per_class_recall(y, hard(pr))
                present = sorted(np.unique(y))           # deduplicated BRISC has NO glioma left
                ext_rows.append({"Dataset": set_name, "Subset": subset, "n": int(mask.sum()), "Model": name,
                                 "n per class": "/".join(str(int((y == k).sum())) for k in range(N_CLASSES)),
                                 "Acc (%)": round(100 * (hard(pr) == y).mean(), 2),
                                 "Macro-F1 (%)": round(100 * macro_f1(y, hard(pr), labels=present), 2),
                                 "Macro-AUC": round(macro_auc(y, pr), 4),
                                 **{f"Recall {c} (%)": round(100 * v, 1) for c, v in rec.items()}})
    df_ext = pd.DataFrame(ext_rows)
    if len(df_ext):
        print(df_ext.to_string(index=False))
        df_ext.to_csv(OUT_DIR / "tables" / "table_external.csv", index=False)
        print("Macro-F1 / AUC use only the classes present in each subset (BRISC dedup: meningioma + pituitary only, "
              "because every BRISC glioma is a Figshare duplicate).")

In [ ]:
# ─── 7.2 BraTS-GLI / BraTS-MEN ───
def brats_cluster_ci(fn, pids, n=1000, seed=SPLIT_SEED):
    """95% CI by resampling BraTS PATIENTS (all slices of a patient together)."""
    rng = np.random.default_rng(seed); up = np.unique(pids); groups = {u: np.where(pids == u)[0] for u in up}
    vals = []
    for _ in range(n):
        idx = np.concatenate([groups[u] for u in rng.choice(up, len(up), replace=True)])
        try:
            vals.append(fn(idx))
        except ValueError:
            pass
    return np.percentile(vals, [2.5, 97.5])

def brats_auc_scores(name):
    yb = labels_of("brats"); gm = np.isin(yb, [0, 1]); P = ensemble_probs(name, "brats")
    return yb[gm] == 0, P[gm, 0] / (P[gm, 0] + P[gm, 1] + 1e-12), BRATS_META["patient_id"].values[gm]

brats_rows = []
if RUN_EVALUATION and "brats" in EVAL_SETS:
    yb = labels_of("brats"); pid_b = BRATS_META["patient_id"].values; ds = BRATS_META["dataset"].values
    for name in READY:
        P = ensemble_probs(name, "brats"); pred = hard(P)
        for d, cls in [("gli", "glioma"), ("men", "meningioma")]:
            m = ds == d
            if not m.any():
                continue
            k = CLASS2IDX[cls]
            pids = np.unique(pid_b[m])
            def _vote(v):
                v = v[v >= 0]
                return int(np.bincount(v, minlength=N_CLASSES).argmax()) if len(v) else -1
            pat_pred = [_vote(pred[m & (pid_b == u)]) for u in pids]
            dist = np.bincount(pred[m][pred[m] >= 0], minlength=N_CLASSES) / m.sum()
            brats_rows.append({"Model": name, "Subset": f"BraTS-{d.upper()}", "Slices": int(m.sum()), "Patients": len(pids),
                               f"Image recall (%)": round(100 * (pred[m] == k).mean(), 1),
                               f"Patient recall (%)": round(100 * np.mean(np.array(pat_pred) == k), 1),
                               **{f"→{c} (%)": round(100 * dist[j], 1) for j, c in enumerate(CLASSES)},
                               "→abstained (%)": round(100 * (pred[m] < 0).mean(), 1)})
        gm = np.isin(yb, [0, 1])
        score = P[gm, 0] / (P[gm, 0] + P[gm, 1] + 1e-12)
        auc_ci = brats_cluster_ci(lambda i: roc_auc_score(yb[gm][i] == 0, score[i]), pid_b[gm])
        brats_rows.append({"Model": name, "Subset": "GLI + MEN", "Slices": int(gm.sum()),
                           "Glioma-vs-meningioma AUC": round(roc_auc_score(yb[gm] == 0, score), 4),
                           "AUC 95% CI": f"[{auc_ci[0]:.3f}, {auc_ci[1]:.3f}]"})
    df_brats = pd.DataFrame(brats_rows)
    print("BraTS (new hospital, never used for training). Recall = share of slices given the correct tumour type;")
    print("'→ class' = where the slices went; AUC = glioma vs meningioma ranking (CI by patient resampling).")
    print(df_brats.to_string(index=False))
    df_brats.to_csv(OUT_DIR / "tables" / "table_brats.csv", index=False)
    print("Note: BraTS images are skull-stripped; Figshare images are not (acquisition/preprocessing shift).")

In [ ]:
# ─── 7.2b Is the proposed model better on BraTS? (paired, patient-level bootstrap, Holm-corrected) ───
def brats_cluster_boot(fn, pids, n=1000, seed=SPLIT_SEED):
    rng = np.random.default_rng(seed); up = np.unique(pids); groups = {u: np.where(pids == u)[0] for u in up}
    vals = []
    for _ in range(n):
        try:
            vals.append(fn(np.concatenate([groups[u] for u in rng.choice(up, len(up), replace=True)])))
        except ValueError:
            pass
    return np.array(vals)

def holm(pvals):
    pv = np.asarray(pvals, float); o = np.argsort(pv); m_ = len(pv)
    adj = np.maximum.accumulate([min(1.0, (m_ - i) * pv[j]) for i, j in enumerate(o)]) if m_ else np.array([])
    out = np.empty(m_); out[o] = adj; return out

def brats_paired(ref, others):
    yt, s_ref, pids = brats_auc_scores(ref); rows = []
    for other in others:
        _, s_o, _ = brats_auc_scores(other)
        d = roc_auc_score(yt, s_ref) - roc_auc_score(yt, s_o)
        v = brats_cluster_boot(lambda i: roc_auc_score(yt[i], s_ref[i]) - roc_auc_score(yt[i], s_o[i]), pids)
        rows.append({"Comparison": f"{ref} − {other}", "ΔAUC": round(d, 4),
                     "95% CI": f"[{np.percentile(v, 2.5):+.3f}, {np.percentile(v, 97.5):+.3f}]",
                     "p (bootstrap)": round(float(min(1, 2 * min((v <= 0).mean(), (v >= 0).mean()))), 4)})
    df_ = pd.DataFrame(rows)
    if len(df_):
        df_["p (Holm)"] = np.round(holm(df_["p (bootstrap)"].values), 4)
        df_["Significant (Holm)"] = np.where(df_["p (Holm)"] < 0.05, "yes", "no")
    return df_.sort_values("ΔAUC") if len(df_) else df_

if RUN_EVALUATION and "brats" in EVAL_SETS and PROPOSED in READY:
    df_bd_cmp = brats_paired(PROPOSED, [n for n in READY if n != PROPOSED])
    print(f"BraTS glioma-vs-meningioma AUC: {PROPOSED} minus each other model (positive = {PROPOSED} better).")
    print("CI and p from 1,000 resamples of BraTS PATIENTS; Holm-corrected over all comparisons.")
    print(df_bd_cmp.to_string(index=False))
    df_bd_cmp.to_csv(OUT_DIR / "tables" / "table_brats_paired.csv", index=False)

In [ ]:
# ─── 7.2c BDNeuro-MRI (independent Bangladeshi hospital, deduplicated, image level) ───
bd_rows = []
if RUN_EVALUATION and "bdneuro" in EVAL_SETS and BD_CLEAN:
    keep = np.isin(paths_of("bdneuro"), [p for p, _ in BD_CLEAN]); yb_ = labels_of("bdneuro")[keep]
    rng_b = np.random.default_rng(SPLIT_SEED)
    boots = [rng_b.integers(0, len(yb_), len(yb_)) for _ in range(1000)]
    P_prop = ensemble_probs(PROPOSED, "bdneuro")[keep] if PROPOSED in READY else None
    for name in READY:
        P = ensemble_probs(name, "bdneuro")[keep]; pr = hard(P)
        ba = bal_acc(yb_, pr); ci = np.percentile([bal_acc(yb_[b], pr[b]) for b in boots], [2.5, 97.5])
        row = {"Model": name, "n": int(keep.sum()), "Balanced acc (%)": round(100 * ba, 2),
               "95% CI": f"[{100 * ci[0]:.1f}, {100 * ci[1]:.1f}]",
               "Macro-F1 (%)": round(100 * macro_f1(yb_, pr), 2), "Macro-AUC": round(macro_auc(yb_, P), 4),
               **{f"Recall {c} (%)": round(100 * v, 1) for c, v in per_class_recall(yb_, pr).items()}}
        if P_prop is not None and name != PROPOSED:
            pp = hard(P_prop)
            dci = np.percentile([bal_acc(yb_[b], pp[b]) - bal_acc(yb_[b], pr[b]) for b in boots], [2.5, 97.5])
            row[f"Δ vs {PROPOSED} (pts)"] = f"{100 * (bal_acc(yb_, pp) - ba):+.1f} [{100 * dci[0]:+.1f}, {100 * dci[1]:+.1f}]"
        bd_rows.append(row)
    df_bdn = pd.DataFrame(bd_rows).sort_values("Balanced acc (%)", ascending=False)
    print(f"BDNeuro-MRI, {int(keep.sum())} unique tumour images (no patient IDs -> image-level bootstrap CIs).")
    print(f"Δ = {PROPOSED} minus the model (positive = {PROPOSED} better), with 95% CI.")
    print(df_bdn.to_string(index=False))
    df_bdn.to_csv(OUT_DIR / "tables" / "table_bdneuro.csv", index=False)

In [ ]:
# ─── 7.2d KEY FIGURE: models that rely on the tumour generalise better ───
from scipy.stats import spearmanr
if RUN_EVALUATION and RELIANCE:
    gen = {}
    if "brats" in EVAL_SETS:
        gen["BraTS AUC (glioma vs meningioma)"] = {n: roc_auc_score(*brats_auc_scores(n)[:2]) for n in RELIANCE}
    if "bdneuro" in EVAL_SETS and BD_CLEAN:
        keep = np.isin(paths_of("bdneuro"), [p for p, _ in BD_CLEAN])
        gen["BDNeuro balanced accuracy (%)"] = {n: 100 * bal_acc(labels_of("bdneuro")[keep],
                                                                hard(ensemble_probs(n, "bdneuro")[keep])) for n in RELIANCE}
    if gen:
        fig, ax = plt.subplots(1, len(gen), figsize=(5 * len(gen), 3.8), squeeze=False)
        corr_rows = []
        for a, (lab, g) in zip(ax[0], gen.items()):
            names = [n for n in RELIANCE if np.isfinite(RELIANCE[n])]
            x = np.array([RELIANCE[n] for n in names]); y = np.array([g[n] for n in names])
            rho, pval = spearmanr(x, y)
            corr_rows.append({"External set": lab, "Spearman rho": round(rho, 3), "p": round(pval, 4), "models": len(names)})
            for n, xi, yi in zip(names, x, y):
                a.scatter(xi, yi, s=55 if n == PROPOSED else 30, color=FAMILY_COLOR[family_of(n)],
                          edgecolor="k" if n == PROPOSED else "none", zorder=3)
                a.annotate(n, (xi, yi), fontsize=6, xytext=(3, 3), textcoords="offset points")
            a.set(xlabel="Tumour-reliance score (tumour-hiding test)", ylabel=lab,
                  title=f"Spearman ρ = {rho:.2f} (p = {pval:.3f}, {len(names)} models)")
        fig.suptitle("Is tumour reliance related to generalisation? (each dot = one model; see the honest checks below)", fontsize=9)
        fig.tight_layout(); fig.savefig(FIG_DIR / "fig_reliance_vs_generalisation.png", bbox_inches="tight"); plt.show()
        df_corr = pd.DataFrame(corr_rows)
        # HONEST CHECK: is the correlation only "pretrained vs scratch"? -> within-family + partial correlation
        fam = {n: ("pretrained" if is_pretrained(n) else "scratch") for n in RELIANCE}
        extra = []
        for lab, g in gen.items():
            names = [n for n in RELIANCE if np.isfinite(RELIANCE[n])]
            if len(names) < 4:
                continue
            for f_ in ["pretrained", "scratch"]:
                nn_ = [n for n in names if fam[n] == f_]
                if len(nn_) >= 4:
                    r_, p_ = spearmanr([RELIANCE[n] for n in nn_], [g[n] for n in nn_])
                    extra.append({"External set": lab, "Models": f"only {f_} ({len(nn_)})", "Spearman rho": round(r_, 3), "p": round(p_, 4)})
            rx = pd.Series([RELIANCE[n] for n in names]).rank().values; ry = pd.Series([g[n] for n in names]).rank().values
            z = np.array([fam[n] == "pretrained" for n in names], float)
            Z = np.column_stack([np.ones_like(z), z])
            res_x = rx - Z @ np.linalg.lstsq(Z, rx, rcond=None)[0]; res_y = ry - Z @ np.linalg.lstsq(Z, ry, rcond=None)[0]
            r_, p_ = sps.pearsonr(res_x, res_y)
            extra.append({"External set": lab, "Models": f"all, controlling for pretrained vs scratch ({len(names)})",
                          "Spearman rho": round(r_, 3), "p": round(p_, 4)})
            # the strongest check: leave out the ROI / Run-5 models (designed to be tumour-reliant) and stay in ONE family
            for lab2, sub in [("pretrained WITHOUT ROI / Run-5 models", [n for n in names if fam[n] == "pretrained"
                                and not n.startswith("ROI") and n not in ("CNX-Transplant", "CNX-Strip")]),
                              ("ConvNeXt backbone only, WITHOUT ROI / Run-5 models", [n for n in names if MODEL_SPECS[n]["kind"] in ("guide", "tamtk")
                                and n not in ("CNX-Transplant", "CNX-Strip")] + (["ConvNeXt-Tiny"] if "ConvNeXt-Tiny" in names else []))]:
                sub = list(dict.fromkeys(sub))
                if len(sub) >= 4:
                    r_, p_ = spearmanr([RELIANCE[n] for n in sub], [g[n] for n in sub])
                    extra.append({"External set": lab, "Models": f"{lab2} ({len(sub)})", "Spearman rho": round(r_, 3), "p": round(p_, 4)})
        df_corr = pd.concat([df_corr.assign(Models="all"), pd.DataFrame(extra)], ignore_index=True)
        print(df_corr[["External set", "Models", "Spearman rho", "p"]].to_string(index=False))
        df_corr.to_csv(OUT_DIR / "tables" / "table_reliance_vs_generalisation.csv", index=False)
        print("\nHow to read: if the correlation disappears inside one backbone family without the ROI models, the overall"
              "\ncorrelation reflects model families, NOT a general law. Report it as descriptive only (Run 4: it disappeared).")

In [ ]:
# ─── 7.3 Figure: per-class recall in every dataset (proposed vs plain baseline vs original MTK) ───
if RUN_EVALUATION and TEST_PROBS:
    show = [n for n in [PROPOSED, "CNX-Plain", "MTK"] if n in READY]
    show = list(dict.fromkeys(show))
    sets = [("Internal test", lambda n: (TEST_Y, TEST_PROBS[n]))]
    if "bdneuro" in EVAL_SETS and BD_CLEAN:
        kb = np.isin(paths_of("bdneuro"), [p for p, _ in BD_CLEAN])
        sets.append(("BDNeuro (new hospital)", lambda n: (labels_of("bdneuro")[kb], ensemble_probs(n, "bdneuro")[kb])))
    if "brats" in EVAL_SETS:
        sets.append(("BraTS (new hospital)", lambda n: (labels_of("brats"), ensemble_probs(n, "brats"))))
    if "mendeley" in EVAL_SETS and MEND_CLEAN:
        km = np.isin(paths_of("mendeley"), [p for p, _ in MEND_CLEAN])
        sets.append(("Mendeley (unique)", lambda n: (labels_of("mendeley")[km], ensemble_probs(n, "mendeley")[km])))
    fig, ax = plt.subplots(1, len(sets), figsize=(3.4 * len(sets), 3.2), sharey=True, squeeze=False)
    w = 0.8 / len(show)
    for a, (title, get) in zip(ax[0], sets):
        for j, n in enumerate(show):
            y_, P_ = get(n); rec = per_class_recall(y_, hard(P_))
            vals = [100 * rec[c] if np.isfinite(rec[c]) else 0 for c in CLASSES]
            a.bar(np.arange(N_CLASSES) + (j - (len(show) - 1) / 2) * w, vals, w, label=n,
                  color=FAMILY_COLOR[family_of(n)] if n != "MTK" else "#888888")
        a.set(xticks=range(N_CLASSES), xticklabels=CLASSES, ylim=(0, 105), title=title)
    ax[0][0].set_ylabel("Recall (%)"); ax[0][0].legend(frameon=False, fontsize=7)
    fig.suptitle("Recall per tumour type in each dataset (BraTS has no pituitary class)", fontsize=9)
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig_external_recall.png", bbox_inches="tight"); plt.show()

In [ ]:
# ─── 7.4 Feature space (t-SNE): do images group by TUMOUR TYPE or by SOURCE DATASET? ───
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
TSNE_MODELS = [n for n in ["CNX-Plain", "CTR-Net", "MTK"] if n in READY]       # ROI models see crops, not slices
TSNE_MODELS = list(dict.fromkeys(TSNE_MODELS))
tsne_rows = []
if RUN_EVALUATION and TSNE_MODELS:
    rng_t = np.random.default_rng(0)
    def sub(items, n):
        idx = rng_t.choice(len(items), min(n, len(items)), replace=False); return [items[i] for i in idx]
    src_items = [("Figshare test", TEST_ITEMS)]
    if MEND_CLEAN: src_items.append(("Mendeley (dedup)", sub(MEND_CLEAN, 400)))
    if BRISC_CLEAN: src_items.append(("BRISC (dedup)", BRISC_CLEAN))
    if BRATS_ITEMS: src_items.append(("BraTS", sub(BRATS_ITEMS, 400)))
    if BD_CLEAN: src_items.append(("BDNeuro", sub(BD_CLEAN, 400)))
    items = [(p, y, s) for s, its in src_items for p, y in its]
    fig, ax = plt.subplots(len(TSNE_MODELS), 2, figsize=(8, 3.6 * len(TSNE_MODELS)), squeeze=False)
    for r, n in enumerate(TSNE_MODELS):
        m = load_checkpoint(n, READY[n][0], 1); mean, std = norm_for(n, 1)
        arrs = [load_u8(p) for p, _, _ in items]; feats = []
        with torch.no_grad():
            for i in range(0, len(arrs), 64):
                with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
                    feats.append(embed(m, to_batch(arrs[i:i + 64], mean, std)).float().cpu().numpy())
        del m; feats = np.concatenate(feats)
        src = np.array([s for _, _, s in items]); yy = np.array([y for _, y, _ in items])
        Z = TSNE(n_components=2, perplexity=30, init="pca", random_state=0).fit_transform(feats)
        for s in dict.fromkeys(src):
            ax[r, 0].scatter(*Z[src == s].T, s=4, alpha=.6, label=s)
        for k, c in enumerate(CLASSES):
            ax[r, 1].scatter(*Z[yy == k].T, s=4, alpha=.6, color=CLASS_COLORS[c], label=c)
        ax[r, 0].set_title(f"{n}: colour = source dataset", fontsize=8); ax[r, 1].set_title(f"{n}: colour = tumour type", fontsize=8)
        for a in ax[r]: a.set(xticks=[], yticks=[]); a.legend(frameon=False, fontsize=6, markerscale=3)
        # how easily can the SOURCE be read from the features? (within glioma + meningioma, present in all sets)
        gm = np.isin(yy, [0, 1])
        acc_src = cross_val_score(LogisticRegression(max_iter=2000), feats[gm], src[gm], cv=5).mean()
        acc_cls = cross_val_score(LogisticRegression(max_iter=2000), feats[gm], yy[gm], cv=5).mean()
        tsne_rows.append({"Model": n, "Source predictable from features (%)": round(100 * acc_src, 1),
                          "Tumour type predictable (%)": round(100 * acc_cls, 1),
                          "Chance (source, %)": round(100 * pd.Series(src[gm]).value_counts(normalize=True).max(), 1)})
    fig.suptitle("t-SNE of penultimate features (fold-1 model, seed 0)", fontsize=9)
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig_tsne_source_vs_class.png"); plt.show()
    df_tsne = pd.DataFrame(tsne_rows); print(df_tsne.to_string(index=False))
    df_tsne.to_csv(OUT_DIR / "tables" / "table_feature_source.csv", index=False)

## 7.5 The solutions side by side (experiments 1, 1b, 2, 3)
One table and four figures that answer: *does the solution keep the accuracy, use the tumour, and transfer?*

In [ ]:
# ─── 7.5a Solution table + solution map ───
SOL_MODELS = [n for n in ["ROI-Loc", "ROI-m06-Transplant", "ROI-Crop", "ROI-Crop-m06", "ROI-Crop-m15", "ROI-Mask", "CNX-Transplant", "CNX-Strip",
                          "CNX-Plain", "CTR-Net", "MTL-Seg", "RRR", "CAM-Guide", "ConvNeXt-Tiny", "MTK"] if n in READY]
def cv_f1(n):
    return 100 * np.mean([rec_of(n, s, f)["val_f1"] for s in READY[n] for f in range(1, N_FOLDS + 1)])   # abstain-corrected
SOLV = {}
if RUN_EVALUATION and SOL_MODELS:
    bd_keep = np.isin(paths_of("bdneuro"), [p for p, _ in BD_CLEAN]) if "bdneuro" in EVAL_SETS and BD_CLEAN else None
    d_rel = df_rel.set_index("Model") if RELIANCE else None
    d_r2 = df_rel2.set_index("Model") if len(df_rel2) else None
    d_rc = df_relc.set_index("Model") if RELIANCE else None
    rows = []
    for n in SOL_MODELS:
        r = {"Model": n, "CV macro-F1 (%)": round(cv_f1(n), 2),
             "Test macro-F1 (%)": round(100 * macro_f1(TEST_Y, hard(ensemble_probs(n, "test"))), 2)}
        if d_rel is not None and n in d_rel.index:
            r["Reliance (swap)"] = d_rel.loc[n, "Reliance (swap)"]
            r["Skull/position-only BA (%)"] = d_rel.loc[n, "skull / position only"]
            r["Pituitary correct when hidden (%)"] = d_rc.loc[n, "pituitary still correct (%)"]
        if d_r2 is not None and n in d_r2.index:
            r["Reliance (other patient)"] = d_r2.loc[n, "R other patient 9"]; r["Reliance (swap 41)"] = d_r2.loc[n, "R swap 41"]
        if "brats" in EVAL_SETS:
            r["BraTS AUC"] = round(roc_auc_score(*brats_auc_scores(n)[:2]), 4)
        if bd_keep is not None:
            r["BDNeuro BA (%)"] = round(100 * bal_acc(labels_of("bdneuro")[bd_keep], hard(ensemble_probs(n, "bdneuro")[bd_keep])), 1)
        if "mendeley" in EVAL_SETS:
            r["Mendeley macro-F1 (%)"] = round(100 * macro_f1(labels_of("mendeley"), hard(ensemble_probs(n, "mendeley"))), 1)
        if "bfree" in EVAL_SETS:
            r["Confident on tumour-free (%)"] = round(100 * (ensemble_probs(n, "bfree", calibrated=True).max(1) >= 0.85).mean(), 1)
        rows.append(r)
    df_sol = pd.DataFrame(rows); SOLV = df_sol.set_index("Model").to_dict("index")
    print("ALL SOLUTIONS AND REFERENCES (seed 0 ensembles; CV over all trained seeds). Goal: high accuracy AND high reliance")
    print("AND high BraTS AUC AND low confidence on tumour-free slices.")
    print(df_sol.to_string(index=False)); df_sol.to_csv(OUT_DIR / "tables" / "table_solutions.csv", index=False)

    if "Reliance (swap)" in df_sol:
        fig, ax = plt.subplots(figsize=(7.2, 4.6))
        ax.add_patch(plt.Rectangle((0.6, 96), 0.42, 3.5, color="#2a9d8f", alpha=0.10, lw=0))
        ax.text(0.61, 99.2, "goal: uses the tumour AND accurate", fontsize=7, color="#2a9d8f", va="top")
        for _, r in df_sol.iterrows():
            if pd.isna(r.get("Reliance (swap)")):
                continue
            sz = 40 + 1200 * max(r.get("BraTS AUC", 0.8) - 0.8, 0)
            ax.scatter(r["Reliance (swap)"], r["CV macro-F1 (%)"], s=sz, color=FAMILY_COLOR[family_of(r.Model)],
                       edgecolor="k" if r.Model == PROPOSED else "white", lw=1.2, zorder=3, alpha=0.9)
            ax.annotate(r.Model, (r["Reliance (swap)"], r["CV macro-F1 (%)"]), fontsize=6.5, xytext=(5, -3),
                        textcoords="offset points", annotation_clip=True)
        f1s = df_sol["CV macro-F1 (%)"].dropna()
        ax.set(xlabel="Tumour-reliance score  (0 = ignores the tumour, 1 = needs it)", ylabel="Cross-validation macro-F1 (%)",
               xlim=(0, 1.02), ylim=(min(88, f1s.min() - 1), 100),
               title="Solution map: accuracy vs. use of the tumour (dot size = BraTS AUC)")
        fig.tight_layout(); fig.savefig(FIG_DIR / "fig_solution_map.png", bbox_inches="tight"); plt.show()

In [ ]:
# ─── 7.5b Context sweep: how much surrounding tissue turns into a shortcut? (experiment 1b) ───
SWEEP = [(m, lab) for m, lab in [("ROI-Crop", "tight box\n(margin 0.2)"), ("ROI-Crop-m06", "margin 0.6"),
                                 ("ROI-Crop-m15", "margin 1.5"), ("CNX-Plain", "whole slice")] if m in SOLV]
if RUN_EVALUATION and len(SWEEP) >= 2 and "Reliance (swap)" in df_sol:
    xs = np.arange(len(SWEEP)); g = lambda col: [SOLV[m].get(col, np.nan) for m, _ in SWEEP]
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
    ax[0].plot(xs, g("CV macro-F1 (%)"), "-o", color="#30638e", label="CV macro-F1 (%)")
    ax[0].plot(xs, 100 * np.array(g("BraTS AUC")), "-s", color="#2a9d8f", label="BraTS AUC (x100)")
    lo_ = np.nanmin(np.r_[g("CV macro-F1 (%)"), 100 * np.array(g("BraTS AUC"))])
    ax[0].set(xticks=xs, xticklabels=[l for _, l in SWEEP], ylabel="%", title="Accuracy and transfer",
              ylim=(min(85, lo_ - 2), 100))
    ax[0].legend(frameon=False, fontsize=7)
    ax[1].plot(xs, g("Reliance (swap)"), "-o", color="#d1495b", label="tumour reliance")
    ax[1].plot(xs, np.array(g("Skull/position-only BA (%)")) / 100, "-^", color="#edae49", label="skull/position-only BA (/100)")
    ax[1].axhline(1 / 3, color="k", ls="--", lw=0.8); ax[1].text(xs[-1], 0.35, "guessing", fontsize=6, ha="right")
    if "ROI-m06-Transplant" in SOLV and "ROI-Crop-m06" in dict(SWEEP):
        xm = [m for m, _ in SWEEP].index("ROI-Crop-m06"); c6 = SOLV["ROI-m06-Transplant"]
        ax[0].scatter([xm], [c6["CV macro-F1 (%)"]], marker="*", s=160, color="#264653", zorder=4, label="margin 0.6 + transplant (CV)")
        ax[0].scatter([xm], [100 * c6.get("BraTS AUC", np.nan)], marker="*", s=160, color="#2a9d8f", zorder=4)
        ax[1].scatter([xm], [c6.get("Reliance (swap)", np.nan)], marker="*", s=160, color="#264653", zorder=4,
                      label="margin 0.6 + transplant")
        ax[0].legend(frameon=False, fontsize=7)
    ax[1].set(xticks=xs, xticklabels=[l for _, l in SWEEP], ylim=(0, 1), title="Use of the tumour vs. shortcut")
    ax[1].legend(frameon=False, fontsize=7)
    fig.suptitle("Context sweep: the classifier sees more and more of the slice around the found tumour", fontsize=9)
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig_context_sweep.png", bbox_inches="tight"); plt.show()
elif RUN_EVALUATION:
    print("Context sweep: needs at least ROI-Crop + one wider-margin model (not all trained yet -> next run).")

In [ ]:
# ─── 7.5c ROI-Loc: what does each input contribute? (appearance vs location; location knock-out) ───
if RUN_EVALUATION and "ROI-Loc" in READY and MASKS:
    dloc = df3[df3.filepath.isin(MASKS)].reset_index(drop=True)
    Xl = np.array([loc_feats(load_u8(p), MASKS[p]) for p in dloc.filepath]); yl = dloc.label.values
    dev_m = (dloc.role == "dev").values
    lr_loc = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
    pr_cv = cross_val_predict(lr_loc, Xl[dev_m], yl[dev_m], groups=dloc.pid.values[dev_m], cv=GroupKFold(5))
    lr_loc.fit(Xl[dev_m], yl[dev_m]); pr_te = lr_loc.predict(Xl[~dev_m])
    # knock-out: replace every location by the average training location -> how much accuracy is lost?
    s0 = READY["ROI-Loc"][0]; mean_loc = torch.tensor(Xl[dev_m].mean(0), dtype=torch.float32)
    ko = []
    for f in range(1, N_FOLDS + 1):
        m = load_checkpoint("ROI-Loc", s0, f); mean, std = norm_for("ROI-Loc", f)
        m.loc_override = mean_loc; ko.append(softmax_np(predict_any("ROI-Loc", f, m, ARRS["test"], mean, std))); m.loc_override = None
    P_ko = np.mean(ko, 0)
    parts = [("location only\n(6 numbers, no image)", 100 * macro_f1(yl[dev_m], pr_cv), 100 * macro_f1(yl[~dev_m], pr_te)),
             ("appearance only\n(ROI-Crop)", SOLV.get("ROI-Crop", {}).get("CV macro-F1 (%)", np.nan),
              SOLV.get("ROI-Crop", {}).get("Test macro-F1 (%)", np.nan)),
             ("ROI-Loc with location\nknocked out", np.nan, 100 * macro_f1(TEST_Y, hard(P_ko))),
             ("both\n(ROI-Loc)", SOLV["ROI-Loc"]["CV macro-F1 (%)"], SOLV["ROI-Loc"]["Test macro-F1 (%)"])]
    df_parts = pd.DataFrame(parts, columns=["Input", "CV macro-F1 (%)", "Test macro-F1 (%)"]).round(2)
    print("What each input of ROI-Loc contributes (location-only uses the EXPERT outline = an upper bound):")
    print(df_parts.assign(Input=df_parts.Input.str.replace("\n", " ")).to_string(index=False))
    df_parts.to_csv(OUT_DIR / "tables" / "table_roiloc_parts.csv", index=False)
    fig, ax = plt.subplots(figsize=(7, 3.4)); xs = np.arange(len(df_parts))
    ax.bar(xs - 0.2, df_parts["CV macro-F1 (%)"], 0.4, color="#30638e", label="cross-validation")
    ax.bar(xs + 0.2, df_parts["Test macro-F1 (%)"], 0.4, color="#2a9d8f", label="held-out test")
    for x_, v1, v2 in zip(xs, df_parts["CV macro-F1 (%)"], df_parts["Test macro-F1 (%)"]):
        for dx, v in [(-0.2, v1), (0.2, v2)]:
            if np.isfinite(v): ax.text(x_ + dx, v + 0.5, f"{v:.1f}", ha="center", fontsize=6.5)
    ax.set(xticks=xs, xticklabels=df_parts.Input, ylabel="Macro-F1 (%)", ylim=(0, 104),
           title="ROI-Loc = what the tumour looks like + where it is (and nothing else)")
    ax.axhline(100 / 3, color="k", ls="--", lw=0.8); ax.legend(frameon=False, fontsize=7)
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig_roiloc_parts.png", bbox_inches="tight"); plt.show()

In [ ]:
# ─── 7.5d Transplant (solution 2) and skull control (experiment 3) ───
if RUN_EVALUATION and RELIANCE:
    pair = [n for n in ["CNX-Plain", "CNX-Transplant"] if n in REL]
    if len(pair) == 2:
        fig, ax = plt.subplots(figsize=(7.5, 3.3)); xs = np.arange(N_CLASSES)
        for k_, n in enumerate(pair):
            v = [df_relc.set_index("Model").loc[n, f"{c} still correct (%)"] for c in CLASSES]
            ax.bar(xs + (k_ - 0.5) * 0.38, v, 0.38, color=FAMILY_COLOR[family_of(n)], label=n)
            for x_, vv in zip(xs, v): ax.text(x_ + (k_ - 0.5) * 0.38, vv + 1, f"{vv:.0f}", ha="center", fontsize=7)
        ax.axhline(100 / 3, color="k", ls="--", lw=0.8)
        ax.set(xticks=xs, xticklabels=CLASSES, ylabel="Still correct with the tumour HIDDEN (%)", ylim=(0, 105),
               title="Tumour transplant: does the ordinary model stop answering without the tumour?")
        ax.legend(frameon=False, fontsize=7)
        fig.tight_layout(); fig.savefig(FIG_DIR / "fig_transplant_effect.png", bbox_inches="tight"); plt.show()
if RUN_EVALUATION and "brats" in EVAL_SETS and "CNX-Strip" in READY:
    refs = [n for n in ["ROI-Crop", "ROI-Loc"] if n in READY]
    df_strip = pd.concat([brats_paired(r, ["CNX-Strip"]) for r in refs] +
                         ([brats_paired("CNX-Strip", ["CNX-Plain"])] if "CNX-Plain" in READY else []), ignore_index=True)
    print("SKULL CONTROL on BraTS (glioma-vs-meningioma AUC differences, patient bootstrap):")
    print(df_strip.to_string(index=False)); df_strip.to_csv(OUT_DIR / "tables" / "table_skull_control.csv", index=False)
    print("How to read: if 'CNX-Strip − CNX-Plain' is large and 'ROI-Crop − CNX-Strip' is ~0, ROI's BraTS gain came mostly from"
          "\nremoving the skull. If 'ROI-Crop − CNX-Strip' stays clearly positive, the gain is NOT explained by the skull.")

## 8. Calibration and high-confidence predictions (internal test)

In [ ]:
# ─── 8.1 ECE / Brier before and after temperature scaling ───
def ece_score(y, P, bins=15):
    conf, pred = P.max(1), hard(P); edges = np.linspace(0, 1, bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return float(e)

def brier(y, P):
    return float(((P - np.eye(N_CLASSES)[y]) ** 2).sum(1).mean())

cal_rows = []
if RUN_EVALUATION and len(TEST_Y):
    for name in READY:
        for cal in [False, True]:
            P = ensemble_probs(name, "test", calibrated=cal); conf = P.max(1); ok = hard(P) == TEST_Y
            hi = conf >= CONF_THRESHOLD
            cal_rows.append({"Model": name, "Temperature scaled": cal, "ECE": round(ece_score(TEST_Y, P), 4),
                             "Brier": round(brier(TEST_Y, P), 4),
                             f"Coverage ≥{CONF_THRESHOLD} (%)": round(100 * hi.mean(), 1),
                             f"Precision ≥{CONF_THRESHOLD} (%)": round(100 * ok[hi].mean(), 1) if hi.any() else np.nan,
                             "Confident errors": int((hi & ~ok).sum())})
    df_cal = pd.DataFrame(cal_rows)
    print(df_cal.to_string(index=False))
    df_cal.to_csv(OUT_DIR / "tables" / "table_calibration.csv", index=False)

    show = list(dict.fromkeys(n for n in [PROPOSED, "CNX-Plain", "MTK"] if n in READY))
    fig, ax = plt.subplots(1, len(show), figsize=(3.2 * len(show), 3.1), squeeze=False)
    for a, name in zip(ax[0], show):
        for cal, col in [(False, "#bbbbbb"), (True, "#30638e")]:
            P = ensemble_probs(name, "test", calibrated=cal); conf, ok = P.max(1), hard(P) == TEST_Y
            edges = np.linspace(0, 1, 11); cx, cy = [], []
            for lo, hi_ in zip(edges[:-1], edges[1:]):
                m = (conf > lo) & (conf <= hi_)
                if m.sum() >= 5:
                    cx.append(conf[m].mean()); cy.append(ok[m].mean())
            a.plot(cx, cy, "o-", color=col, label=("TS " if cal else "raw ") + f"ECE {ece_score(TEST_Y, P):.3f}")
        a.plot([0, 1], [0, 1], ":", color="k"); a.set(title=name, xlabel="Confidence", ylabel="Accuracy")
        a.legend(frameon=False, fontsize=7)
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig_calibration.png"); plt.show()

In [ ]:
# ─── 8.2 Conformal prediction (cross-conformal): "confident answer" or "send to a radiologist" ───
# Each fold model is calibrated on its own validation fold (temperature-scaled). For a new image, every fold model gives
# a conformal p-value per class, and the p-values are pooled (cross-conformal, Vovk 2015). Class k is in the
# prediction set if its pooled p-value > alpha. One class in the set = confident answer; 2-3 classes = refer.
# Guarantee: on data like the development data the true class is in the set >= 1 - 2*alpha (in practice ~1 - alpha).
# New hospitals are a robustness test (no guarantee there).
ALPHAS = [0.05, 0.10]
def conformal_sets(name, set_name, alpha, keep=None):
    s = READY[name][0]; cal, test_scores = [], []
    for f in range(1, N_FOLDS + 1):
        _, rec_path, oof_path = run_paths(name, s, f)
        y = np.array(load_json(rec_path)["val_labels"]); T = TEMPS[(name, s, f)]
        Pv = softmax_np(np.load(oof_path) / T); cal.append(np.sort(1 - Pv[np.arange(len(y)), y]))
        Pt = softmax_np(np.load(pred_path(name, s, f, set_name)) / T)
        test_scores.append(1 - (Pt if keep is None else Pt[keep]))
    N = sum(len(c) for c in cal)
    counts = sum(len(c) - np.searchsorted(c, ts, side="left") for c, ts in zip(cal, test_scores))   # #{cal >= score}
    pval = (counts + 1) / (N + 1)
    return pval > alpha

conf_rows = []
if RUN_EVALUATION and len(TEST_Y):
    eval_sets = {"Internal test": ("test", None)}
    if "brats" in EVAL_SETS:
        eval_sets["BraTS"] = ("brats", None)
    if "bdneuro" in EVAL_SETS and BD_CLEAN:
        eval_sets["BDNeuro (unique)"] = ("bdneuro", np.isin(paths_of("bdneuro"), [p for p, _ in BD_CLEAN]))
    for name in READY:
        for alpha in ALPHAS:
            for set_label, (s, keep) in eval_sets.items():
                S = conformal_sets(name, s, alpha, keep); y = labels_of(s) if keep is None else labels_of(s)[keep]
                size = S.sum(1); single = size == 1
                P = ensemble_probs(name, s, calibrated=True); P = P if keep is None else P[keep]
                conf_rows.append({"Model": name, "alpha": alpha, "Set": set_label, "n": len(y),
                                  "Coverage (%)": round(100 * S[np.arange(len(y)), y].mean(), 1),
                                  "Target (%)": round(100 * (1 - alpha)),
                                  "Answered (1 label) (%)": round(100 * single.mean(), 1),
                                  "Accuracy of answers (%)": round(100 * (S[single].argmax(1) == y[single]).mean(), 1) if single.any() else np.nan,
                                  "Referred (%)": round(100 * (~single).mean(), 1)})
    df_conf = pd.DataFrame(conf_rows)
    print("Conformal referral, alpha = 0.05 (target: true class inside the answer set >= 95%):")
    print(df_conf[(df_conf.alpha == 0.05) & df_conf.Model.isin([PROPOSED, "CNX-Plain", "MTL-Seg", "RRR", "CAM-Guide", "MTK"])]
          .to_string(index=False))
    df_conf.to_csv(OUT_DIR / "tables" / "table_conformal.csv", index=False)

    show = list(dict.fromkeys(n for n in [PROPOSED, "CNX-Plain", "MTK"] if n in READY))
    d = df_conf[(df_conf.alpha == 0.05) & df_conf.Model.isin(show)].copy()
    fig, ax = plt.subplots(1, 3, figsize=(11, 3.2))
    for a, col, title in zip(ax, ["Coverage (%)", "Answered (1 label) (%)", "Accuracy of answers (%)"],
                             ["Coverage (target 95%)", "Answered without referral", "Accuracy of the answered cases"]):
        d.pivot(index="Set", columns="Model", values=col)[show].plot.bar(ax=a, rot=0, width=.8,
                                                                         color=[FAMILY_COLOR[family_of(n)] if n != "MTK" else "#888888" for n in show])
        a.set(title=title, ylabel="%", ylim=(0, 105), xlabel=""); a.legend(frameon=False, fontsize=6)
        if col == "Coverage (%)":
            a.axhline(95, color="k", ls="--", lw=1)
    fig.suptitle("Conformal referral (alpha = 0.05): answer when sure, refer the rest", fontsize=9)
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig_conformal.png", bbox_inches="tight"); plt.show()

In [ ]:
# ─── 8.3 Referral using the tumour map: refer when the model cannot find a tumour ───
# Risk-coverage curve: answer the most trustworthy X% of cases, error rate among them. Lower curve = safer.
# CTR-Net can use two signals: softmax confidence, and confidence x "tumour found" (peak of its tumour map).
def risk_coverage(score, correct):
    o = np.argsort(-score); err = 1 - correct[o].astype(float)
    cov = np.arange(1, len(o) + 1) / len(o); risk = np.cumsum(err) / np.arange(1, len(o) + 1)
    return cov, risk, float(np.mean(risk))                      # AURC = mean risk over all coverages

sel_rows = []
if RUN_EVALUATION:
    sets = [("Internal test", "test", None)]
    if "brats" in EVAL_SETS: sets.append(("BraTS", "brats", None))
    if "bdneuro" in EVAL_SETS and BD_CLEAN: sets.append(("BDNeuro (unique)", "bdneuro", None))
    show = list(dict.fromkeys(n for n in [PROPOSED, "ROI-m06-Transplant", "ROI-Crop-m06", "CNX-Transplant", "CNX-Plain", "CTR-Net", "MTL-Seg"] if n in READY))
    fig, ax = plt.subplots(1, len(sets), figsize=(3.8 * len(sets), 3.3), squeeze=False)
    for a, (lab, s, keep) in zip(ax[0], sets):
        y = labels_of(s)
        for n in show:
            P = ensemble_probs(n, s, calibrated=True)
            cov, risk, aurc = risk_coverage(P.max(1), hard(P) == y)
            a.plot(100 * cov, 100 * risk, lw=1.8 if n == PROPOSED else 1,
                   color={"ROI-Crop": "#f08a8f", "ROI-Crop-m06": "#f4a261", "ROI-m06-Transplant": "#264653", "CTR-Net": "#9b5de5",
                          "MTL-Seg": "#edae49"}.get(n, FAMILY_COLOR[family_of(n)]),
                   label=f"{n} (AURC {100 * aurc:.1f})")
            sel_rows.append({"Set": lab, "Model": n, "AURC (%)": round(100 * aurc, 2),
                             "Error when answering 80% (%)": round(100 * risk[int(0.8 * len(risk)) - 1], 1)})
        a.set(title=lab, xlabel="Cases answered (%)", ylabel="Error among answered (%)"); a.legend(frameon=False, fontsize=6)
    fig.suptitle("Selective prediction: lower curve = fewer mistakes for the same number of answered cases", fontsize=9)
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig_risk_coverage.png", bbox_inches="tight"); plt.show()
    df_sel = pd.DataFrame(sel_rows); print(df_sel.to_string(index=False))
    df_sel.to_csv(OUT_DIR / "tables" / "table_selective_prediction.csv", index=False)
    print("AURC = average error over all referral levels (lower = better).")

In [ ]:
# ─── 8.4 Tumour-free slices (BraTS): does the model still confidently name a tumour type? ───
# Real scans contain many slices WITHOUT tumour. A trustworthy model should be unsure there (or refer).
# A shortcut model gives a confident tumour type anyway. Same BraTS patients as the tumour slices above.
free_rows = []
if RUN_EVALUATION and "bfree" in EVAL_SETS:
    for n in READY:
        Pf = ensemble_probs(n, "bfree", calibrated=True); Pt = ensemble_probs(n, "brats", calibrated=True)
        Sf = conformal_sets(n, "bfree", 0.05)
        hf = hard(Pf); dist = np.bincount(hf[hf >= 0], minlength=N_CLASSES) / len(Pf)
        free_rows.append({"Model": n,
                          "Confident (>=0.85) on tumour-free (%)": round(100 * (Pf.max(1) >= 0.85).mean(), 1),
                          "Mean confidence on tumour-free": round(float(Pf.max(1).mean()), 3),
                          "Mean confidence on tumour slices": round(float(Pt.max(1).mean()), 3),
                          "AUROC tumour vs tumour-free (by confidence)": round(roc_auc_score(
                              np.r_[np.ones(len(Pt)), np.zeros(len(Pf))], np.r_[Pt.max(1), Pf.max(1)]), 3),
                          "Referred by conformal (%)": round(100 * (Sf.sum(1) != 1).mean(), 1),
                          **{f"called {c} (%)": round(100 * dist[k], 1) for k, c in enumerate(CLASSES)},
                          "abstained (no tumour found) (%)": round(100 * (hf < 0).mean(), 1)})
    df_free = pd.DataFrame(free_rows).sort_values("Confident (>=0.85) on tumour-free (%)")
    print(f"{len(EVAL_SETS['bfree'])} tumour-free BraTS slices. Lower 'confident' and higher 'referred' = more trustworthy.")
    print("AUROC column = can the model's own confidence tell real tumour slices from tumour-free ones? (0.5 = no, 1 = perfectly)")
    print(df_free.to_string(index=False)); df_free.to_csv(OUT_DIR / "tables" / "table_tumour_free.csv", index=False)
    order = list(df_free.Model)
    fig, ax = plt.subplots(figsize=(max(8, 0.6 * len(order)), 3.6))
    xs = np.arange(len(order))
    ax.bar(xs - 0.2, df_free.set_index("Model").loc[order, "Confident (>=0.85) on tumour-free (%)"], 0.4,
           color=[FAMILY_COLOR[family_of(n)] for n in order], label="confident on TUMOUR-FREE slices (should be low)")
    ax.bar(xs + 0.2, 100 * df_free.set_index("Model").loc[order, "Mean confidence on tumour slices"], 0.4,
           color="#cccccc", label="mean confidence on tumour slices (reference)")
    ax.set_xticks(xs); ax.set_xticklabels(order, rotation=35, ha="right", fontsize=7)
    ax.set(ylabel="%", ylim=(0, 105), title="Do models confidently name a tumour type when there is NO tumour?")
    ax.legend(frameon=False, fontsize=7)
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig_tumour_free.png", bbox_inches="tight"); plt.show()

## 8.5 Shortcut alarm (experiment 4): a per-image safety check that needs no training
For every new image: find the tumour with the tumour map (MTL-Seg, same fold), **hide it**, and predict again.
* If the confidence in the answer **drops** a lot → the model used the tumour → answer.
* If the answer **does not change** (or no tumour was found) → the model answered from the surroundings → **refer**.

This turns the tumour-hiding test into a tool that can run on every case. Compared with plain confidence on the
internal test, BraTS, BDNeuro and the tumour-free slices.

In [ ]:
# ─── 8.5 Shortcut alarm ───
ALARM_MODELS = [n for n in ["CNX-Plain", "CNX-Transplant", "CTR-Net", "MTL-Seg", "CAM-Guide", "ConvNeXt-Tiny"] if n in READY]
ALARM_SETS = [s for s in ["test", "brats", "bdneuro", "bfree"] if s in EVAL_SETS]
ALARM = {}
if RUN_EVALUATION and ALARM_MODELS and MASKS:
    banner("8.5 Shortcut alarm", f"{len(ALARM_MODELS)} models x {len(ALARM_SETS)} sets (cached after the first run)")
    (OUT_DIR / "alarm").mkdir(exist_ok=True)
    alarm_path = lambda n, s_, f, st: OUT_DIR / "alarm" / f"{n}_s{s_}_f{f}_{st}.npy"
    FOUND = {st: np.zeros(len(EVAL_SETS[st]), bool) for st in ALARM_SETS}
    for f in range(1, N_FOLDS + 1):
        need = [(n, st) for n in ALARM_MODELS for st in ALARM_SETS if not cached_ok(alarm_path(n, READY[n][0], f, st), len(EVAL_SETS[st]))]
        found_file = OUT_DIR / "alarm" / f"found_f{f}.npz"
        if not need and found_file.exists():
            z = np.load(found_file)
            for st in ALARM_SETS: FOUND[st] |= z[st]
            continue
        rng_a = np.random.default_rng(100 + f); HID, fnd = {}, {}
        for st in ALARM_SETS:
            maps = roi_seg_maps(ARRS[st], f); hs, fl = [], []
            for a, mp in zip(ARRS[st], maps):
                b = box_from_mask(mp >= 0.5)
                if b is None:
                    hs.append(a); fl.append(False); continue
                reg = cv2.dilate(b[4].astype(np.uint8), HIDE_KER).astype(bool); bm = brain_mask(a)
                sh = shifted_region(reg, bm, rng_a)
                if sh is None:
                    hs.append(cv2.inpaint(a, reg.astype(np.uint8) * 255, 7, cv2.INPAINT_TELEA))
                else:
                    dy, dx, _ = sh; ys, xs = np.where(reg); h_ = a.copy(); h_[ys, xs] = a[ys + dy, xs + dx]; hs.append(h_)
                fl.append(True)
            HID[st] = hs; fnd[st] = np.array(fl); FOUND[st] |= fnd[st]
        np.savez_compressed(found_file, **fnd)
        for n, st in need:
            m = load_checkpoint(n, READY[n][0], f); mean, std = norm_for(n, f); _CUR_SET["name"] = st
            np.save(alarm_path(n, READY[n][0], f, st), predict_any(n, f, m, HID[st], mean, std))
        _CUR_SET["name"] = None; del HID; free_mem()
        print(f"  fold {f} done", flush=True)

    def alarm_probs(n, st):
        s_ = READY[n][0]
        return np.mean([softmax_np(np.load(alarm_path(n, s_, f, st)) / TEMPS[(n, s_, f)]) for f in range(1, N_FOLDS + 1)], 0)

    rows, curves = [], {}
    for n in ALARM_MODELS:
        for st in ALARM_SETS:
            P = ensemble_probs(n, st, calibrated=True); Ph = alarm_probs(n, st); yhat = hard(P)
            conf = P.max(1); drop = conf - Ph[np.arange(len(P)), yhat]
            ALARM[(n, st)] = dict(conf=conf, drop=drop, combo=np.where(drop >= ALARM_DROP, conf, conf - 1), yhat=yhat)
        for st, lab in [("test", "Internal test"), ("brats", "BraTS"), ("bdneuro", "BDNeuro (unique)")]:
            if st not in ALARM_SETS:
                continue
            keep = np.isin(paths_of(st), [p for p, _ in BD_CLEAN]) if st == "bdneuro" else np.ones(len(EVAL_SETS[st]), bool)
            A = ALARM[(n, st)]; corr = (A["yhat"] == labels_of(st))[keep]
            r = {"Model": n, "Set": lab}
            for sc in ["conf", "drop", "combo"]:
                cov, risk, aurc = risk_coverage(A[sc][keep], corr); r[f"AURC {sc} (%)"] = round(100 * aurc, 2)
                if st == "brats": curves[(n, sc)] = (cov, risk)
            ans = A["drop"][keep] >= ALARM_DROP
            r["Answered by alarm rule (%)"] = round(100 * ans.mean(), 1)
            r["Accuracy of answered (%)"] = round(100 * corr[ans].mean(), 1) if ans.any() else np.nan
            r["Accuracy of all (%)"] = round(100 * corr.mean(), 1)
            rows.append(r)
        if "bfree" in ALARM_SETS and "brats" in ALARM_SETS:
            t_, f_ = ALARM[(n, "brats")], ALARM[(n, "bfree")]; yy = np.r_[np.ones(len(t_["conf"])), np.zeros(len(f_["conf"]))]
            rows.append({"Model": n, "Set": "BraTS tumour vs tumour-free",
                         "AUROC conf": round(roc_auc_score(yy, np.r_[t_["conf"], f_["conf"]]), 3),
                         "AUROC drop": round(roc_auc_score(yy, np.r_[t_["drop"], f_["drop"]]), 3),
                         "AUROC combo": round(roc_auc_score(yy, np.r_[t_["combo"], f_["combo"]]), 3),
                         "Tumour-free answered by alarm rule (%)": round(100 * (f_["drop"] >= ALARM_DROP).mean(), 1),
                         "Tumour-free confident >= 0.85 (%)": round(100 * (f_["conf"] >= 0.85).mean(), 1)})
    df_alarm = pd.DataFrame(rows)
    print("conf = plain confidence | drop = how much the confidence falls when the found tumour is hidden |"
          f"\ncombo = answer only if drop >= {ALARM_DROP} (then rank by confidence). AURC: lower = safer. AUROC: higher = better.")
    a1 = df_alarm[df_alarm.Set != "BraTS tumour vs tumour-free"].dropna(axis=1, how="all")
    a2 = df_alarm[df_alarm.Set == "BraTS tumour vs tumour-free"].dropna(axis=1, how="all")
    print(a1.to_string(index=False)); print("\nCan the score tell real tumour slices from tumour-free slices?")
    print(a2.to_string(index=False))
    df_alarm.to_csv(OUT_DIR / "tables" / "table_shortcut_alarm.csv", index=False)

    ref = "CNX-Plain" if "CNX-Plain" in ALARM_MODELS else ALARM_MODELS[0]
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
    for sc, col, lab in [("conf", "#888888", "confidence only"), ("drop", "#edae49", "confidence drop when tumour hidden"),
                         ("combo", "#d1495b", f"alarm rule (drop >= {ALARM_DROP}) + confidence")]:
        if (ref, sc) in curves:
            cov, risk = curves[(ref, sc)]; ax[0].plot(100 * cov, 100 * risk, color=col, label=lab)
    ax[0].set(xlabel="BraTS cases answered (%)", ylabel="Error among answered (%)", title=f"{ref} on BraTS: risk vs coverage")
    ax[0].legend(frameon=False, fontsize=6.5)
    if "bfree" in ALARM_SETS:
        t_, f_ = ALARM[(ref, "brats")], ALARM[(ref, "bfree")]
        bins = np.linspace(-0.4, 1, 29)
        ax[1].hist(t_["drop"], bins, alpha=0.6, color="#30638e", density=True, label="slices WITH tumour")
        ax[1].hist(f_["drop"], bins, alpha=0.6, color="#d1495b", density=True, label="tumour-FREE slices")
        ax[1].axvline(ALARM_DROP, color="k", ls="--", lw=1)
        ax[1].set(xlabel="Confidence drop when the found tumour is hidden", ylabel="density", title=f"{ref}: the alarm signal")
        ax[1].legend(frameon=False, fontsize=6.5)
        d_ = a2.set_index("Model")
        xs = np.arange(len(d_)); w = 0.38
        ax[2].bar(xs - w / 2, d_["AUROC conf"], w, color="#888888", label="confidence")
        ax[2].bar(xs + w / 2, d_["AUROC combo"], w, color="#d1495b", label="alarm rule")
        ax[2].axhline(0.5, color="k", ls="--", lw=0.8)
        ax[2].set(xticks=xs, ylim=(0.3, 1), ylabel="AUROC", title="Tumour vs tumour-free slices")
        ax[2].set_xticklabels(d_.index, rotation=30, ha="right", fontsize=7); ax[2].legend(frameon=False, fontsize=6.5)
    fig.suptitle("Shortcut alarm: hide the found tumour at test time - if the answer does not change, refer the case", fontsize=9)
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig_shortcut_alarm.png", bbox_inches="tight"); plt.show()
else:
    df_alarm = pd.DataFrame()

## 9. Explainability against expert tumour masks (LayerCAM at 14x14 for every model, with chance baselines)
Raw maps only (no brain masking, no smoothing) for all metrics; maps are averaged over the 5 fold models.
Sets: deduplicated BRISC tumour images (100 per class) and BraTS slices (150 per subset), all with expert masks.
Baselines: a random map hits the mask with probability = mask area fraction; "centre" = mask contains the image centre.

In [ ]:
# ─── 9.1 LayerCAM engine ───
class GradCAMpp:   # name kept for compatibility; computes LayerCAM
    def __init__(self, model, layer):
        self.model, self.acts, self.grads = model, None, None
        def fwd(_, __, out):
            self.acts = out
            out.register_hook(lambda g: setattr(self, "grads", g))
        self.handle = layer.register_forward_hook(fwd)

    def __call__(self, x, target):
        self.model.zero_grad(set_to_none=True)
        logits = self.model(x)
        logits[0, target].backward()
        A, G = self.acts.detach().float(), self.grads.detach().float()
        cam = F.relu((F.relu(G) * A).sum(1, keepdim=True))       # LayerCAM (Jiang et al. 2021): pixel-wise weights
        cam = F.interpolate(cam, (IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)[0, 0]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam.cpu().numpy(), torch.softmax(logits.detach(), 1)[0].cpu().numpy()

    def close(self):
        self.handle.remove()

def ensemble_cams(name, items, target_of, seed=None):
    """Mean LayerCAM over the 5 fold models of one seed. target_of(i) -> class index."""
    seed = READY[name][0] if seed is None else seed
    cams = np.zeros((len(items), IMG_SIZE, IMG_SIZE), np.float32)
    for f in range(1, N_FOLDS + 1):
        m = load_checkpoint(name, seed, f); mean, std = norm_for(name, f)
        engine = GradCAMpp(m, cam_target_layer(m, name))
        for i, (p, _) in enumerate(tqdm(items, desc=f"  LayerCAM {name} fold {f}", leave=False)):
            cams[i] += engine(to_batch([load_u8(p)], mean, std), target_of(i))[0] / N_FOLDS
        engine.close(); del m; gc.collect()
        if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return cams

In [ ]:
# ─── 9.2 Quantitative attribution metrics (cached per model -> later runs reuse them) ───
xai_items = []          # (path, label, mask_source, dataset)
for p_, y_ in TEST_ITEMS:                                           # internal test: expert masks from the .mat files
    if p_ in MASKS:
        xai_items.append((p_, y_, "figshare", "Internal test"))
rng = random.Random(SPLIT_SEED)
for k in range(N_CLASSES):
    pool = [(p, y) for p, y in BRISC_CLEAN if y == k and Path(p).stem in BRISC_MASKS]
    for p, y in rng.sample(pool, min(100, len(pool))):
        xai_items.append((p, y, BRISC_MASKS[Path(p).stem], "BRISC (dedup)"))
if len(BRATS_META):
    for d in ["gli", "men"]:
        sub = BRATS_META[BRATS_META.dataset == d]
        sub = sub.sample(min(150, len(sub)), random_state=SPLIT_SEED)
        for r in sub.itertuples():
            xai_items.append((r.path, r.y, str(BRATS_DIR / d / "masks_tc" / Path(r.file).name), f"BraTS-{d.upper()}"))
print(f"XAI images: {dict(Counter(d for *_, d in xai_items))}")

def xai_mask(i):
    p, _, src, _ = xai_items[i]
    return MASKS[p] if src == "figshare" else load_mask(src)

def up224(c):
    return np.asarray(Image.fromarray(np.asarray(c, np.float32)).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR))

XAI_KEY = "layercam14_" + hashlib.md5("|".join(p for p, *_ in xai_items).encode()).hexdigest()[:10]
xai_rows, ALL_CAMS = [], {}                  # ALL_CAMS[name] = (n, 56, 56) float16 heat maps (shown upsampled)
if RUN_XAI and RUN_EVALUATION and XAI_MODELS and xai_items:
    masks = [xai_mask(i) for i in range(len(xai_items))]
    pos = {}
    for s_name in ("test", "brisc", "brats"):
        if s_name in EVAL_SETS:
            pos.update({p: (s_name, i) for i, p in enumerate(paths_of(s_name))})
    banner("9.2 XAI vs expert tumour masks", f"{len(XAI_MODELS)} models x {len(xai_items)} images x 5 folds "
           "(~3-8 min per model; cached models are instant)")
    for name in XAI_MODELS:
        cache = OUT_DIR / "xai" / f"{name}_{XAI_KEY}.npz"; rows_csv = OUT_DIR / "xai" / f"{name}_{XAI_KEY}_rows.csv"
        if cache.exists() and rows_csv.exists():
            ALL_CAMS[name] = np.load(cache)["cams"]; xai_rows += pd.read_csv(rows_csv).to_dict("records")
            print(f"{name:16s} XAI loaded from cache"); continue
        t0 = time.time()
        cams = ensemble_cams(name, [(p, y) for p, y, _, _ in xai_items], lambda i: xai_items[i][1])
        P = {s: ensemble_probs(name, s) for s in {"test", "brisc", "brats"} & set(EVAL_SETS)}
        rows_m = []
        for i, ((p, y, _, d), cam, mk) in enumerate(zip(xai_items, cams, masks)):
            if mk.sum() == 0:
                continue
            s, j = pos[p]
            yx = np.unravel_index(cam.argmax(), cam.shape)
            thr = cam >= 0.3
            rows_m.append({"Model": name, "item": i, "Dataset": d, "Class": IDX2CLASS[y],
                           "correct": bool(P[s][j].argmax() == y), "pred": int(P[s][j].argmax()), "conf": float(P[s][j].max()),
                           "pointing": float(mk[yx]), "energy_in_mask": float(cam[mk].sum() / (cam.sum() + 1e-8)),
                           "iou@0.3": float((thr & mk).sum() / ((thr | mk).sum() + 1e-8)),
                           "chance": float(mk.mean()), "centre": float(mk[IMG_SIZE // 2, IMG_SIZE // 2])})
        small = F.avg_pool2d(torch.from_numpy(cams)[:, None], 4)[:, 0].numpy().astype(np.float16)
        np.savez_compressed(cache, cams=small); pd.DataFrame(rows_m).to_csv(rows_csv, index=False)
        ALL_CAMS[name] = small; xai_rows += rows_m
        print(f"{name:16s} XAI done ({(time.time() - t0) / 60:.1f} min)")
    df_xai_img = pd.DataFrame(xai_rows)
    df_xai_img.to_csv(OUT_DIR / "tables" / "xai_per_image.csv", index=False)
    agg = (df_xai_img.groupby(["Dataset", "Model"])
           [["pointing", "energy_in_mask", "iou@0.3", "chance", "centre"]].mean().round(3))
    agg["n"] = df_xai_img.groupby(["Dataset", "Model"]).size()
    agg_c = (df_xai_img[df_xai_img.correct].groupby(["Dataset", "Model"])
             [["pointing", "energy_in_mask"]].mean().round(3).add_suffix(" (correct only)"))
    df_xai = agg.join(agg_c)
    print(df_xai.to_string())
    df_xai.to_csv(OUT_DIR / "tables" / "table_xai.csv")
    (df_xai_img.groupby(["Dataset", "Class", "Model"])[["pointing", "energy_in_mask", "iou@0.3", "chance"]]
     .mean().round(3).to_csv(OUT_DIR / "tables" / "table_xai_by_class.csv"))
    print("pointing = heat-map peak inside the expert tumour outline; energy_in_mask = share of heat inside it; "
          "chance = what a random map would get (tumour area).")

    fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
    for a, metric in zip(ax, ["pointing", "energy_in_mask"]):
        piv = df_xai_img.groupby(["Dataset", "Model"])[metric].mean().unstack()[XAI_MODELS]
        piv.plot.bar(ax=a, rot=0, width=0.8)
        ch = df_xai_img.groupby("Dataset")["chance"].mean()
        for i, dname in enumerate(piv.index):
            a.hlines(ch[dname], i - 0.4, i + 0.4, colors="k", linestyles="--", lw=1)
        a.set(title=f"{metric} (dashed = random map)", ylim=(0, 1)); a.legend(frameon=False, fontsize=6)
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig_xai_metrics.png"); plt.show()

In [ ]:
# ─── 9.3 Main explainability figure: where does each model look? ───
def xai_compare_panel(picks, fname):
    cols = [("MRI + expert outline", None, None)]
    if SEG_MAPS.get("Internal test") and SEG_MODEL:
        cols.append((f"{PROPOSED}: region used" if is_roi(PROPOSED) else f"{SEG_MODEL}: predicted tumour", SEG_MODEL,
                     "roi" if is_roi(PROPOSED) else "seg"))
    for n in ["CAM-Guide", "CNX-Transplant", "CNX-Plain", "CTR-Net", "MTK"]:
        if n in ALL_CAMS: cols.append((f"{n}: heat map", n, "cam"))
    seg_lookup = {p: mp for its, maps in SEG_MAPS.values() for (p, _), mp in zip(its, maps)}
    info = {n: df_xai_img[df_xai_img.Model == n].set_index("item") for n in ALL_CAMS}
    test_index = {q: i for i, q in enumerate(paths_of("test"))}
    brats_index = {q: i for i, q in enumerate(paths_of("brats"))} if "brats" in EVAL_SETS else {}
    BRATS_P = ensemble_probs(PROPOSED, "brats") if "brats" in EVAL_SETS and PROPOSED in READY else None
    fig, ax = plt.subplots(len(picks), len(cols), figsize=(2.25 * len(cols), 2.45 * len(picks)), squeeze=False)
    for r, i in enumerate(picks):
        p, y, _, d = xai_items[i]; img = load_u8(p); mk = xai_mask(i)
        for c, (title, n, kind) in enumerate(cols):
            a = ax[r, c]; a.imshow(img, cmap="gray"); a.axis("off")
            if kind is None:
                a.contour(mk, levels=[0.5], colors="lime", linewidths=1.0)
                a.set_title(f"True: {IDX2CLASS[y]}", fontsize=8, color=CLASS_COLORS[IDX2CLASS[y]], fontweight="bold")
            elif kind == "cam":
                cam = up224(ALL_CAMS[n][i].astype(np.float32)); cam = cam / max(cam.max(), 1e-8); row = info[n].loc[i]
                a.imshow(cam, cmap=BLOOD, vmin=0, vmax=1)
                a.contour(mk, levels=[0.5], colors="lime", linewidths=0.8)
                ok = bool(row["correct"])
                a.set_title(f"{'✓' if ok else '✗'} {IDX2CLASS[int(row['pred'])][:5]} ({row['conf']:.2f}) | "
                            f"in tumour {100 * row['energy_in_mask']:.0f}%", fontsize=6.5, color="green" if ok else "red")
            else:
                sm = seg_lookup.get(p)
                if sm is not None:
                    a.imshow(np.ma.masked_less(sm, 0.5), cmap="autumn", alpha=0.55, vmin=0, vmax=1)
                    a.contour(mk, levels=[0.5], colors="lime", linewidths=0.8)
                    txt = f"Dice {dice(sm >= 0.5, mk):.2f}"
                    if kind == "roi":
                        b = box_from_mask(sm >= 0.5)
                        if b is not None:
                            x, y_, w, h, _ = b; s = max(max(w, h) * (1 + 2 * ROI_MARGIN), 32)
                            a.add_patch(plt.Rectangle((x + w / 2 - s / 2, y_ + h / 2 - s / 2), s, s, fill=False, ec="cyan", lw=1.2))
                        ti = test_index.get(p); bi = brats_index.get(p)
                        pr = (TEST_PROBS[PROPOSED][ti] if ti is not None and PROPOSED in TEST_PROBS else
                              BRATS_P[bi] if bi is not None and BRATS_P is not None else None)
                        if pr is not None:
                            hp = int(hard(pr[None])[0]); ok = hp == y
                            txt = f"{'✓' if ok else '✗'} {IDX2CLASS[hp][:5]} ({pr.max():.2f}) | " + txt
                    a.set_title(txt, fontsize=6.5)
            if r == 0:
                a.set_title(f"{title}\n" + a.get_title(), fontsize=a.title.get_fontsize(), color=a.title.get_color(),
                            fontweight="bold" if kind is None else "normal")
    fig.suptitle("Green = expert tumour outline. Cyan box = the only region the ROI model sees. Heat maps: LayerCAM, "
                 "14x14, mean of 5 fold models (red = important; same 0-1 scale for every model); "
                 "'in tumour' = share of heat inside the outline. Sharper maps: section 9.7.", fontsize=8, y=1.01)
    fig.tight_layout(); fig.savefig(FIG_DIR / fname, bbox_inches="tight", dpi=300); plt.show()

XAI_REF = next((n for n in ["CAM-Guide", "CNX-Plain", "CTR-Net"] if n in ALL_CAMS), None)
if RUN_XAI and ALL_CAMS and XAI_REF:
    R_ = df_xai_img[(df_xai_img.Model == XAI_REF) & (df_xai_img.Dataset == "Internal test")]
    rng_x = np.random.default_rng(2026); picks = []
    for c in CLASSES:                                   # one random correct example per class (fixed seed)
        cand = R_[(R_.Class == c) & R_.correct].item.values
        if len(cand): picks.append(int(rng_x.choice(cand)))
    wrong = R_[~R_.correct].item.values                 # + up to 2 random errors (honest figure)
    picks += [int(i) for i in rng_x.choice(wrong, min(2, len(wrong)), replace=False)] if len(wrong) else []
    xai_compare_panel(picks, "fig_xai_main_panel.png")
    ext = [int(i) for d in ["BraTS-GLI", "BraTS-MEN", "BRISC (dedup)"]
           for i in rng_x.choice([j for j, it in enumerate(xai_items) if it[3] == d] or [-1], 1) if i >= 0]
    if ext:
        xai_compare_panel(ext, "fig_xai_external_panel.png")

In [ ]:
# ─── 9.4 Heat-map comparison across models, with the expert tumour outline (green) ───
if RUN_XAI and ALL_CAMS:
    rng = np.random.default_rng(7); pick = []
    for d, k in [("Internal test", 2), ("BRISC (dedup)", 1), ("BraTS-GLI", 2), ("BraTS-MEN", 2)]:
        idx = [i for i, it in enumerate(xai_items) if it[3] == d]
        pick += [int(i) for i in rng.choice(idx, min(k, len(idx)), replace=False)] if idx else []
    names = [n for n in XAI_MODELS if n in ALL_CAMS]
    fig, ax = plt.subplots(len(pick), 1 + len(names), figsize=(1.9 * (1 + len(names)), 1.9 * len(pick)), squeeze=False)
    for r, i in enumerate(pick):
        p, y, _, d = xai_items[i]
        img, mk = load_u8(p), xai_mask(i)
        ax[r, 0].imshow(img, cmap="gray"); ax[r, 0].contour(mk, levels=[0.5], colors="lime", linewidths=0.8)
        ax[r, 0].set_title(f"{d}\n{IDX2CLASS[y]}", fontsize=7)
        for c, n in enumerate(names, 1):
            cam = up224(ALL_CAMS[n][i].astype(np.float32))
            ax[r, c].imshow(img, cmap="gray")
            ax[r, c].imshow(cam, cmap=BLOOD, vmin=0, vmax=1)
            ax[r, c].contour(mk, levels=[0.5], colors="lime", linewidths=0.6)
            yx = np.unravel_index(cam.argmax(), cam.shape)
            ax[r, c].plot(yx[1], yx[0], "w+", ms=7, mew=1.5)
            if r == 0:
                ax[r, c].set_title(n, fontsize=8)
        for a in ax[r]: a.axis("off")
    fig.suptitle("LayerCAM (mean of 5 fold models). Green = expert tumour outline, + = map peak", fontsize=8)
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig_xai_heatmaps_models.png"); plt.show()

In [ ]:
# ─── 9.5 Counterfactual figure: same slice, tumour visible vs tumour hidden ───
if RUN_EVALUATION and REL and PROPOSED in REL and "CNX-Plain" in REL:
    hid = REL["CNX-Plain"]["hidden (swap)"]; ok_plain = hid.argmax(1) == REL_Y
    rng_c = np.random.default_rng(11); rows_ = []
    for k in range(N_CLASSES):                          # slices where the plain model is right WITHOUT the tumour
        cand = np.where(ok_plain & (REL_Y == k) & (REL_DF.role.values == "test"))[0]
        cand = cand if len(cand) else np.where(ok_plain & (REL_Y == k))[0]
        if len(cand): rows_.append(int(rng_c.choice(cand)))
    models_ = list(dict.fromkeys(m for m in ["CNX-Plain", "CNX-Transplant", "CTR-Net", PROPOSED] if m in REL))
    fig, ax = plt.subplots(len(rows_), 2 + len(models_), figsize=(2.3 * (2 + len(models_)), 2.3 * len(rows_)), squeeze=False)
    for r, i in enumerate(rows_):
        p = REL_DF.filepath.values[i]
        ax[r, 0].imshow(load_u8(p), cmap="gray"); ax[r, 0].contour(MASKS[p], levels=[0.5], colors="lime", linewidths=0.8)
        ax[r, 1].imshow(HIDE[p]["swap"], cmap="gray")
        ax[r, 0].set_title(f"Original ({CLASSES[REL_Y[i]]})", fontsize=7); ax[r, 1].set_title("Tumour hidden", fontsize=7)
        for a in ax[r, :2]: a.axis("off")
        for c, m in enumerate(models_, 2):
            po, ph = REL[m]["original"][i], REL[m]["hidden (swap)"][i]; xs = np.arange(N_CLASSES)
            ax[r, c].bar(xs - 0.2, po, 0.4, color="#30638e", label="tumour visible")
            ax[r, c].bar(xs + 0.2, ph, 0.4, color="#d1495b", label="tumour hidden")
            ax[r, c].axhline(1 / 3, color="k", ls=":", lw=0.8)
            ax[r, c].set(ylim=(0, 1), xticks=xs, title=m)
            ax[r, c].set_xticklabels([k[:4] for k in CLASSES], fontsize=7)
            if r == 0 and c == 2: ax[r, c].legend(frameon=False, fontsize=6)
    fig.suptitle("Predicted probabilities with the tumour visible vs hidden. A model that looks at the tumour should "
                 "become unsure (dotted line = 1/3) when it is hidden.", fontsize=8)
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig_counterfactual_examples.png", bbox_inches="tight"); plt.show()

In [ ]:
# ─── 9.6 Heat-map sanity check (Adebayo et al. 2018): do the maps depend on what the model learned? ───
# The trained model's layers are replaced by random weights step by step (head first, then deeper stages).
# If a heat map stays the SAME after randomisation, it shows image edges, not the model's reasoning (fails).
# Measure: rank correlation between the trained model's map and the randomised model's map (low = passes).
import copy
from scipy.stats import spearmanr as _spr
SANITY_MODELS = [n for n in ["CAM-Guide", "CNX-Plain", "CTR-Net", "MTL-Seg"] if n in READY]
if RUN_XAI and SANITY_MODELS:
    banner("9.6 Heat-map sanity check (cascading randomisation)", f"{len(SANITY_MODELS)} models x 60 test slices (~2 min)")
    rng_s = np.random.default_rng(5)
    sitems = [TEST_ITEMS[i] for i in rng_s.choice(len(TEST_ITEMS), min(60, len(TEST_ITEMS)), replace=False)]
    steps = [("head", ["head", "head_norm"]), ("+ stage 4", ["backbone.stages_3"]),
             ("+ stage 3", ["backbone.stages_2"]), ("+ stage 2", ["backbone.stages_1"])]
    def maps_of(model, name, targets, mean, std):
        eng = GradCAMpp(model, cam_target_layer(model, name)); out = []
        for (p, _), t in zip(sitems, targets):
            out.append(F.avg_pool2d(torch.from_numpy(eng(to_batch([load_u8(p)], mean, std), t)[0])[None, None], 4)[0, 0].numpy())
        eng.close(); return np.stack(out)
    san_rows, examples = [], {}
    for n in SANITY_MODELS:
        m0 = copy.deepcopy(load_checkpoint(n, READY[n][0], 1)); mean, std = norm_for(n, 1)
        targets = predict_logits(m0, [load_u8(p) for p, _ in sitems], mean, std).argmax(1)
        base = maps_of(m0, n, targets, mean, std); examples[n] = [base[0]]
        for label, mods in steps:
            for mod_name in mods:
                for sub in m0.get_submodule(mod_name).modules():
                    if hasattr(sub, "reset_parameters"):
                        sub.reset_parameters()
            cur = maps_of(m0, n, targets, mean, std); examples[n].append(cur[0])
            rho = np.nanmean([_spr(a.ravel(), b.ravel())[0] for a, b in zip(base, cur)])
            san_rows.append({"Model": n, "Randomised up to": label, "Similarity to trained map (rank corr.)": round(float(rho), 3)})
        del m0; free_mem()
    df_san = pd.DataFrame(san_rows).pivot(index="Model", columns="Randomised up to", values="Similarity to trained map (rank corr.)")
    df_san = df_san[[s for s, _ in steps]]
    print("Rank correlation between the trained model's heat map and the randomised model's map.")
    print("PASS = drops clearly (e.g. below ~0.3) as the model is randomised; FAIL = stays high (map = edge detector).")
    print(df_san.to_string()); df_san.to_csv(OUT_DIR / "tables" / "table_xai_sanity.csv")
    fig, ax = plt.subplots(len(SANITY_MODELS), 1 + len(steps), figsize=(1.9 * (1 + len(steps)), 2 * len(SANITY_MODELS)), squeeze=False)
    img0 = load_u8(sitems[0][0])
    for r, n in enumerate(SANITY_MODELS):
        for c, (mp, lab) in enumerate(zip(examples[n], ["trained"] + [s for s, _ in steps])):
            ax[r, c].imshow(img0, cmap="gray"); ax[r, c].imshow(up224(mp / (mp.max() + 1e-8)), cmap=BLOOD, vmin=0, vmax=1)
            ax[r, c].axis("off"); ax[r, c].set_title(lab, fontsize=7)
        ax[r, 0].text(-10, 112, n, rotation=90, va="center", ha="right", fontsize=8)
    fig.suptitle("Sanity check: heat maps should change when the model's weights are randomised", fontsize=9)
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig_xai_sanity.png", bbox_inches="tight"); plt.show()

## 9.7 High-resolution explanations for EVERY model, including the ROI models (Run 6)
* **Sharper maps:** LayerCAM on two feature levels (28x28 and 14x14) fused, instead of 14x14 only.
* **ROI models:** the map is computed on the crop the model actually sees and pasted back onto the full slice
  (so it can be compared with the expert outline like every other model).
* **Fair baseline for ROI:** a crop is mostly tumour, so a random map already scores well *inside the crop*.
  We therefore also report **chance inside the crop** and the ratio "heat in tumour / chance".
* One colour scale for every model: **blood red = important** (0-1, same scale everywhere).

In [ ]:
# ─── 9.7a High-resolution heat maps (cached per model) ───
def _bb(m):
    return m.cnn.backbone if isinstance(m, RoiLocNet) else m.backbone

class HiResCAM:
    """Multi-level LayerCAM (Jiang et al. 2021) for ConvNeXt-based models: maps at 28x28 and 14x14, each normalised,
    averaged. Returns (map in input space 224x224, class probabilities)."""
    def __init__(self, model):
        self.model, self.feats = model, None
        self.h = _bb(model).register_forward_hook(lambda mod, inp, out: setattr(self, "feats", out))
    def __call__(self, x, target, loc=None):
        self.model.zero_grad(set_to_none=True)
        with torch.enable_grad():
            out = self.model(x.float(), loc=loc) if loc is not None else self.model(x.float())
            f1, f2 = self.feats[0], self.feats[1]
            g1, g2 = torch.autograd.grad(out[0, int(target)], [f1, f2])
        maps = []
        for A, G in [(f1, g1), (f2, g2)]:
            c = F.relu((F.relu(G) * A).sum(1, keepdim=True)).detach().float()
            c = F.interpolate(c, (IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)[0, 0]
            maps.append((c - c.min()) / (c.max() - c.min() + 1e-8))
        cam = (maps[0] + maps[1]) / 2; cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam.cpu().numpy(), torch.softmax(out.detach().float(), 1)[0].cpu().numpy()
    def close(self):
        self.h.remove()

def roi_window(b, margin):
    """Exactly the square that crop_roi cuts out: (x0, y0, x1, y1)."""
    x, y, w, h, _ = b; s = max(max(w, h) * (1 + 2 * margin), 32); cx, cy = x + w / 2, y + h / 2
    return int(cx - s / 2), int(cy - s / 2), int(cx + s / 2), int(cy + s / 2)

def paste_back(cam_crop, win):
    x0, y0, x1, y1 = win; big = np.zeros((IMG_SIZE, IMG_SIZE), np.float32)
    cs = np.asarray(Image.fromarray(cam_crop.astype(np.float32)).resize((x1 - x0, y1 - y0), Image.BILINEAR))
    xa, ya, xb, yb = max(x0, 0), max(y0, 0), min(x1, IMG_SIZE), min(y1, IMG_SIZE)
    if xb > xa and yb > ya:
        big[ya:yb, xa:xb] = cs[ya - y0:yb - y0, xa - x0:xb - x0]
    return big

def model_inputs(name, f, arrs):
    """What the model really sees: full slices, or (crop, location numbers, window) for ROI models."""
    if not is_roi(name):
        return [(a, None, None) for a in arrs]
    spec = MODEL_SPECS[name]; mg = spec.get("margin", ROI_MARGIN); out = []
    for a, mp in zip(arrs, roi_seg_maps(arrs, f)):
        b = box_from_mask(mp >= 0.5)
        if b is None:
            out.append((None, None, None)); continue
        lc = torch.from_numpy(loc_feats(a, b[4])[None]).to(DEVICE) if spec.get("loc") else None
        out.append((crop_roi(a, b, spec["crop"], margin=mg), lc, roi_window(b, mg)))
    return out

def hires_cams(name, arrs, targets, seed=None):
    """Mean (over the 5 fold models) high-resolution map in FULL-SLICE coordinates + fraction of folds that found a tumour."""
    seed = READY[name][0] if seed is None else seed
    cams = np.zeros((len(arrs), IMG_SIZE, IMG_SIZE), np.float32); found = np.zeros(len(arrs)); wins = [[] for _ in arrs]
    for f in range(1, N_FOLDS + 1):
        m = load_checkpoint(name, seed, f); mean, std = norm_for(name, f); eng = HiResCAM(m)
        for i, (inp, lc, win) in enumerate(model_inputs(name, f, arrs)):
            if inp is None:
                continue
            cam, _ = eng(to_batch([inp], mean, std), targets[i], lc)
            cams[i] += (paste_back(cam, win) if win is not None else cam) / N_FOLDS; found[i] += 1 / N_FOLDS
            if win is not None: wins[i].append(win)
        eng.close(); free_mem()
    return cams, found, wins

HR_MODELS = [n for n in ["ROI-Loc", "ROI-m06-Transplant", "ROI-Crop-m06", "CNX-Transplant", "CAM-Guide", "CNX-Plain",
                         "CTR-Net", "MTL-Seg"] if n in READY]
HR_CAMS, hr_rows, HR_WINS = {}, [], {}
if RUN_XAI and RUN_EVALUATION and HR_MODELS and xai_items:
    banner("9.7a High-resolution heat maps", f"{len(HR_MODELS)} models x {len(xai_items)} images x 5 folds (~3-6 min per model, cached)")
    masks = [xai_mask(i) for i in range(len(xai_items))]
    arrs_x = [load_u8(p) for p, *_ in xai_items]; tg = [y for _, y, _, _ in xai_items]
    pos = {}
    for s_name in ("test", "brisc", "brats"):
        if s_name in EVAL_SETS:
            pos.update({p: (s_name, i) for i, p in enumerate(paths_of(s_name))})
    HR_KEY = "hires_" + XAI_KEY
    for name in HR_MODELS:
        cache = OUT_DIR / "xai" / f"{name}_{HR_KEY}.npz"; rows_csv = OUT_DIR / "xai" / f"{name}_{HR_KEY}_rows.csv"
        if cache.exists() and rows_csv.exists():
            HR_CAMS[name] = np.load(cache)["cams"].astype(np.float32); hr_rows += pd.read_csv(rows_csv).to_dict("records")
            print(f"{name:20s} loaded from cache"); continue
        t0 = time.time(); _CUR_SET["name"] = None
        cams, found, wins = hires_cams(name, arrs_x, tg)
        P = {s: ensemble_probs(name, s) for s in {"test", "brisc", "brats"} & set(EVAL_SETS)}
        rows_m = []
        for i, ((p, y, _, d), cam, mk) in enumerate(zip(xai_items, cams, masks)):
            if mk.sum() == 0:
                continue
            s, j = pos[p]; hp = int(hard(P[s][j][None])[0])
            if wins[i]:
                win_mask = np.zeros_like(mk)
                for (x0, y0, x1, y1) in wins[i]:
                    win_mask[max(y0, 0):max(y1, 0), max(x0, 0):max(x1, 0)] = True
                ch_crop = float(mk[win_mask].mean()) if win_mask.any() else float(mk.mean())
            else:
                ch_crop = float(mk.mean())
            has = cam.max() > 0
            yx = np.unravel_index(cam.argmax(), cam.shape); thr = cam >= 0.3 * cam.max() if has else np.zeros_like(mk)
            rows_m.append({"Model": name, "item": i, "Dataset": d, "Class": IDX2CLASS[y], "correct": hp == y, "pred": hp,
                           "conf": float(P[s][j].max()), "tumour found (share of folds)": float(found[i]),
                           "pointing": float(mk[yx]) if has else 0.0,
                           "energy_in_mask": float(cam[mk].sum() / (cam.sum() + 1e-8)),
                           "iou@0.3": float((thr & mk).sum() / ((thr | mk).sum() + 1e-8)),
                           "chance (full slice)": float(mk.mean()), "chance (inside what the model sees)": ch_crop})
        small = F.avg_pool2d(torch.from_numpy(cams)[:, None], 2)[:, 0].numpy().astype(np.float16)   # 112x112
        np.savez_compressed(cache, cams=small); pd.DataFrame(rows_m).to_csv(rows_csv, index=False)
        HR_CAMS[name] = small.astype(np.float32); hr_rows += rows_m
        print(f"{name:20s} done ({(time.time() - t0) / 60:.1f} min)", flush=True)
    # The 5 fold maps were each scaled 0-1 and then averaged; folds disagree a little, so the averaged peak is lower than 1
    # and the maps looked faint. Re-scale every averaged map to 0-1 (standard practice; all metrics above are scale-free).
    for name in HR_CAMS:
        mx = HR_CAMS[name].reshape(len(HR_CAMS[name]), -1).max(1)[:, None, None]
        HR_CAMS[name] = np.where(mx > 0, HR_CAMS[name] / np.maximum(mx, 1e-8), 0).astype(np.float32)
    df_hr = pd.DataFrame(hr_rows)
    df_hr.to_csv(OUT_DIR / "tables" / "xai_hires_per_image.csv", index=False)
    cols_ = ["pointing", "energy_in_mask", "iou@0.3", "chance (full slice)", "chance (inside what the model sees)",
             "tumour found (share of folds)"]
    agg_hr = df_hr.groupby(["Dataset", "Model"])[cols_].mean()
    agg_hr["heat in tumour / chance"] = agg_hr["energy_in_mask"] / agg_hr["chance (inside what the model sees)"]
    agg_hr["n"] = df_hr.groupby(["Dataset", "Model"]).size()
    agg_hr = agg_hr.round(3)
    print("HIGH-RESOLUTION explanations vs expert tumour outline (mean over images; 5-fold average maps):")
    print(agg_hr.to_string()); agg_hr.to_csv(OUT_DIR / "tables" / "table_xai_hires.csv")
    print("\nHow to read: energy_in_mask = share of the red heat inside the expert outline. For ROI models compare it with")
    print("'chance (inside what the model sees)', because their input is already a crop around the tumour.")
    print("'heat in tumour / chance' > 1 means the model concentrates on the tumour more than a random map would.")

    fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
    for a, metric, ttl in [(ax[0], "energy_in_mask", "Share of heat inside the tumour"),
                           (ax[1], "heat in tumour / chance", "Heat in tumour ÷ random-map baseline (fair for ROI)")]:
        piv = agg_hr[metric].unstack()[HR_MODELS]
        piv.plot.bar(ax=a, rot=0, width=0.85, color=[FAMILY_COLOR[family_of(n)] if not n.startswith("ROI-") or n == PROPOSED
                                                      else {"ROI-m06-Transplant": "#264653", "ROI-Crop-m06": "#f4a261"}.get(n, "#f08a8f")
                                                      for n in HR_MODELS])
        if metric == "heat in tumour / chance":
            a.axhline(1, color="k", ls="--", lw=1); a.set_yscale("log")
        a.set(title=ttl, xlabel=""); a.legend(frameon=False, fontsize=6, ncol=2)
    fig.suptitle("High-resolution heat maps: how much of the 'red' falls on the tumour?", fontsize=9)
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig_xai_hires_metrics.png", bbox_inches="tight"); plt.show()
else:
    df_hr = pd.DataFrame()

In [ ]:
# ─── 9.7b Main explanation figure (journal style, blood-red scale, large panels) ───
# DISPLAY ONLY: each map is smoothed (Gaussian, sigma = 6 px) and re-scaled to 0-1 so that the important region shows as
# one coherent red area; weak heat (< 35% of the peak) is not drawn. All NUMBERS (tables, "in tumour %") are computed on the
# raw, unsmoothed maps in 9.7a.
DISP_SIGMA, DISP_FLOOR = 6, 0.35

def disp_map(n, i):
    cam = np.asarray(Image.fromarray(HR_CAMS[n][i].astype(np.float32)).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR))
    cam = cv2.GaussianBlur(cam, (0, 0), DISP_SIGMA)
    cam = cam / max(cam.max(), 1e-8)
    return np.where(cam >= DISP_FLOOR, cam, 0.0)

def _zoom_box(mk, extra=2.4):
    ys, xs = np.where(mk); cy, cx = ys.mean(), xs.mean()
    s = int(min(max(max(np.ptp(ys), np.ptp(xs), 20) * extra, 64), IMG_SIZE))
    return int(np.clip(cx - s / 2, 0, IMG_SIZE - s)), int(np.clip(cy - s / 2, 0, IMG_SIZE - s)), s

def hires_panel(picks, fname, title, models):
    cols = [("MRI + expert outline", None, "img")]
    for n, kind in models:
        if n in HR_CAMS:
            cols.append((n + (" (zoom)" if kind == "zoom" else ""), n, kind))
    info = {n: df_hr[df_hr.Model == n].set_index("item") for n in HR_CAMS}
    S = 3.0
    fig, ax = plt.subplots(len(picks), len(cols), figsize=(S * len(cols) + 0.9, S * len(picks) + 0.6), squeeze=False)
    for r, i in enumerate(picks):
        p, y, _, d = xai_items[i]; img = load_u8(p); mk = xai_mask(i)
        for c, (ttl, n, kind) in enumerate(cols):
            a = ax[r, c]; a.axis("off")
            if kind == "img":
                a.imshow(img, cmap="gray"); a.contour(mk, levels=[0.5], colors="lime", linewidths=1.4)
                a.set_title(("MRI + expert outline\n" if r == 0 else "") + f"{d} | true: {IDX2CLASS[y]}", fontsize=9,
                            color=CLASS_COLORS[IDX2CLASS[y]], fontweight="bold")
                continue
            row = info[n].loc[i]; hp = int(row["pred"]); ok = hp == y; found = float(row["tumour found (share of folds)"])
            head = (ttl + "\n") if r == 0 else ""
            if is_roi(n) and found == 0:
                a.imshow(img, cmap="gray"); a.contour(mk, levels=[0.5], colors="lime", linewidths=1.0)
                a.text(IMG_SIZE / 2, IMG_SIZE * 0.85, "no tumour found -> unsure", color="yellow", ha="center",
                       fontsize=9, fontweight="bold")
                a.set_title(head + "abstained (referred)", fontsize=9, color="#b8860b"); continue
            cam = disp_map(n, i)
            if kind == "zoom":
                x0, y0, s = _zoom_box(mk); img_, cam_, mk_ = img[y0:y0 + s, x0:x0 + s], cam[y0:y0 + s, x0:x0 + s], mk[y0:y0 + s, x0:x0 + s]
            else:
                img_, cam_, mk_ = img, cam, mk
            a.imshow(img_, cmap="gray"); a.imshow(cam_, cmap=BLOOD, vmin=0, vmax=1, interpolation="bilinear")
            a.contour(mk_, levels=[0.5], colors="lime", linewidths=1.4 if kind == "zoom" else 1.0)
            lab = IDX2CLASS[hp][:5] if hp >= 0 else "unsure"
            extra = f" | {round(found * 5)}/5 folds" if is_roi(n) and found < 1 else ""
            a.set_title(head + f"{'✓' if ok else '✗'} {lab} {row['conf']:.2f} | {100 * row['energy_in_mask']:.0f}% in tumour{extra}",
                        fontsize=8.5, color="green" if ok else "red")
    sm = plt.cm.ScalarMappable(cmap=BLOOD_SOLID, norm=plt.Normalize(0, 1))
    cb = fig.colorbar(sm, ax=ax.ravel().tolist(), fraction=0.015, pad=0.01)
    cb.set_label("importance (0-1, same scale for every model)", fontsize=9)
    fig.suptitle(title, fontsize=11, y=1.0)
    fig.savefig(FIG_DIR / fname, bbox_inches="tight", dpi=300); plt.show()

MAIN_XAI = [("ROI-m06-Transplant", "zoom"), ("ROI-m06-Transplant", "cam"), ("CNX-Transplant", "cam"),
            ("CAM-Guide", "cam"), ("CNX-Plain", "cam")]
SECOND_XAI = [("ROI-Loc", "zoom"), ("ROI-Loc", "cam"), ("CTR-Net", "cam"), ("MTL-Seg", "cam")]
if RUN_XAI and len(df_hr):
    ref = df_hr[(df_hr.Model == HR_MODELS[0]) & (df_hr.Dataset == "Internal test")]
    rng_h = np.random.default_rng(2026); picks = []
    for c in CLASSES:                                         # one random correct example per class (fixed seed)
        cand = ref[(ref.Class == c) & ref.correct].item.values
        if len(cand): picks.append(int(rng_h.choice(cand)))
    wrong = ref[~ref.correct].item.values                     # + one random error (honest figure)
    if len(wrong): picks.append(int(rng_h.choice(wrong)))
    cap = ("Green = expert tumour outline, red = what the model used (LayerCAM, 28+14 px levels, mean of 5 fold models; "
           f"smoothed sigma={DISP_SIGMA}px, weak heat <{int(100 * DISP_FLOOR)}% of peak hidden; % = share of RAW heat inside the tumour)")
    hires_panel(picks, "fig_xai_hires_main.png", "Our solutions (left) vs. standard models (right) - internal test\n" + cap, MAIN_XAI)
    hires_panel(picks, "fig_xai_hires_other.png", "Other models on the same slices\n" + cap, SECOND_XAI)
    ext = []
    for d in ["BraTS-GLI", "BraTS-MEN", "BRISC (dedup)"]:
        cand = df_hr[(df_hr.Model == HR_MODELS[0]) & (df_hr.Dataset == d)].item.values
        if len(cand): ext.append(int(rng_h.choice(cand)))
    if ext:
        hires_panel(ext, "fig_xai_hires_external.png", "New hospitals (BraTS) and BRISC - no fine-tuning\n" + cap, MAIN_XAI)

In [ ]:
# ─── 9.7c 3D view of the heat maps: is there ONE sharp peak on the tumour, or many peaks elsewhere? ───
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
if RUN_XAI and len(df_hr):
    m3 = [n for n in ["ROI-m06-Transplant", "CNX-Transplant", "CNX-Plain"] if n in HR_CAMS]
    cases = picks[:3]
    fig = plt.figure(figsize=(5.6 * len(m3), 5.0 * len(cases)))
    g = np.arange(0, IMG_SIZE, 3); X, Y = np.meshgrid(g, g)
    for r, i in enumerate(cases):
        img = load_u8(xai_items[i][0]); mk = xai_mask(i)
        for c, n in enumerate(m3):
            a = fig.add_subplot(len(cases), len(m3), r * len(m3) + c + 1, projection="3d")
            cam = np.asarray(Image.fromarray(HR_CAMS[n][i].astype(np.float32)).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR))
            cam = cv2.GaussianBlur(cam, (0, 0), DISP_SIGMA); cam = cam / max(cam.max(), 1e-8)
            a.plot_surface(X, Y, np.zeros_like(X, float), facecolors=plt.cm.gray(img[::3, ::3] / 255.0), rstride=1, cstride=1,
                           shade=False, linewidth=0, antialiased=False)
            Z = cam[::3, ::3]
            a.plot_surface(X, Y, np.where(Z >= 0.05, Z, np.nan), cmap=BLOOD_SOLID, vmin=0, vmax=1, rstride=1, cstride=1,
                           linewidth=0, antialiased=True, alpha=0.95)
            ys, xs = np.where(cv2.morphologyEx(mk.astype(np.uint8), cv2.MORPH_GRADIENT, np.ones((3, 3), np.uint8)) > 0)
            a.plot(xs[::2], ys[::2], np.zeros(len(xs[::2])) + 0.01, ".", color="lime", ms=2)
            raw = HR_CAMS[n][i].astype(np.float32); mk_s = np.asarray(Image.fromarray(mk.astype(np.uint8)).resize(raw.shape[::-1])) > 0
            e = float(raw[mk_s].sum() / (raw.sum() + 1e-8))
            a.set(zlim=(0, 1.05), xticks=[], yticks=[], zticks=[0, 0.5, 1]); a.invert_yaxis(); a.view_init(elev=42, azim=-60)
            a.set_title(f"{n}\n{IDX2CLASS[xai_items[i][1]]} | {100 * e:.0f}% of heat in tumour", fontsize=11)
    fig.suptitle("3D heat maps: height + red = importance, floor = MRI slice, green = expert tumour outline on the floor.\n"
                 "A good explanation is ONE peak standing on the green outline; a shortcut model shows peaks elsewhere.",
                 fontsize=12)
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig_xai_3d.png", bbox_inches="tight", dpi=220); plt.show()

In [ ]:
# ─── 9.7d Faithfulness (deletion test): do the RED pixels really drive the decision? ───
# Remove the most important pixels step by step (replace by the typical intensity of HEALTHY brain tissue of the same
# slice - a blurred copy would still look like a tumour to an ROI model) and watch the confidence in the true
# class fall. Compare with removing the SAME number of RANDOM pixels. Faster fall than random = faithful explanation.
DEL_STEPS = [0, 1, 2, 5, 10, 20, 35, 50]                     # % of the pixels the model looks at
DEL_MODELS = [n for n in HR_MODELS if n in ["ROI-Loc", "ROI-m06-Transplant", "CNX-Transplant", "CAM-Guide", "CNX-Plain", "CTR-Net"]]
if RUN_XAI and DEL_MODELS:
    banner("9.7d Deletion test", f"{len(DEL_MODELS)} models x 120 internal-test slices x 5 folds (~2 min per model, cached)")
    rng_d = np.random.default_rng(9)
    _pool = {c: [j for j, it in enumerate(xai_items) if it[3] == "Internal test" and it[1] == c] for c in range(N_CLASSES)}
    di = [i for c in range(N_CLASSES) for i in rng_d.choice(_pool[c], min(40, len(_pool[c])), replace=False)]
    del_rows, CURVES = [], {}
    for n in DEL_MODELS:
        cache = OUT_DIR / "xai" / f"{n}_deletion2_{XAI_KEY}.npz"
        if cache.exists():
            z = np.load(cache); Ch, Cr = z["heat"], z["rand"]
        else:
            Ch = np.full((N_FOLDS, len(di), len(DEL_STEPS)), np.nan); Cr = Ch.copy()
            arrs_d = [load_u8(xai_items[i][0]) for i in di]
            for f in range(1, N_FOLDS + 1):
                m = load_checkpoint(n, READY[n][0], f); mean, std = norm_for(n, f); eng = HiResCAM(m)
                for k, (inp, lc, win) in enumerate(model_inputs(n, f, arrs_d)):
                    if inp is None:
                        continue
                    t = xai_items[di[k]][1]; cam, _ = eng(to_batch([inp], mean, std), t, lc)
                    region = np.ones_like(inp, bool) if is_roi(n) else brain_mask(inp)
                    full = arrs_d[k]; bm_f = brain_mask(full); mk_f = xai_mask(di[k])
                    healthy = np.median(full[bm_f & ~mk_f]) if (bm_f & ~mk_f).any() else np.median(full)
                    blur = np.full_like(inp, int(healthy))        # removed pixels = typical HEALTHY brain intensity
                    order = np.argsort(-np.where(region, cam, -1).ravel()); nreg = int(region.sum())
                    ridx = rng_d.permutation(np.flatnonzero(region.ravel()))
                    batch = []
                    for q in DEL_STEPS:
                        kq = int(nreg * q / 100)
                        for idx in (order[:kq], ridx[:kq]):
                            v = inp.copy().ravel(); v[idx] = blur.ravel()[idx]; batch.append(v.reshape(inp.shape))
                    xb = to_batch(batch, mean, std)
                    with torch.no_grad():
                        lo = m(xb, loc=lc.expand(len(batch), -1)) if lc is not None else m(xb)
                    pr = torch.softmax(lo.float(), 1)[:, t].cpu().numpy()
                    Ch[f - 1, k] = pr[0::2] / max(pr[0], 1e-6); Cr[f - 1, k] = pr[1::2] / max(pr[1], 1e-6)
                eng.close(); free_mem()
            np.savez_compressed(cache, heat=Ch, rand=Cr)
        ch, cr = np.nanmean(Ch, 0), np.nanmean(Cr, 0)                         # (images, steps)
        okr = np.isfinite(ch).all(1) & np.isfinite(cr).all(1)
        _trapz = getattr(np, "trapezoid", None) or np.trapz
        auc_h = _trapz(ch[okr], DEL_STEPS, axis=1) / DEL_STEPS[-1]; auc_r = _trapz(cr[okr], DEL_STEPS, axis=1) / DEL_STEPS[-1]
        gap = auc_r - auc_h; boot = [gap[rng_d.integers(0, len(gap), len(gap))].mean() for _ in range(2000)]
        CURVES[n] = (np.nanmean(ch[okr], 0), np.nanmean(cr[okr], 0))
        del_rows.append({"Model": n, "images": int(okr.sum()), "deletion AUC - red pixels": round(float(auc_h.mean()), 3),
                         "deletion AUC - random pixels": round(float(auc_r.mean()), 3),
                         "faithfulness gap (random - red)": round(float(gap.mean()), 3),
                         "95% CI": f"[{np.percentile(boot, 2.5):.3f}, {np.percentile(boot, 97.5):.3f}]",
                         "p (Wilcoxon)": float(sps.wilcoxon(auc_r, auc_h).pvalue) if len(gap) > 5 else np.nan})
    df_del = pd.DataFrame(del_rows).sort_values("faithfulness gap (random - red)", ascending=False)
    print("Deletion test (confidence in the true class, relative to the untouched image; lower AUC = falls faster).")
    print("Faithfulness gap > 0 with a CI above 0 = the red pixels matter more than random pixels.")
    print(df_del.to_string(index=False)); df_del.to_csv(OUT_DIR / "tables" / "table_xai_faithfulness.csv", index=False)
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
    for n, (h_, r_) in CURVES.items():
        col = {"ROI-m06-Transplant": "#264653"}.get(n, FAMILY_COLOR[family_of(n)])
        ax[0].plot(DEL_STEPS, h_, "-o", ms=3, color=col, label=f"{n} (red pixels)")
        ax[0].plot(DEL_STEPS, r_, "--", color=col, alpha=0.6)
    ax[0].set(xlabel="Pixels removed (% of what the model sees)", ylabel="Confidence (relative to original)",
              title="Deletion curves (solid = red pixels first, dashed = random)", ylim=(0, 1.15))
    ax[0].legend(frameon=False, fontsize=6)
    d_ = df_del.set_index("Model")
    lo_ = [float(s.strip("[]").split(",")[0]) for s in d_["95% CI"]]; hi_ = [float(s.strip("[]").split(",")[1]) for s in d_["95% CI"]]
    g_ = d_["faithfulness gap (random - red)"].values
    ax[1].bar(range(len(d_)), g_, color=[{"ROI-m06-Transplant": "#264653"}.get(n, FAMILY_COLOR[family_of(n)]) for n in d_.index],
              yerr=[g_ - np.array(lo_), np.array(hi_) - g_], capsize=3)
    ax[1].axhline(0, color="k", lw=0.8)
    ax[1].set(xticks=range(len(d_)), ylabel="faithfulness gap", title="How much more the red pixels matter than random ones")
    ax[1].set_xticklabels(d_.index, rotation=25, ha="right", fontsize=7)
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig_xai_deletion.png", bbox_inches="tight"); plt.show()
else:
    df_del = pd.DataFrame()

In [ ]:
# ─── 9.7e Sanity check of the high-resolution maps, incl. the ROI model (cascading randomisation) ───
SAN2 = [n for n in [next((x for x in [PROPOSED, "ROI-m06-Transplant"] if x in HR_CAMS), None), "CNX-Transplant"] if n in READY]
if RUN_XAI and SAN2:
    rng_s2 = np.random.default_rng(6)
    _pool2 = [j for j, it in enumerate(xai_items) if it[3] == "Internal test"]
    sel = [i for i in rng_s2.choice(_pool2, min(40, len(_pool2)), replace=False)]
    arrs_s = [load_u8(xai_items[i][0]) for i in sel]; rows_s = []
    for n in SAN2:
        m0 = copy.deepcopy(load_checkpoint(n, READY[n][0], 1)); mean, std = norm_for(n, 1)
        ins = model_inputs(n, 1, arrs_s); keep = [k for k, t in enumerate(ins) if t[0] is not None]
        pre = "cnn." if isinstance(m0, RoiLocNet) else ""
        steps2 = [("head", [pre + "head", pre + "head_norm"] + (["head", "loc"] if pre else [])),
                  ("+ stage 4", [pre + "backbone.stages_3"]), ("+ stage 3", [pre + "backbone.stages_2"]),
                  ("+ stage 2", [pre + "backbone.stages_1"])]
        def maps2():
            eng = HiResCAM(m0)
            out = [eng(to_batch([ins[k][0]], mean, std), xai_items[sel[k]][1], ins[k][1])[0] for k in keep]
            eng.close(); return out
        base = maps2()
        for lab, mods in steps2:
            for mn in mods:
                for sub in m0.get_submodule(mn).modules():
                    if hasattr(sub, "reset_parameters"): sub.reset_parameters()
            cur = maps2()
            rows_s.append({"Model": n, "Randomised up to": lab,
                           "rank corr.": round(float(np.nanmean([_spr(a[::4, ::4].ravel(), b[::4, ::4].ravel())[0] for a, b in zip(base, cur)])), 3)})
        del m0; free_mem()
    df_san2 = pd.DataFrame(rows_s).pivot(index="Model", columns="Randomised up to", values="rank corr.")[[s for s, _ in steps2]]
    print("Sanity check of the HIGH-RESOLUTION maps (rank correlation trained vs randomised; low = PASS):")
    print(df_san2.to_string()); df_san2.to_csv(OUT_DIR / "tables" / "table_xai_sanity_hires.csv")

## 10. Shortcut case study with the ORIGINAL 4-class models (no training)
Question: did the 4-class model detect "no tumour" from the brain, or from the image source (IXI vs Figshare)?
(a) Can non-anatomical features alone separate no-tumour from tumour images?
(b) What do the old models predict when the brain is erased (only background left), or when only the brain is kept?

In [ ]:
# ─── 10.1 Old 4-class model definition (exact copy of the submitted v2, needed to load its checkpoints) ───
class BSplineKAN(nn.Module):
    def __init__(self, in_features, out_features, grid_size=5, degree=3):
        super().__init__()
        self.grid_size = grid_size
        self.base_weight = nn.Parameter(torch.randn(out_features, in_features) * 0.1)
        self.spline_weight = nn.Parameter(torch.randn(out_features, in_features, grid_size) * 0.1)
        self.register_buffer("grid", torch.linspace(-1, 1, grid_size + degree + 1))
        self.act = nn.SiLU()
    def forward(self, x):
        centers = torch.linspace(-1, 1, self.grid_size, device=x.device)
        basis = torch.exp(-((x.unsqueeze(-1) - centers) ** 2) / 0.5)
        return F.linear(self.act(x), self.base_weight) + torch.einsum("big,oig->bo", basis, self.spline_weight)

class GIK(nn.Module):
    def __init__(self, i, o):
        super().__init__(); self.kan = BSplineKAN(i, o); self.norm = nn.LayerNorm(o)
    def forward(self, x):
        return self.norm(self.kan(x))

class TriConvBlock(ConvBlock):
    pass

class SpatialMambaLite(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.norm, self.proj_in = nn.LayerNorm(dim), nn.Linear(dim, dim)
        self.dwconv = nn.Conv1d(dim, dim, 5, padding=2, groups=dim, bias=False)
        self.gate, self.proj_out, self.act = nn.Linear(dim, dim), nn.Linear(dim, dim), nn.GELU()
    def forward(self, x):
        x0 = x; x = self.norm(x); z = torch.sigmoid(self.gate(x))
        x = self.proj_out(self.act(self.dwconv(self.proj_in(x).transpose(1, 2)).transpose(1, 2)))
        return x0 + z * x

class TriKANHead(nn.Module):
    def __init__(self, i, h1=128, h2=64):
        super().__init__(); self.gik1, self.gik2 = GIK(i, h1), GIK(h1, h2)
    def forward(self, x):
        return self.gik2(self.gik1(x))

class FeatureFusionGate(nn.Module):
    def __init__(self, c, m, k, d=128):
        super().__init__()
        self.cnn_proj, self.mamba_proj, self.kan_proj = nn.Linear(c, d), nn.Linear(m, d), nn.Linear(k, d)
        self.gate = nn.Sequential(nn.Linear(d * 3, d), nn.GELU(), nn.Linear(d, 3), nn.Softmax(dim=1))
        self.out_norm, self.out_drop = nn.LayerNorm(d), nn.Dropout(0.25)
    def forward(self, c, m, k):
        c, m, k = self.cnn_proj(c), self.mamba_proj(m), self.kan_proj(k)
        w = self.gate(torch.cat([c, m, k], 1))
        return self.out_drop(self.out_norm(w[:, :1] * c + w[:, 1:2] * m + w[:, 2:] * k)), w

class MedTriMambaKAN_v2_4class(nn.Module):
    def __init__(self, num_classes=4, base_dim=32):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, base_dim, 3, 1, 1, bias=False), nn.BatchNorm2d(base_dim), nn.GELU())
        self.stage1, self.stage2 = TriConvBlock(base_dim, 64, 2), TriConvBlock(64, 128, 2)
        self.stage3, self.stage4 = TriConvBlock(128, 192, 2), TriConvBlock(192, 256, 2)
        self.feature_map_layer = self.stage4.block[0]
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.cnn_proj = nn.Sequential(nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.20))
        self.mamba_reduce, self.mamba_bn = nn.Conv2d(256, 128, 1, bias=False), nn.BatchNorm2d(128)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, 128))
        self.mamba_block1, self.mamba_block2 = SpatialMambaLite(128), SpatialMambaLite(128)
        self.mamba_norm = nn.LayerNorm(128)
        self.kan_pool, self.kan_head = nn.AdaptiveAvgPool2d(1), TriKANHead(128, 128, 64)
        self.fusion = FeatureFusionGate(128, 128, 64, 128)
        self.skip_alpha = nn.Parameter(torch.tensor(0.1))
        self.classifier = nn.Sequential(nn.Linear(128, 64), nn.GELU(), nn.Dropout(0.30), nn.Linear(64, num_classes))
    def forward(self, x):
        x2 = self.stage2(self.stage1(self.stem(x))); x4 = self.stage4(self.stage3(x2))
        c = self.cnn_proj(self.global_pool(x4).flatten(1))
        m = self.mamba_bn(self.mamba_reduce(x4)).flatten(2).transpose(1, 2)
        m = torch.cat([self.cls_token.expand(m.size(0), -1, -1), m], 1)
        m = self.mamba_norm(self.mamba_block2(self.mamba_block1(m)))[:, 0]
        k = self.kan_head(self.kan_pool(x2).flatten(1))
        fused, _ = self.fusion(c, m, k)
        return self.classifier(fused + self.skip_alpha * c)

In [ ]:
# ─── 10.2 (a) Non-anatomical feature probe: tumour vs no-tumour, patient-grouped CV ───
import cv2
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_predict

def brain_mask_old(u8):
    _, th = cv2.threshold(cv2.GaussianBlur(u8, (5, 5), 0), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8))
    n, lab, st, _ = cv2.connectedComponentsWithStats(th)
    if n <= 1:
        return th > 0
    m = (lab == 1 + np.argmax(st[1:, cv2.CC_STAT_AREA])).astype(np.uint8)
    return cv2.dilate(m, np.ones((11, 11), np.uint8)) > 0

if RUN_SHORTCUT:
    banner("10.2 Shortcut probe", "~2-5 min")
    feats_meta, feats_bg = [], []
    for p in tqdm(df_all.filepath, desc="  Image features (no anatomy)"):
        w, h, _, ext, nbytes = native_props(p)
        u8 = load_u8(p); bm = brain_mask_old(u8); bg = u8[~bm] if (~bm).sum() > 50 else np.zeros(1, np.uint8)
        feats_meta.append([w, h, ext in (".jpg", ".jpeg"), nbytes / 1024])
        feats_bg.append([bg.mean(), bg.std(), bm.mean(), *np.histogram(bg, 16, (0, 255), density=True)[0]])
    y_nt = (df_all.label4 == "notumor").astype(int).values
    probe_rows = []
    for fname, X in [("File metadata (size, format, bytes)", np.array(feats_meta, float)),
                     ("Background intensities only", np.array(feats_bg, float)),
                     ("Both", np.hstack([feats_meta, feats_bg]).astype(float))]:
        prob = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)), X, y_nt,
                                 groups=df_all.pid.values, cv=GroupKFold(5), method="predict_proba")[:, 1]
        probe_rows.append({"Features (no anatomy)": fname, "AUC tumour vs no-tumour": round(roc_auc_score(y_nt, prob), 4)})
    df_probe = pd.DataFrame(probe_rows)
    print("Can 'no tumour' be recognised WITHOUT looking at the brain? (AUC 0.5 = no, 1.0 = perfectly)")
    print(df_probe.to_string(index=False))
    df_probe.to_csv(OUT_DIR / "tables" / "table_shortcut_probe.csv", index=False)

In [ ]:
# ─── 10.3 (b) Old 4-class models on the original 713-image test set: original / brain erased / brain only ───
def find_old_ckpt(k):
    p = OLD_ARTIFACT_DIR / "models" / f"MedTriMambaKAN_v2_fold{k}_best.pt"
    if p.exists():
        return p
    hits = list(KAGGLE_IN.rglob(p.name)) if KAGGLE_IN.exists() else []
    return hits[0] if hits else None

def old_fold_stats():
    js = OLD_ARTIFACT_DIR / "outputs" / "fold_stats.json"
    if js.exists():
        return {int(k): v for k, v in json.load(open(js)).items()}
    out = {}                                   # exact replication of the original Cell 10
    for k in range(1, N_FOLDS + 1):
        random.seed(SPLIT_SEED)
        tr = df_all[(df_all.role == "dev") & (df_all.val_fold != k)]
        s = s2 = n = 0.0
        for cls in CLASSES_4:
            files = list(tr[tr.label4 == cls].filepath)
            for f in random.sample(files, min(200, len(files))):
                a = np.asarray(Image.open(f).convert("L").resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC), np.float64) / 255
                s += a.sum(); s2 += (a ** 2).sum(); n += a.size
        mu = s / n; out[k] = {"mean": [mu] * 3, "std": [math.sqrt(s2 / n - mu ** 2)] * 3}
    return out

shortcut_rows = []
if RUN_SHORTCUT:
    old_ckpts = {k: find_old_ckpt(k) for k in range(1, N_FOLDS + 1)}
    old_ckpts = {k: v for k, v in old_ckpts.items() if v is not None}
    print("Old 4-class checkpoints found:", sorted(old_ckpts))
    if old_ckpts:
        ostats = old_fold_stats()
        te4 = df_all[df_all.role == "test"].reset_index(drop=True)
        y4 = te4.label4_idx.values
        conds = {"original": [], "brain erased": [], "brain only": []}
        for p in tqdm(te4.filepath, desc="  Building erased images"):
            u8 = load_u8(p); bm = brain_mask_old(u8)
            conds["original"].append(u8)
            conds["brain erased"].append(np.where(bm, 0, u8).astype(np.uint8))
            conds["brain only"].append(np.where(bm, u8, 0).astype(np.uint8))
        probs4 = {c: [] for c in conds}
        for k, ck in old_ckpts.items():
            m = MedTriMambaKAN_v2_4class().to(DEVICE)
            m.load_state_dict(torch.load(ck, map_location=DEVICE), strict=True); m.eval()
            for c, arrs in conds.items():
                probs4[c].append(softmax_np(predict_logits(m, arrs, ostats[k]["mean"], ostats[k]["std"])))
            del m
        for c in conds:
            pr = np.mean(probs4[c], 0).argmax(1)
            for k4, cname in enumerate(CLASSES_4):
                mk = y4 == k4
                shortcut_rows.append({"Input": c, "True class": cname, "n": int(mk.sum()),
                                      "Predicted no-tumour (%)": round(100 * (pr[mk] == 3).mean(), 1),
                                      "Correct (%)": round(100 * (pr[mk] == k4).mean(), 1)})
        df_short = pd.DataFrame(shortcut_rows)
        print("Share of test images that the OLD 4-class model calls 'NO TUMOUR' (%):")
        print(df_short.pivot(index="True class", columns="Input", values="Predicted no-tumour (%)")
              [["original", "brain erased", "brain only"]])
        print("\nShare of test images classified CORRECTLY (%):")
        print(df_short.pivot(index="True class", columns="Input", values="Correct (%)")[["original", "brain erased", "brain only"]])
        df_short.to_csv(OUT_DIR / "tables" / "table_shortcut_erasure.csv", index=False)

        fig, ax = plt.subplots(2, 3, figsize=(6, 4.2))
        for r, cls in enumerate(["glioma", "notumor"]):
            i = int(np.where(te4.label4 == cls)[0][0])
            for c, cond in enumerate(conds):
                ax[r, c].imshow(conds[cond][i], cmap="gray"); ax[r, c].axis("off")
                ax[r, c].set_title(f"{cls} | {cond}", fontsize=7)
        fig.tight_layout(); fig.savefig(FIG_DIR / "fig_shortcut_inputs.png"); plt.show()

## 10b. Do large foundation models also classify without the tumour?
Frozen image features of large pretrained models + a simple linear classifier (same folds, same tests).
No fine-tuning, so this measures what the foundation features themselves contain.

In [ ]:
# ─── 10b.1 Foundation models: tumour-hiding test, BraTS, BDNeuro, tumour-free slices ───
FM_SPECS = [("DINOv2-S", "timm", "vit_small_patch14_dinov2.lvd142m"),
            ("DINOv2-B", "timm", "vit_base_patch14_dinov2.lvd142m"),
            ("BiomedCLIP", "openclip", "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224")]

def fm_encoder(kind, ref):
    if kind == "timm":
        m = timm.create_model(ref, pretrained=True, num_classes=0, img_size=IMG_SIZE).to(DEVICE).eval()
        return (lambda x: m(x)), IMAGENET_MEAN, IMAGENET_STD
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch"], check=False)
    import open_clip
    m, _ = open_clip.create_model_from_pretrained(ref); m = m.to(DEVICE).eval()
    return (lambda x: m.encode_image(x)), [0.48145466, 0.4578275, 0.40821073], [0.26862954, 0.26130258, 0.27577711]

def fm_embed(enc, arrs, mean, std):
    out = []
    with torch.no_grad():
        for i in range(0, len(arrs), 64):
            with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
                out.append(enc(to_batch(arrs[i:i + 64], mean, std)).float().cpu().numpy())
    return np.concatenate(out)

fm_rows = []
if RUN_EVALUATION and len(REL_DF):
    banner("10b.1 Foundation models (frozen features + linear classifier)", "~3-6 min per model (cached)")
    conds = {"original": None, "hidden (swap)": "swap", "random spot hidden": "random", "skull / position only": "skull"}
    ext = {s: EVAL_SETS[s] for s in ["brats", "bdneuro", "bfree"] if s in EVAL_SETS}
    for fm_name, kind, ref in FM_SPECS:
        cache = OUT_DIR / "fm" / f"{fm_name}.npz"
        try:
            if cache.exists():
                E = dict(np.load(cache))
            else:
                enc, mean, std = fm_encoder(kind, ref); E = {}
                for c, key in conds.items():
                    E[c] = fm_embed(enc, [load_u8(p) if key is None else HIDE[p][key] for p in REL_DF.filepath], mean, std)
                for s, items in ext.items():
                    E[s] = fm_embed(enc, [load_u8(p) for p, _ in items], mean, std)
                cache.parent.mkdir(exist_ok=True); np.savez_compressed(cache, **E); del enc; free_mem()
        except Exception as ex:
            print(f"  {fm_name}: skipped ({type(ex).__name__}: {str(ex)[:120]})"); continue
        P = {c: np.zeros((len(REL_DF), N_CLASSES)) for c in conds}; Pe = {s: 0 for s in ext}
        test_idx = np.where(REL_DF.role == "test")[0]
        for f in range(1, N_FOLDS + 1):
            tr = np.where((REL_DF.role == "dev") & (REL_DF.val_fold != f))[0]
            va = np.where((REL_DF.role == "dev") & (REL_DF.val_fold == f))[0]
            clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.5)).fit(E["original"][tr], REL_Y[tr])
            for c in conds:
                P[c][va] = clf.predict_proba(E[c][va]); P[c][test_idx] += clf.predict_proba(E[c][test_idx]) / N_FOLDS
            for s in ext:
                Pe[s] = Pe[s] + clf.predict_proba(E[s]) / N_FOLDS
        ba = {c: bal_acc(REL_Y, P[c].argmax(1)) for c in conds}
        row = {"Model": fm_name, "original": round(100 * ba["original"], 1), "hidden (swap)": round(100 * ba["hidden (swap)"], 1),
               "skull / position only": round(100 * ba["skull / position only"], 1),
               "Tumour reliance": round(float(np.clip((ba["original"] - ba["hidden (swap)"]) / (ba["original"] - 1 / 3), 0, 1)), 2),
               "Internal test bal. acc (%)": round(100 * bal_acc(REL_Y[test_idx], P["original"][test_idx].argmax(1)), 1)}
        if "brats" in Pe:
            yb_ = labels_of("brats"); g = Pe["brats"][:, 0] / (Pe["brats"][:, 0] + Pe["brats"][:, 1] + 1e-12)
            row["BraTS AUC"] = round(roc_auc_score(yb_ == 0, g), 4)
        if "bdneuro" in Pe:
            row["BDNeuro bal. acc (%)"] = round(100 * bal_acc(labels_of("bdneuro"), Pe["bdneuro"].argmax(1)), 1)
        if "bfree" in Pe:
            row["Confident on tumour-free (%)"] = round(100 * (Pe["bfree"].max(1) >= 0.85).mean(), 1)
        fm_rows.append(row); print(f"  {fm_name:11s} done")
    if fm_rows:
        df_fm = pd.DataFrame(fm_rows)
        ref = [r for r in rel_rows if r["Model"] in ("CNX-Plain", PROPOSED)]
        print("\nFoundation models (frozen) vs trained models - balanced accuracy (%), guessing = 33.3%:")
        print(df_fm.to_string(index=False)); df_fm.to_csv(OUT_DIR / "tables" / "table_foundation_models.csv", index=False)
        print("Reference (fine-tuned):", {r["Model"]: (r["original"], r["hidden (swap)"]) for r in ref})

## 11. Export for GitHub / Zenodo + numbers for the paper

In [ ]:
# ─── 11.0 Study-design figure (Figure 1 of the paper) ───
def box(ax, x, y, w, h, text, fc):
    ax.add_patch(plt.Rectangle((x, y), w, h, fc=fc, ec="#333333", lw=0.8))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=6.5, wrap=True)
def arrow(ax, x1, y1, x2, y2):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="->", lw=0.8, color="#333333"))
n_dev = int((df3.role == "dev").sum()); n_te = len(TEST_ITEMS)
fig, ax = plt.subplots(figsize=(13, 5.0)); ax.set(xlim=(0, 13), ylim=(0.6, 5.6)); ax.axis("off")
C_DATA, C_AUDIT, C_MODEL, C_EVAL = "#e8f1f8", "#fbe9e7", "#e3f4f1", "#fff4e0"
for x_, t_ in [(1.25, "1. DATA"), (4.35, "2. DIAGNOSE the problem"), (7.65, "3. SOLVE it"), (11.0, "4. VERIFY (new hospitals + safety)")]:
    ax.text(x_, 5.4, t_, ha="center", fontsize=9, fontweight="bold", color="#333333")
box(ax, 0.1, 4.0, 2.3, 1.1, f"Figshare T1c, 3 tumour types\n{len(df3)} slices / {df3.pid.nunique()} patients\n+ expert masks (.mat)", C_DATA)
box(ax, 0.1, 2.5, 2.3, 1.1, f"Patient-level split\ndev {n_dev} slices (5 folds)\ntest {n_te} slices / {df3[df3.role=='test'].pid.nunique()} patients", C_DATA)
box(ax, 0.1, 1.0, 2.3, 1.1, f"Duplicate audit (5 public sets)\nBRISC {len(BRISC_ITEMS)}->{len(BRISC_CLEAN)}, Mendeley {len(MEND_ITEMS)}->{len(MEND_CLEAN)}\nBDNeuro {len(BD_ITEMS)}->{len(BD_CLEAN)} unique", C_AUDIT)
box(ax, 2.9, 4.0, 2.9, 1.1, "Tumour-hiding test (20+ models)\nhide tumour / random spot / skull only\n-> tumour-reliance score", C_AUDIT)
box(ax, 2.9, 2.5, 2.9, 1.1, "Metric check: bigger hidden areas,\ntissue from another patient;\nplane + location baselines", C_AUDIT)
box(ax, 2.9, 1.0, 2.9, 1.1, "Existing fixes fail: multi-task seg,\nRRR, CAM guidance, counterfactual\ntraining (CTR), foundation models", C_AUDIT)
box(ax, 6.3, 4.0, 2.7, 1.1, "SOLUTION 1  ROI-Loc\ntumour crop + 6 location numbers\n(no whole slice -> no shortcut)", C_MODEL)
box(ax, 6.3, 2.5, 2.7, 1.1, "SOLUTION 2  Tumour transplant\nordinary model; surroundings from a\npatient of another class", C_MODEL)
box(ax, 6.3, 1.0, 2.7, 1.1, "Context sweep (box 0.2 / 0.6 / 1.5 / whole)\n+ context with transplant (combined)\n+ skull-strip control", C_MODEL)
box(ax, 9.5, 4.0, 3.3, 1.1, "Internal test + CV\npatient-level metrics, cluster bootstrap,\nNadeau-Bengio, Holm", C_EVAL)
box(ax, 9.5, 2.5, 3.3, 1.1, f"New hospitals, no fine-tuning\nBraTS-GLI/MEN {len(BRATS_ITEMS)} slices,\nBDNeuro {len(BD_CLEAN)}, Mendeley {len(MEND_CLEAN)} unique", C_EVAL)
box(ax, 9.5, 1.0, 3.3, 1.1, f"Safety: {len(BFREE_ITEMS)} tumour-free slices, shortcut\nalarm, conformal referral, calibration,\nLayerCAM + sanity check", C_EVAL)
for y_ in (4.55, 3.05, 1.55):
    arrow(ax, 2.4, y_, 2.9, y_); arrow(ax, 5.8, y_, 6.3, y_); arrow(ax, 9.0, y_, 9.5, y_)
ax.set_title("Study design: measure the shortcut, show why existing fixes fail, remove it, verify on new hospitals",
             fontsize=10); fig.tight_layout(); fig.savefig(FIG_DIR / "fig_study_design.png", bbox_inches="tight"); plt.show()

In [ ]:
# ─── 11.1 Checkpoint manifest, environment, README ───
manifest = []
for name in READY:
    for s in READY[name]:
        for f in range(1, N_FOLDS + 1):
            r = load_json(run_paths(name, s, f)[1])
            manifest.append({"model": name, "seed": s, "fold": f, "checkpoint": r["ckpt"], "sha256": r["sha256"],
                             "lr": r["lr"], "best_epoch": r["best_epoch"], "val_f1": round(r["val_f1"], 4)})
pd.DataFrame(manifest).to_csv(OUT_DIR / "checkpoint_manifest.csv", index=False)
env = {"python": platform.python_version(), "torch": torch.__version__, "torchvision": torchvision.__version__,
       "timm": timm.__version__, "numpy": np.__version__, "pandas": pd.__version__,
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
       "settings": {k: globals()[k] for k in ["IMG_SIZE", "BATCH_SIZE", "NUM_EPOCHS", "PATIENCE", "WARMUP_EPOCHS",
                                               "WEIGHT_DECAY", "LABEL_SMOOTHING", "MIX_PROB", "N_FOLDS", "SPLIT_SEED"]},
       "learning_rates": {n: get_lr(n) for n in MODEL_ORDER}}
save_json(env, OUT_DIR / "environment.json")
(OUT_DIR / "README.md").write_text(
    "# Brain-tumour shortcut study (Run 5) - reproducibility package\n\n"
    "* `split_3class.csv` - patient-level split (test patients + validation fold of every image)\n"
    "* `overlap_audit.csv` - near-duplicate audit of BRISC / Mendeley vs Figshare\n"
    "* `checkpoint_manifest.csv` - every trained model with SHA-256\n"
    "* `tables/` - every number reported in the paper\n"
    "* `environment.json` - library versions and hyper-parameters\n\n"
    "* `figshare_masks_224.npz` - expert tumour masks matched to every Figshare PNG slice\n"
    "* `xai/` - cached LayerCAM maps (56x56) and per-image XAI scores\n\n"
    "Re-run: attach the checkpoints as a Kaggle dataset, set `RUN_TRAINING=False`, run all cells.\n")
make_artifact_zip()   # final update: same zip, now also with all tables of this run

In [ ]:
# ─── 11.2 Key numbers (copy into the manuscript from these tables only) ───
for t in ["table_success_criteria", "table_solutions", "table_xai_hires", "table_xai_faithfulness", "table_xai_sanity_hires",
          "table_roiloc_parts", "table_skull_control", "table_shortcut_alarm", "table_metric_check",
          "table_reliance", "table_tumour_free", "table_location_plane_baselines", "table_accuracy_by_plane",
          "table_foundation_models", "table_xai_sanity", "table_reliance_vs_generalisation", "table_ablation", "table_cv", "table_cv_ttest",
          "table_test", "table_test_paired", "table_brats", "table_brats_paired", "table_bdneuro",
          "table_segmentation_head", "table_selective_prediction", "table_xai", "table_feature_source",
          "table_calibration", "table_overlap", "table_shortcut_probe", "table_complexity"]:
    p = OUT_DIR / "tables" / f"{t}.csv"
    if p.exists():
        print(f"\n===== {t} =====")
        print(pd.read_csv(p).to_string(index=False))

## 12. Did the solutions work? (automatic check of the criteria fixed in cell 0.4)
✔ = criterion met, ✘ = not met, ? = model not trained yet (next run). Report ✘ results honestly - they are findings too.

In [ ]:
# ─── 12.1 Success-criteria check ───
def _get(n, col):
    return SOLV.get(n, {}).get(col, np.nan) if SOLV else np.nan
def _alarm(model, set_, col):
    if not len(df_alarm): return np.nan
    d_ = df_alarm[(df_alarm.Model == model) & (df_alarm.Set == set_)]
    return d_[col].iloc[0] if len(d_) and col in d_ else np.nan
res = []
def judge(cid, ok, value):
    res.append({"ID": cid, "Result": "?" if ok is None else ("✔" if ok else "✘"), "Measured": value})
v = (_get("ROI-Loc", "CV macro-F1 (%)"),); judge("S1a", None if np.isnan(v[0]) else v[0] >= 96.0, f"CV macro-F1 {v[0]:.2f}%")
r_, sk = _get("ROI-Loc", "Reliance (swap)"), _get("ROI-Loc", "Skull/position-only BA (%)")
judge("S1b", None if np.isnan(r_) else (r_ >= 0.60 and sk <= 36), f"reliance {r_:.2f}, skull/position-only {sk:.1f}%")
a1, a0 = _get("ROI-Loc", "BraTS AUC"), _get("ROI-Crop", "BraTS AUC")
judge("S1c", None if np.isnan(a1) or np.isnan(a0) else a1 >= a0 - 0.01, f"BraTS AUC {a1:.4f} vs ROI-Crop {a0:.4f}")
r_ = _get("CNX-Transplant", "Reliance (swap)"); judge("S2a", None if np.isnan(r_) else r_ >= 0.50, f"reliance (swap) {r_:.2f}")
r_ = _get("CNX-Transplant", "Reliance (other patient)"); judge("S2b", None if np.isnan(r_) else r_ >= 0.45, f"reliance (other patient) {r_:.2f}")
f_, pit = _get("CNX-Transplant", "CV macro-F1 (%)"), _get("CNX-Transplant", "Pituitary correct when hidden (%)")
judge("S2c", None if np.isnan(f_) else (f_ >= 95.0 and pit < 80), f"CV macro-F1 {f_:.2f}%, pituitary when hidden {pit:.1f}%")
a_s = _get("CNX-Strip", "BraTS AUC")
judge("S3", None if np.isnan(a_s) or np.isnan(a0) else a_s < a0 - 0.02,
      f"CNX-Strip {a_s:.4f} vs ROI-Crop {a0:.4f} vs CNX-Plain {_get('CNX-Plain', 'BraTS AUC'):.4f}")
c_, k_ = _alarm("CNX-Plain", "BraTS", "AURC conf (%)"), _alarm("CNX-Plain", "BraTS", "AURC combo (%)")
judge("S4a", None if np.isnan(c_) else k_ < c_, f"AURC alarm {k_} vs confidence {c_} (%)")
u_ = _alarm("CNX-Plain", "BraTS tumour vs tumour-free", "AUROC combo")
judge("S4b", None if np.isnan(u_) else u_ >= 0.75, f"AUROC {u_} (confidence alone: {_alarm('CNX-Plain', 'BraTS tumour vs tumour-free', 'AUROC conf')})")
judge("S5", None if np.isnan(RHO_METRIC) else RHO_METRIC >= 0.80, f"Spearman {RHO_METRIC:.2f}")
f6, r6, b6 = _get("ROI-m06-Transplant", "CV macro-F1 (%)"), _get("ROI-m06-Transplant", "Reliance (swap)"), _get("ROI-m06-Transplant", "BraTS AUC")
judge("S6a", None if np.isnan(f6) else f6 >= 96.0, f"CV macro-F1 {f6:.2f}%")
judge("S6b", None if np.isnan(r6) else r6 >= 0.60, f"reliance {r6:.2f}")
judge("S6c", None if np.isnan(b6) or np.isnan(a0) else b6 >= a0 - 0.01, f"BraTS AUC {b6:.4f} vs ROI-Crop {a0:.4f}")
try:
    rr = agg_hr.loc[("Internal test", "ROI-Loc")]; ratio = float(rr["heat in tumour / chance"])
    extra7 = (f"; info: ROI-m06-Transplant ratio {float(agg_hr.loc[('Internal test', 'ROI-m06-Transplant')]['heat in tumour / chance']):.2f}"
              if ("Internal test", "ROI-m06-Transplant") in agg_hr.index else "")
    judge("S7", ratio >= 2.0, f"heat in tumour {100 * rr['energy_in_mask']:.1f}% vs chance in crop "
                               f"{100 * rr['chance (inside what the model sees)']:.1f}% (ratio {ratio:.2f}){extra7}")
except Exception:
    judge("S7", None, "not computed")
try:
    rd = df_del.set_index("Model").loc["ROI-Loc"]; lo7 = float(rd["95% CI"].strip("[]").split(",")[0])
    judge("S8", lo7 > 0, f"faithfulness gap {rd['faithfulness gap (random - red)']:.3f} {rd['95% CI']}")
except Exception:
    judge("S8", None, "not computed")
df_check = pd.DataFrame(CRITERIA, columns=["ID", "Experiment", "Success if ..."]).merge(pd.DataFrame(res), on="ID")
print(df_check[["ID", "Result", "Experiment", "Measured", "Success if ..."]].to_string(index=False))
df_check.to_csv(OUT_DIR / "tables" / "table_success_criteria.csv", index=False)
print(f"\n{(df_check.Result == '✔').sum()} met | {(df_check.Result == '✘').sum()} not met | {(df_check.Result == '?').sum()} not testable yet")

In [ ]:
# ─── 12.2 Main results at a glance (one figure for the paper / presentation) ───
GL = [n for n in ["ROI-Loc", "ROI-m06-Transplant", "ROI-Crop-m06", "CNX-Transplant", "CNX-Plain", "CTR-Net", "MTL-Seg",
                  "CAM-Guide", "ConvNeXt-Tiny", "MTK"] if n in SOLV]
if GL:
    panels = [("Reliance (swap)", "Tumour reliance (higher = uses the tumour)", (0, 1)),
              ("CV macro-F1 (%)", "Cross-validation macro-F1 (%)", (85, 100)),
              ("BraTS AUC", "BraTS AUC (new hospital)", (0.8, 1.0)),
              ("BDNeuro BA (%)", "BDNeuro balanced accuracy (%)", (75, 100)),
              ("Confident on tumour-free (%)", "Confident on TUMOUR-FREE slices (%) - lower is safer", (0, 70)),
              ("Pituitary correct when hidden (%)", "Pituitary correct with tumour HIDDEN (%) - lower = no shortcut", (0, 100))]
    fig, ax = plt.subplots(2, 3, figsize=(15, 7.2))
    for a, (col, ttl, lim) in zip(ax.ravel(), panels):
        v = [SOLV[n].get(col, np.nan) for n in GL]
        cols = [{"ROI-m06-Transplant": "#264653", "ROI-Crop-m06": "#f4a261"}.get(n, FAMILY_COLOR[family_of(n)]) for n in GL]
        a.barh(range(len(GL)), v, color=cols)
        for k_, vv in enumerate(v):
            if np.isfinite(vv): a.text(vv, k_, f" {vv:.3g}", va="center", fontsize=7)
        a.set(yticks=range(len(GL)), xlim=lim, title=ttl); a.set_yticklabels(GL, fontsize=7); a.invert_yaxis()
        a.title.set_fontsize(8.5)
    fig.suptitle("Main results at a glance: the solutions (top rows) use the tumour, transfer to a new hospital and stay "
                 "unsure without a tumour; standard models do not", fontsize=10)
    fig.tight_layout(); fig.savefig(FIG_DIR / "fig_results_at_a_glance.png", bbox_inches="tight", dpi=300); plt.show()

In [ ]:
# ─── 12.3 ⬇ Download links again (same files as in cell 3.6, now including all tables and figures) ───
make_artifact_zip()
show_downloads()